# CLO-SKET — Final Validation Shield

## Purpose

This notebook performs reviewer-oriented validation of the frozen CLO-SKET
morphology–radial–angular results.

It does not:

- modify the frozen 135-D morphology representation;
- modify the frozen 28-D radial–angular representation;
- select features using category performance;
- replace any previously frozen result object;
- search for a more favourable model.

It evaluates:

1. source-garment grouped cross-validation;
2. grouped recovery of radial–angular measurements;
3. grouped downstream complementarity;
4. within-category radial–angular permutation controls;
5. identity-aware uncertainty intervals;
6. canonical result and provenance export.

All newly created objects use the prefix `shield_`.

In [ ]:
# ============================================================
# CELL 1 — VALIDATION-SHIELD ENVIRONMENT
# ============================================================

from pathlib import Path
import json
import pickle
import hashlib
import platform
import sys
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from scipy.stats import spearmanr

warnings.filterwarnings("default")

SHIELD_RANDOM_STATE = 20260820
SHIELD_N_SPLITS = 5

rng = np.random.default_rng(SHIELD_RANDOM_STATE)

shield_environment = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn_version,
    "random_state": SHIELD_RANDOM_STATE,
    "n_splits": SHIELD_N_SPLITS,
}

print("=" * 72)
print("🛡️ CLO-SKET — FINAL VALIDATION SHIELD")
print("=" * 72)

for key, value in shield_environment.items():
    print(f"{key:22s}: {value}")

print("\n🟢 Environment initialized")
print("🟢 Existing frozen objects have not been modified")

🛡️ CLO-SKET — FINAL VALIDATION SHIELD
created_utc           : 2026-08-20T05:03:02.050946+00:00
python_version        : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
platform              : Linux-6.6.122+-x86_64-with-glibc2.35
numpy_version         : 2.0.2
pandas_version        : 2.2.3
sklearn_version       : 1.6.1
random_state          : 20260820
n_splits              : 5

🟢 Environment initialized
🟢 Existing frozen objects have not been modified


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ============================================================
# CELL 2 — FROZEN BACKUP INTEGRITY AND INVENTORY
# ============================================================

shield_backup_paths = {
    "initial_radial_angular": Path(
        "/content/CLO_SKET_runtime_backup.pkl"
    ),
    "post_cell25": Path(
        "/content/CLO_SKET_runtime_backup_AFTER_CELL25.pkl"
    ),
}

print("=" * 72)
print("🛡️ CELL 2 — FROZEN BACKUP INTEGRITY AND INVENTORY")
print("=" * 72)

# ------------------------------------------------------------
# 1. Verify that both files exist
# ------------------------------------------------------------

for backup_name, path in shield_backup_paths.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Required frozen backup not found: {path}"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Expected a file but found another object: {path}"
        )

print("🟢 Both frozen backup files located")

# ------------------------------------------------------------
# 2. Compute SHA-256 without modifying either file
# ------------------------------------------------------------

def shield_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while chunk := file_handle.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


shield_backup_manifest = {}

for backup_name, path in shield_backup_paths.items():
    shield_backup_manifest[backup_name] = {
        "path": str(path),
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": shield_sha256(path),
    }

    print("\n" + "-" * 72)
    print(f"Backup: {backup_name}")
    print(f"File  : {path.name}")
    print(f"Size  : {path.stat().st_size:,} bytes")
    print(f"SHA256: {shield_backup_manifest[backup_name]['sha256']}")

# ------------------------------------------------------------
# 3. Load trusted frozen backups into separate variables
# ------------------------------------------------------------

with shield_backup_paths["initial_radial_angular"].open("rb") as file_handle:
    shield_initial_backup = pickle.load(file_handle)

with shield_backup_paths["post_cell25"].open("rb") as file_handle:
    shield_post25_backup = pickle.load(file_handle)

if not isinstance(shield_initial_backup, dict):
    raise TypeError(
        "Initial backup is not a dictionary."
    )

if not isinstance(shield_post25_backup, dict):
    raise TypeError(
        "Post-Cell-25 backup is not a dictionary."
    )

# ------------------------------------------------------------
# 4. Print inventories
# ------------------------------------------------------------

shield_loaded_backups = {
    "initial_radial_angular": shield_initial_backup,
    "post_cell25": shield_post25_backup,
}

for backup_name, backup_object in shield_loaded_backups.items():
    print("\n" + "=" * 72)
    print(f"{backup_name}: {len(backup_object)} objects")
    print("-" * 72)

    for object_name in sorted(backup_object):
        value = backup_object[object_name]
        shape = getattr(value, "shape", None)

        print(
            f"{object_name:40s} "
            f"type={type(value).__name__:18s} "
            f"shape={str(shape)}"
        )

print("\n" + "=" * 72)
print("🟢 BACKUP INTEGRITY RECORDED")
print("🟢 BACKUPS LOADED INTO SEPARATE VARIABLES")
print("🟢 NO FROZEN OBJECT WAS MODIFIED")
print("=" * 72)

🛡️ CELL 2 — FROZEN BACKUP INTEGRITY AND INVENTORY
🟢 Both frozen backup files located

------------------------------------------------------------------------
Backup: initial_radial_angular
File  : CLO_SKET_runtime_backup.pkl
Size  : 139,198,448 bytes
SHA256: 78f93a42cf82c7515500f65032b22d102c392400236caf150eb4f5b57df04b69

------------------------------------------------------------------------
Backup: post_cell25
File  : CLO_SKET_runtime_backup_AFTER_CELL25.pkl
Size  : 147,060,919 bytes
SHA256: 4e7d6ea942b3fd4b506c330f624178e154022683756e6edcffcf7aa65bd69f9f

initial_radial_angular: 22 objects
------------------------------------------------------------------------
F2                                       type=ndarray            shape=(2300, 72)
F2_complex                               type=ndarray            shape=(2300, 72)
F2_integral                              type=ndarray            shape=(2300,)
F2_mag                                   type=ndarray            shape=(2300, 72)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ============================================================
# CELL 3 — RA INTEGRITY, PATH STRUCTURE,
#          ORIGINAL CV, AND FAST ARTIFACT DISCOVERY
# ============================================================

import re

print("=" * 72)
print("🛡️ CELL 3 — REPRESENTATION AND IDENTITY DISCOVERY")
print("=" * 72)

# ------------------------------------------------------------
# 1. Create independent working copies
# ------------------------------------------------------------

shield_X_ra = np.asarray(
    shield_post25_backup["X_canonical"]
).copy()

shield_labels = np.asarray(
    shield_post25_backup["category_labels"]
).astype(str).copy()

shield_image_paths = np.asarray(
    shield_post25_backup["image_paths"]
).astype(str).copy()

# Prevent accidental modification of the working copies
shield_X_ra.setflags(write=False)
shield_labels.setflags(write=False)
shield_image_paths.setflags(write=False)

# ------------------------------------------------------------
# 2. Validate dimensions and numerical integrity
# ------------------------------------------------------------

assert shield_X_ra.shape == (2300, 28), (
    f"Expected RA shape (2300, 28), "
    f"found {shield_X_ra.shape}"
)

assert shield_labels.shape == (2300,), (
    f"Expected 2300 labels, found {shield_labels.shape}"
)

assert shield_image_paths.shape == (2300,), (
    f"Expected 2300 image paths, "
    f"found {shield_image_paths.shape}"
)

if not np.isfinite(shield_X_ra).all():
    bad_count = (
        shield_X_ra.size
        - np.isfinite(shield_X_ra).sum()
    )

    raise ValueError(
        f"RA matrix contains {bad_count} non-finite values"
    )

if len(np.unique(shield_image_paths)) != 2300:
    raise ValueError(
        "The frozen image paths are not unique."
    )

shield_category_counts = (
    pd.Series(shield_labels, name="category")
    .value_counts()
    .sort_index()
)

print("\nRADIAL–ANGULAR REPRESENTATION")
print("-" * 72)
print(f"Shape                  : {shield_X_ra.shape}")
print(f"Finite values          : {np.isfinite(shield_X_ra).all()}")
print(f"Unique paths           : {len(np.unique(shield_image_paths))}")
print(f"Number of categories   : {shield_category_counts.size}")
print(
    "Samples per category   : "
    f"{shield_category_counts.min()} → "
    f"{shield_category_counts.max()}"
)

# ------------------------------------------------------------
# 3. Construct the path table
# ------------------------------------------------------------

shield_path_table = pd.DataFrame({
    "row_index": np.arange(2300),
    "category": shield_labels,
    "full_path": shield_image_paths,
})

shield_path_table["filename"] = (
    shield_path_table["full_path"]
    .map(lambda value: Path(value).name)
)

shield_path_table["stem"] = (
    shield_path_table["filename"]
    .map(lambda value: Path(value).stem)
)

shield_path_table["parent"] = (
    shield_path_table["full_path"]
    .map(lambda value: Path(value).parent.name)
)

shield_path_table["suffix"] = (
    shield_path_table["filename"]
    .map(lambda value: Path(value).suffix.lower())
)

print("\nFIRST 20 PATH RECORDS")
print("-" * 72)

with pd.option_context(
    "display.max_colwidth", 100,
    "display.width", 180,
):
    display(
        shield_path_table[
            [
                "row_index",
                "category",
                "parent",
                "filename",
                "stem",
            ]
        ].head(20)
    )

# ------------------------------------------------------------
# 4. Correct filename-token inspection
# ------------------------------------------------------------

print("\nFIRST 30 FILENAME TOKENS")
print("-" * 72)

for stem in shield_path_table["stem"].head(30):
    tokens = stem.split("-")
    print(f"{stem:12s} -> {tokens}")

# ------------------------------------------------------------
# 5. Verify the original cross-validation structure
# ------------------------------------------------------------

shield_original_cv = shield_post25_backup["cv"]
shield_original_splits = shield_post25_backup["cv_splits"]

if len(shield_original_splits) != 5:
    raise ValueError(
        f"Expected five stored folds, "
        f"found {len(shield_original_splits)}"
    )

print("\nORIGINAL CROSS-VALIDATION")
print("-" * 72)
print(f"CV object              : {shield_original_cv}")
print(f"Stored folds           : {len(shield_original_splits)}")

for fold_number, split in enumerate(
    shield_original_splits,
    start=1,
):
    train_index, test_index = split

    train_labels = shield_labels[train_index]
    test_labels = shield_labels[test_index]

    print(
        f"Fold {fold_number}: "
        f"train={len(train_index):4d}, "
        f"test={len(test_index):3d}, "
        f"train categories="
        f"{len(np.unique(train_labels)):2d}, "
        f"test categories="
        f"{len(np.unique(test_labels)):2d}"
    )

# ------------------------------------------------------------
# 6. Fast artifact search — /content only, no recursion
# ------------------------------------------------------------

shield_search_directory = Path("/content")

shield_artifact_extensions = {
    ".pkl",
    ".pickle",
    ".joblib",
    ".npz",
    ".npy",
    ".csv",
    ".parquet",
}

shield_relevance_terms = (
    "clo",
    "sket",
    "morph",
    "canonical",
    "paper1",
    "135",
    "feature",
)

shield_excluded_paths = {
    path.resolve()
    for path in shield_backup_paths.values()
}

shield_candidate_files = []

if shield_search_directory.exists():
    for candidate in shield_search_directory.iterdir():
        if not candidate.is_file():
            continue

        if candidate.resolve() in shield_excluded_paths:
            continue

        if (
            candidate.suffix.lower()
            not in shield_artifact_extensions
        ):
            continue

        candidate_name = candidate.name.lower()

        if any(
            term in candidate_name
            for term in shield_relevance_terms
        ):
            shield_candidate_files.append(
                candidate.resolve()
            )

shield_candidate_files = sorted(
    set(shield_candidate_files),
    key=lambda path: str(path),
)

print("\nCANDIDATE MORPHOLOGY ARTIFACTS")
print("-" * 72)

if not shield_candidate_files:
    print(
        "🔴 No morphology candidate found directly "
        "under /content"
    )
else:
    for index, candidate in enumerate(
        shield_candidate_files
    ):
        size_mb = (
            candidate.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"[{index:02d}] "
            f"{candidate} "
            f"({size_mb:.3f} MB)"
        )

# ------------------------------------------------------------
# 7. Completion status
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("🟢 28-D RA working copy validated and locked")
print("🟢 Labels and paths validated")
print("🟢 Original five folds recovered")
print("🟢 Fast non-recursive artifact search completed")
print("🟢 No frozen object was modified")
print("=" * 72)

🛡️ CELL 3 — REPRESENTATION AND IDENTITY DISCOVERY

RADIAL–ANGULAR REPRESENTATION
------------------------------------------------------------------------
Shape                  : (2300, 28)
Finite values          : True
Unique paths           : 2300
Number of categories   : 23
Samples per category   : 100 → 100

FIRST 20 PATH RECORDS
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,row_index,category,parent,filename,stem
0,0,A-Line,A-Line,1-1.tif,1-1
1,1,A-Line,A-Line,1-10.tif,1-10
2,2,A-Line,A-Line,1-2.tif,1-2
3,3,A-Line,A-Line,1-3.tif,1-3
4,4,A-Line,A-Line,1-4.tif,1-4
5,5,A-Line,A-Line,1-5.tif,1-5
6,6,A-Line,A-Line,1-6.tif,1-6
7,7,A-Line,A-Line,1-7.tif,1-7
8,8,A-Line,A-Line,1-8.tif,1-8
9,9,A-Line,A-Line,1-9.tif,1-9



FIRST 30 FILENAME TOKENS
------------------------------------------------------------------------
1-1          -> ['1', '1']
1-10         -> ['1', '10']
1-2          -> ['1', '2']
1-3          -> ['1', '3']
1-4          -> ['1', '4']
1-5          -> ['1', '5']
1-6          -> ['1', '6']
1-7          -> ['1', '7']
1-8          -> ['1', '8']
1-9          -> ['1', '9']
10-1         -> ['10', '1']
10-10        -> ['10', '10']
10-2         -> ['10', '2']
10-3         -> ['10', '3']
10-4         -> ['10', '4']
10-5         -> ['10', '5']
10-6         -> ['10', '6']
10-7         -> ['10', '7']
10-8         -> ['10', '8']
10-9         -> ['10', '9']
2-1          -> ['2', '1']
2-10         -> ['2', '10']
2-2          -> ['2', '2']
2-3          -> ['2', '3']
2-4          -> ['2', '4']
2-5          -> ['2', '5']
2-6          -> ['2', '6']
2-7          -> ['2', '7']
2-8          -> ['2', '8']
2-9          -> ['2', '9']

ORIGINAL CROSS-VALIDATION
---------------------------------------------------

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ============================================================
# CELL 4 — SOURCE IDENTITY RECONSTRUCTION
#          AND ORIGINAL-FOLD OVERLAP AUDIT
#
# This version preserves the observed unbalanced replication
# structure. It makes no 10-replicate factorial assumption.
# ============================================================

print("=" * 72)
print("🛡️ CELL 4 — SOURCE IDENTITY RECONSTRUCTION")
print("=" * 72)

# ------------------------------------------------------------
# 1. Parse:
#
#     source_ID <non-numeric separator> replicate_ID
#
# Observed separators are audited, not normalized in the
# original filenames.
# ------------------------------------------------------------

shield_filename_pattern = re.compile(
    r"^(?P<source_id>\d+)"
    r"(?P<separator>\D+)"
    r"(?P<replicate_id>\d+)$"
)

shield_parsed_records = []
shield_unmatched_records = []

for row in shield_path_table.itertuples(index=False):
    stem = str(row.stem).strip()
    match = shield_filename_pattern.fullmatch(stem)

    if match is None:
        shield_unmatched_records.append({
            "row_index": int(row.row_index),
            "category": str(row.category),
            "filename": str(row.filename),
            "stem": stem,
        })
        continue

    shield_parsed_records.append({
        "row_index": int(row.row_index),
        "category": str(row.category),
        "source_id_within_category": int(
            match.group("source_id")
        ),
        "replicate_id": int(
            match.group("replicate_id")
        ),
        "separator": match.group("separator"),
        "filename": str(row.filename),
        "full_path": str(row.full_path),
    })

if shield_unmatched_records:
    shield_unmatched_table = pd.DataFrame(
        shield_unmatched_records
    )

    print("\nUNMATCHED FILENAMES")
    print("-" * 72)
    display(shield_unmatched_table)

    raise ValueError(
        f"{len(shield_unmatched_records)} filename(s) "
        "could not be parsed."
    )

shield_identity_table = (
    pd.DataFrame(shield_parsed_records)
    .sort_values("row_index")
    .reset_index(drop=True)
)

assert len(shield_identity_table) == 2300
assert np.array_equal(
    shield_identity_table["row_index"].to_numpy(),
    np.arange(2300),
)

# ------------------------------------------------------------
# 2. Preserve the original filenames but construct a stable
#    analytical identity:
#
#    garment identity = category + source ID
# ------------------------------------------------------------

shield_identity_table["garment_identity"] = (
    shield_identity_table["category"]
    + "::"
    + shield_identity_table[
        "source_id_within_category"
    ].astype(str)
)

shield_groups = shield_identity_table[
    "garment_identity"
].to_numpy(dtype=str)

shield_replicates = shield_identity_table[
    "replicate_id"
].to_numpy(dtype=int)

shield_groups.setflags(write=False)
shield_replicates.setflags(write=False)

# ------------------------------------------------------------
# 3. Describe the observed replication structure
# ------------------------------------------------------------

shield_separator_counts = (
    shield_identity_table["separator"]
    .map(repr)
    .value_counts()
    .sort_index()
)

shield_group_sizes = (
    shield_identity_table
    .groupby("garment_identity")
    .size()
)

shield_identities_per_category = (
    shield_identity_table
    .groupby("category")["garment_identity"]
    .nunique()
)

shield_replicate_counts = (
    shield_identity_table["replicate_id"]
    .value_counts()
    .sort_index()
)

shield_unique_replicates_per_identity = (
    shield_identity_table
    .groupby("garment_identity")["replicate_id"]
    .nunique()
)

shield_duplicate_mask = (
    shield_identity_table
    .duplicated(
        subset=[
            "garment_identity",
            "replicate_id",
        ],
        keep=False,
    )
)

shield_duplicate_identity_replicates = (
    shield_identity_table.loc[
        shield_duplicate_mask,
        [
            "row_index",
            "category",
            "garment_identity",
            "replicate_id",
            "separator",
            "filename",
        ],
    ]
    .sort_values(
        [
            "garment_identity",
            "replicate_id",
            "filename",
        ]
    )
    .reset_index(drop=True)
)

shield_nonstandard_separator_rows = (
    shield_identity_table.loc[
        ~shield_identity_table[
            "separator"
        ].isin(["-", "_"]),
        [
            "row_index",
            "category",
            "garment_identity",
            "replicate_id",
            "separator",
            "filename",
        ],
    ]
    .reset_index(drop=True)
)

shield_unbalanced_groups = (
    shield_group_sizes[
        shield_group_sizes !=
        shield_group_sizes.median()
    ]
    .rename("n_images")
    .reset_index()
)

print("\nFILENAME SEPARATOR AUDIT")
print("-" * 72)

for separator, count in shield_separator_counts.items():
    print(f"{separator:20s}: {count:4d}")

print("\nOBSERVED DATASET STRUCTURE")
print("-" * 72)
print(f"Rows                         : {len(shield_identity_table)}")
print(
    f"Categories                   : "
    f"{shield_identity_table['category'].nunique()}"
)
print(
    f"Source garment identities    : "
    f"{shield_identity_table['garment_identity'].nunique()}"
)
print(
    f"Replicate identifiers        : "
    f"{shield_identity_table['replicate_id'].nunique()}"
)
print(
    f"Images per identity          : "
    f"{shield_group_sizes.min()} → "
    f"{shield_group_sizes.max()}"
)
print(
    f"Median images per identity   : "
    f"{shield_group_sizes.median():.1f}"
)
print(
    f"Identities per category      : "
    f"{shield_identities_per_category.min()} → "
    f"{shield_identities_per_category.max()}"
)
print(
    f"Unique replicates/identity   : "
    f"{shield_unique_replicates_per_identity.min()} → "
    f"{shield_unique_replicates_per_identity.max()}"
)
print(
    f"Images per replicate ID      : "
    f"{shield_replicate_counts.min()} → "
    f"{shield_replicate_counts.max()}"
)
print(
    f"Rows in repeated identity/"
    f"replicate combinations       : "
    f"{len(shield_duplicate_identity_replicates)}"
)

# ------------------------------------------------------------
# 4. Display—not alter—unusual provenance records
# ------------------------------------------------------------

print("\nNONSTANDARD-SEPARATOR RECORDS")
print("-" * 72)

if shield_nonstandard_separator_rows.empty:
    print("None")
else:
    display(shield_nonstandard_separator_rows)

print("\nREPLICATE-ID COUNTS")
print("-" * 72)
display(
    shield_replicate_counts
    .rename("n_images")
    .to_frame()
)

print("\nREPEATED IDENTITY–REPLICATE RECORDS")
print("-" * 72)

if shield_duplicate_identity_replicates.empty:
    print("None")
else:
    display(shield_duplicate_identity_replicates)

print("\nNON-MEDIAN IDENTITY GROUP SIZES")
print("-" * 72)

if shield_unbalanced_groups.empty:
    print("None")
else:
    display(shield_unbalanced_groups)

# ------------------------------------------------------------
# 5. Structural assertions supported by the actual data
# ------------------------------------------------------------

assert shield_identity_table["category"].nunique() == 23

assert (
    shield_identity_table["garment_identity"].nunique()
    == 230
)

assert (shield_identities_per_category == 10).all()

assert shield_group_sizes.min() >= 2

assert not shield_identity_table[
    ["category", "garment_identity", "filename"]
].isna().any().any()

assert len(np.unique(shield_image_paths)) == 2300

print(
    "\n🟢 Core design verified: "
    "23 categories and 230 source garment identities"
)
print(
    "🟡 Replication is observed to be unbalanced; "
    "no factorial assumption imposed"
)

# ------------------------------------------------------------
# 6. Audit source-identity overlap in original folds
# ------------------------------------------------------------

shield_original_fold_audit_records = []

print("\nORIGINAL-FOLD SOURCE-IDENTITY OVERLAP")
print("-" * 72)

for fold_number, (
    train_index,
    test_index,
) in enumerate(
    shield_original_splits,
    start=1,
):
    train_groups = set(shield_groups[train_index])
    test_groups = set(shield_groups[test_index])

    overlapping_groups = (
        train_groups.intersection(test_groups)
    )

    test_seen_mask = np.isin(
        shield_groups[test_index],
        list(train_groups),
    )

    test_rows_with_seen_identity = int(
        test_seen_mask.sum()
    )

    shield_original_fold_audit_records.append({
        "fold": fold_number,
        "train_rows": len(train_index),
        "test_rows": len(test_index),
        "train_identities": len(train_groups),
        "test_identities": len(test_groups),
        "overlapping_identities": len(
            overlapping_groups
        ),
        "test_rows_with_seen_identity": (
            test_rows_with_seen_identity
        ),
        "test_fraction_seen_identity": (
            test_rows_with_seen_identity
            / len(test_index)
        ),
    })

    print(
        f"Fold {fold_number}: "
        f"train identities={len(train_groups):3d}, "
        f"test identities={len(test_groups):3d}, "
        f"overlap={len(overlapping_groups):3d}, "
        f"seen test rows="
        f"{test_rows_with_seen_identity:3d}/"
        f"{len(test_index)}"
    )

shield_original_fold_audit = pd.DataFrame(
    shield_original_fold_audit_records
)

print("\n" + "=" * 72)
print("🟢 Source garment identities reconstructed")
print("🟢 Observed replication irregularities preserved and audited")
print("🟢 Original-fold source-identity overlap quantified")
print("🟢 Exact grouped-fold construction deferred to Cell 4B")
print("🟢 No frozen result object was modified")
print("=" * 72)

🛡️ CELL 4 — SOURCE IDENTITY RECONSTRUCTION

FILENAME SEPARATOR AUDIT
------------------------------------------------------------------------
'+'                 :    1
'-'                 : 1211
'_'                 : 1088

OBSERVED DATASET STRUCTURE
------------------------------------------------------------------------
Rows                         : 2300
Categories                   : 23
Source garment identities    : 230
Replicate identifiers        : 12
Images per identity          : 9 → 11
Median images per identity   : 10.0
Identities per category      : 10 → 10
Unique replicates/identity   : 9 → 11
Images per replicate ID      : 1 → 232
Rows in repeated identity/replicate combinations       : 8

NONSTANDARD-SEPARATOR RECORDS
------------------------------------------------------------------------


,row_index,category,garment_identity,replicate_id,separator,filename
0,1450,Sarong,Sarong::5,8,+,5+8.tif



REPLICATE-ID COUNTS
------------------------------------------------------------------------


,n_images
replicate_id,
1,230
2,230
3,230
4,228
5,229
6,232
7,229
8,230
9,230



REPEATED IDENTITY–REPLICATE RECORDS
------------------------------------------------------------------------


,row_index,category,garment_identity,replicate_id,separator,filename
0,500,Dress,Dress::1,3,-,1-3.tif
1,514,Dress,Dress::1,3,_,1_3.tif
2,754,Harem,Harem::5,6,-,5-6.tif
3,758,Harem,Harem::5,6,_,5_6.tif
4,765,Harem,Harem::6,6,-,6-6.tif
5,769,Harem,Harem::6,6,_,6_6.tif
6,885,Hoodie,Hoodie::8,6,-,8-6.tif
7,889,Hoodie,Hoodie::8,6,_,8_6.tif



NON-MEDIAN IDENTITY GROUP SIZES
------------------------------------------------------------------------


,garment_identity,n_images
0,Dress::1,11
1,Dress::10,9
2,Harem::2,9
3,Harem::6,11
4,Jumpsuit::2,9
5,Jumpsuit::6,11



🟢 Core design verified: 23 categories and 230 source garment identities
🟡 Replication is observed to be unbalanced; no factorial assumption imposed

ORIGINAL-FOLD SOURCE-IDENTITY OVERLAP
------------------------------------------------------------------------
Fold 1: train identities=230, test identities=209, overlap=209, seen test rows=460/460
Fold 2: train identities=230, test identities=204, overlap=204, seen test rows=460/460
Fold 3: train identities=230, test identities=212, overlap=212, seen test rows=460/460
Fold 4: train identities=230, test identities=209, overlap=209, seen test rows=460/460
Fold 5: train identities=230, test identities=213, overlap=213, seen test rows=460/460

🟢 Source garment identities reconstructed
🟢 Observed replication irregularities preserved and audited
🟢 Original-fold source-identity overlap quantified
🟢 Exact grouped-fold construction deferred to Cell 4B
🟢 No frozen result object was modified


In [ ]:
# ============================================================
# CELL 4B — EXACT CATEGORY-BALANCED GROUPED FOLDS
#
# Two source garment identities from every category are
# assigned to each of five test folds.
# ============================================================

print("=" * 72)
print("🛡️ CELL 4B — EXACT BALANCED GROUPED FOLDS")
print("=" * 72)

shield_fold_rng = np.random.default_rng(
    SHIELD_RANDOM_STATE
)

shield_categories_sorted = np.sort(
    shield_identity_table["category"].unique()
)

# One set of test identities for each fold
shield_test_groups_by_fold = [
    set() for _ in range(SHIELD_N_SPLITS)
]

shield_category_fold_assignment_records = []

# ------------------------------------------------------------
# 1. Allocate exactly two identities per category per fold
# ------------------------------------------------------------

for category in shield_categories_sorted:
    category_groups = np.sort(
        shield_identity_table.loc[
            shield_identity_table["category"] == category,
            "garment_identity",
        ].unique()
    )

    if len(category_groups) != 10:
        raise ValueError(
            f"Category {category!r} has "
            f"{len(category_groups)} identities; expected 10."
        )

    shuffled_groups = shield_fold_rng.permutation(
        category_groups
    )

    category_chunks = np.array_split(
        shuffled_groups,
        SHIELD_N_SPLITS,
    )

    for fold_index, fold_groups in enumerate(
        category_chunks
    ):
        if len(fold_groups) != 2:
            raise ValueError(
                f"Category {category!r}, fold "
                f"{fold_index + 1}: expected 2 identities, "
                f"found {len(fold_groups)}."
            )

        shield_test_groups_by_fold[
            fold_index
        ].update(fold_groups.tolist())

        for garment_identity in fold_groups:
            shield_category_fold_assignment_records.append({
                "category": category,
                "fold": fold_index + 1,
                "garment_identity": garment_identity,
            })

shield_category_fold_assignment = pd.DataFrame(
    shield_category_fold_assignment_records
)

# ------------------------------------------------------------
# 2. Convert identity assignments into row-index splits
# ------------------------------------------------------------

shield_group_splits = []
shield_group_fold_audit_records = []

all_indices = np.arange(2300)

print("\nEXACT GROUPED-FOLD AUDIT")
print("-" * 72)

for fold_index, test_group_set in enumerate(
    shield_test_groups_by_fold
):
    fold_number = fold_index + 1

    test_mask = np.isin(
        shield_groups,
        list(test_group_set),
    )

    test_index = all_indices[test_mask]
    train_index = all_indices[~test_mask]

    train_groups = set(shield_groups[train_index])
    test_groups = set(shield_groups[test_index])

    overlapping_groups = (
        train_groups.intersection(test_groups)
    )

    train_category_counts = (
        pd.Series(shield_labels[train_index])
        .value_counts()
        .sort_index()
    )

    test_category_counts = (
        pd.Series(shield_labels[test_index])
        .value_counts()
        .sort_index()
    )

    test_identities_per_category = (
        shield_identity_table.iloc[test_index]
        .groupby("category")[
            "garment_identity"
        ]
        .nunique()
        .sort_index()
    )

    # Strict fold assertions
    assert len(train_index) + len(test_index) == 2300
    assert len(overlapping_groups) == 0

    assert len(train_groups) == 184
    assert len(test_groups) == 46

    assert len(train_category_counts) == 23
    assert len(test_category_counts) == 23

    assert (
        test_identities_per_category == 2
    ).all()

    shield_group_splits.append(
        (
            train_index.copy(),
            test_index.copy(),
        )
    )

    shield_group_fold_audit_records.append({
        "fold": fold_number,
        "train_rows": len(train_index),
        "test_rows": len(test_index),
        "train_identities": len(train_groups),
        "test_identities": len(test_groups),
        "overlapping_identities": len(
            overlapping_groups
        ),
        "test_categories": len(
            test_category_counts
        ),
        "test_identities_per_category_min": int(
            test_identities_per_category.min()
        ),
        "test_identities_per_category_max": int(
            test_identities_per_category.max()
        ),
        "test_images_per_category_min": int(
            test_category_counts.min()
        ),
        "test_images_per_category_max": int(
            test_category_counts.max()
        ),
    })

    print(
        f"Fold {fold_number}: "
        f"train rows={len(train_index):4d}, "
        f"test rows={len(test_index):3d}, "
        f"train identities={len(train_groups):3d}, "
        f"test identities={len(test_groups):2d}, "
        f"overlap={len(overlapping_groups)}, "
        f"categories={len(test_category_counts)}, "
        f"identities/category="
        f"{test_identities_per_category.min()}→"
        f"{test_identities_per_category.max()}, "
        f"images/category="
        f"{test_category_counts.min()}→"
        f"{test_category_counts.max()}"
    )

shield_group_fold_audit = pd.DataFrame(
    shield_group_fold_audit_records
)

# ------------------------------------------------------------
# 3. Verify complete row and identity coverage
# ------------------------------------------------------------

shield_test_row_coverage = np.zeros(
    2300,
    dtype=int,
)

shield_test_identity_coverage = Counter()

for _, test_index in shield_group_splits:
    shield_test_row_coverage[test_index] += 1

    for garment_identity in np.unique(
        shield_groups[test_index]
    ):
        shield_test_identity_coverage[
            garment_identity
        ] += 1

assert np.all(shield_test_row_coverage == 1)

assert len(shield_test_identity_coverage) == 230

assert all(
    count == 1
    for count in shield_test_identity_coverage.values()
)

# ------------------------------------------------------------
# 4. Lock split indices against accidental modification
# ------------------------------------------------------------

for train_index, test_index in shield_group_splits:
    train_index.setflags(write=False)
    test_index.setflags(write=False)

print("\nGROUPED DESIGN SUMMARY")
print("-" * 72)
display(shield_group_fold_audit)

print("\n" + "=" * 72)
print("🟢 Five exact category-balanced grouped folds created")
print("🟢 Every fold contains all 23 categories")
print("🟢 Every fold tests two identities per category")
print("🟢 No source identity crosses train and test")
print("🟢 Every row is tested exactly once")
print("🟢 Every identity is tested exactly once")
print("=" * 72)

🛡️ CELL 4B — EXACT BALANCED GROUPED FOLDS

EXACT GROUPED-FOLD AUDIT
------------------------------------------------------------------------
Fold 1: train rows=1839, test rows=461, train identities=184, test identities=46, overlap=0, categories=23, identities/category=2→2, images/category=20→21
Fold 2: train rows=1840, test rows=460, train identities=184, test identities=46, overlap=0, categories=23, identities/category=2→2, images/category=20→20
Fold 3: train rows=1841, test rows=459, train identities=184, test identities=46, overlap=0, categories=23, identities/category=2→2, images/category=19→20
Fold 4: train rows=1840, test rows=460, train identities=184, test identities=46, overlap=0, categories=23, identities/category=2→2, images/category=19→21
Fold 5: train rows=1840, test rows=460, train identities=184, test identities=46, overlap=0, categories=23, identities/category=2→2, images/category=20→20

GROUPED DESIGN SUMMARY
------------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,fold,train_rows,test_rows,train_identities,test_identities,overlapping_identities,test_categories,test_identities_per_category_min,test_identities_per_category_max,test_images_per_category_min,test_images_per_category_max
0,1,1839,461,184,46,0,23,2,2,20,21
1,2,1840,460,184,46,0,23,2,2,20,20
2,3,1841,459,184,46,0,23,2,2,19,20
3,4,1840,460,184,46,0,23,2,2,19,21
4,5,1840,460,184,46,0,23,2,2,20,20



🟢 Five exact category-balanced grouped folds created
🟢 Every fold contains all 23 categories
🟢 Every fold tests two identities per category
🟢 No source identity crosses train and test
🟢 Every row is tested exactly once
🟢 Every identity is tested exactly once


In [ ]:
# ============================================================
# CELL 5 — LOCATE THE FROZEN 135-D MORPHOLOGY ARTIFACT
# ============================================================

print("=" * 72)
print("🛡️ CELL 5 — MORPHOLOGY ARTIFACT DISCOVERY")
print("=" * 72)

SHIELD_EXPECTED_MORPHOLOGY_SHA256 = (
    "66ae04156ee3fbf3f2605f382a16fc41"
    "cf19af34b50e59dd43f6c9427d96b2ee"
)

# ------------------------------------------------------------
# 1. Check whether X_raw already exists in notebook memory
# ------------------------------------------------------------

shield_x_raw_in_memory = (
    "X_raw" in globals()
)

print("\nIN-MEMORY CHECK")
print("-" * 72)
print(f"X_raw present: {shield_x_raw_in_memory}")

if shield_x_raw_in_memory:
    in_memory_shape = getattr(
        globals()["X_raw"],
        "shape",
        None,
    )

    print(f"X_raw shape  : {in_memory_shape}")

# ------------------------------------------------------------
# 2. Inventory candidate files directly under /content
# ------------------------------------------------------------

shield_content_files = []

for candidate in Path("/content").iterdir():
    if not candidate.is_file():
        continue

    shield_content_files.append({
        "filename": candidate.name,
        "path": str(candidate.resolve()),
        "suffix": candidate.suffix.lower(),
        "size_mb": (
            candidate.stat().st_size
            / (1024 ** 2)
        ),
    })

shield_content_inventory = (
    pd.DataFrame(shield_content_files)
    .sort_values(
        ["suffix", "filename"],
        kind="stable",
    )
    .reset_index(drop=True)
)

shield_supported_suffixes = {
    ".pkl",
    ".pickle",
    ".joblib",
    ".npz",
    ".npy",
    ".parquet",
    ".csv",
}

shield_candidate_inventory = (
    shield_content_inventory.loc[
        shield_content_inventory[
            "suffix"
        ].isin(shield_supported_suffixes)
    ]
    .reset_index(drop=True)
)

print("\nSERIALIZED FILES DIRECTLY UNDER /content")
print("-" * 72)

if shield_candidate_inventory.empty:
    print("None found")
else:
    with pd.option_context(
        "display.max_colwidth", 120,
        "display.width", 180,
    ):
        display(shield_candidate_inventory)

# ------------------------------------------------------------
# 3. Highlight likely morphology candidates
# ------------------------------------------------------------

shield_candidate_terms = (
    "morph",
    "canonical",
    "paper1",
    "135",
    "checkpoint",
    "final",
    "clo",
    "sket",
)

if shield_candidate_inventory.empty:
    shield_likely_morphology_candidates = (
        shield_candidate_inventory.copy()
    )
else:
    candidate_mask = (
        shield_candidate_inventory["filename"]
        .str.lower()
        .map(
            lambda name: any(
                term in name
                for term in shield_candidate_terms
            )
        )
    )

    shield_likely_morphology_candidates = (
        shield_candidate_inventory.loc[
            candidate_mask
        ]
        .reset_index(drop=True)
    )

print("\nLIKELY MORPHOLOGY CANDIDATES")
print("-" * 72)

if shield_likely_morphology_candidates.empty:
    print("🔴 No likely morphology artifact found")
else:
    with pd.option_context(
        "display.max_colwidth", 120,
        "display.width", 180,
    ):
        display(shield_likely_morphology_candidates)

# ------------------------------------------------------------
# 4. Do not load candidates yet
# ------------------------------------------------------------

print("\nEXPECTED FROZEN MORPHOLOGY")
print("-" * 72)
print("Object name : X_raw")
print("Shape       : (2300, 135)")
print("Dtype       : float32")
print(
    "SHA-256    : "
    f"{SHIELD_EXPECTED_MORPHOLOGY_SHA256}"
)

print("\n" + "=" * 72)
print("🟢 Grouped folds remain frozen in shield_group_splits")
print("🟢 Candidate files inventoried without loading")
print("🟢 No artifact was modified")
print("=" * 72)

🛡️ CELL 5 — MORPHOLOGY ARTIFACT DISCOVERY

IN-MEMORY CHECK
------------------------------------------------------------------------
X_raw present: False

SERIALIZED FILES DIRECTLY UNDER /content
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,filename,path,suffix,size_mb
0,CLO_SKET_runtime_backup.pkl,/content/CLO_SKET_runtime_backup.pkl,.pkl,132.749985
1,CLO_SKET_runtime_backup_AFTER_CELL25.pkl,/content/CLO_SKET_runtime_backup_AFTER_CELL25.pkl,.pkl,140.248221



LIKELY MORPHOLOGY CANDIDATES
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,filename,path,suffix,size_mb
0,CLO_SKET_runtime_backup.pkl,/content/CLO_SKET_runtime_backup.pkl,.pkl,132.749985
1,CLO_SKET_runtime_backup_AFTER_CELL25.pkl,/content/CLO_SKET_runtime_backup_AFTER_CELL25.pkl,.pkl,140.248221



EXPECTED FROZEN MORPHOLOGY
------------------------------------------------------------------------
Object name : X_raw
Shape       : (2300, 135)
Dtype       : float32
SHA-256    : 66ae04156ee3fbf3f2605f382a16fc41cf19af34b50e59dd43f6c9427d96b2ee

🟢 Grouped folds remain frozen in shield_group_splits
🟢 Candidate files inventoried without loading
🟢 No artifact was modified


In [ ]:
# ============================================================
# CELL 5B — EXACT FROZEN 135-D MORPHOLOGY RECONSTRUCTION
#
# Extractor copied exactly from 01_morphology_discovery.ipynb.
# Images are processed in the frozen RA row order.
# The result is accepted only if its SHA-256 matches the
# previously recorded canonical fingerprint.
# ============================================================

from PIL import Image

print("=" * 72)
print("🛡️ CELL 5B — EXACT MORPHOLOGY RECONSTRUCTION")
print("=" * 72)

SHIELD_EXPECTED_MORPHOLOGY_SHA256 = (
    "66ae04156ee3fbf3f2605f382a16fc41"
    "cf19af34b50e59dd43f6c9427d96b2ee"
)

# ------------------------------------------------------------
# 1. Resolve frozen source paths
# ------------------------------------------------------------

shield_source_paths = [
    Path(path)
    for path in shield_image_paths
]

shield_missing_source_paths = [
    str(path)
    for path in shield_source_paths
    if not path.exists()
]

print("\nSOURCE-PATH CHECK")
print("-" * 72)
print(f"Frozen paths            : {len(shield_source_paths)}")
print(f"Existing paths          : "
      f"{len(shield_source_paths) - len(shield_missing_source_paths)}")
print(f"Missing paths           : {len(shield_missing_source_paths)}")

if shield_missing_source_paths:
    print("\nFirst missing paths:")

    for path in shield_missing_source_paths[:10]:
        print(path)

    raise FileNotFoundError(
        "The original TIFF paths are not currently accessible. "
        "Mount the Drive containing the CLO-SKET dataset and "
        "rerun Cell 5B."
    )

if len(set(map(str, shield_source_paths))) != 2300:
    raise ValueError(
        "Frozen source-path sequence is not unique."
    )

print("🟢 All frozen TIFF paths are accessible")

# ------------------------------------------------------------
# 2. Exact frozen feature definition
# ------------------------------------------------------------

def shield_morphology_features(path, size=64):
    image = Image.open(path).convert("L")

    array = np.asarray(
        image,
        dtype=np.float32,
    )

    # Exact frozen intensity normalization
    array = array / 255.0

    # Exact frozen foreground definition
    foreground = array < 0.8

    # Exact frozen resize operation
    foreground_image = Image.fromarray(
        foreground.astype(np.uint8) * 255
    ).resize(
        (size, size)
    )

    mask = (
        np.asarray(
            foreground_image,
            dtype=np.float32,
        )
        / 255.0
    )

    # 64 horizontal occupancy coordinates
    horizontal = mask.mean(axis=1)

    # 64 vertical occupancy coordinates
    vertical = mask.mean(axis=0)

    # Seven global descriptors
    total = mask.sum() + 1e-8

    yy, xx = np.indices(mask.shape)

    centroid_x = (
        (xx * mask).sum()
        / total
    )

    centroid_y = (
        (yy * mask).sum()
        / total
    )

    centroid_x /= size
    centroid_y /= size

    ys, xs = np.where(mask > 0)

    if len(xs) > 0:
        bbox_width = (
            xs.max()
            - xs.min()
            + 1
        ) / size

        bbox_height = (
            ys.max()
            - ys.min()
            + 1
        ) / size

        aspect_ratio = (
            bbox_width
            / (bbox_height + 1e-8)
        )
    else:
        bbox_width = 0.0
        bbox_height = 0.0
        aspect_ratio = 0.0

    flipped = np.fliplr(mask)

    symmetry = (
        1.0
        - np.mean(
            np.abs(
                mask - flipped
            )
        )
    )

    features = np.concatenate([
        horizontal,
        vertical,
        np.array([
            centroid_x,
            centroid_y,
            bbox_width,
            bbox_height,
            aspect_ratio,
            symmetry,
            mask.mean(),
        ]),
    ])

    return features.astype(np.float32)

# ------------------------------------------------------------
# 3. Verify the extractor before full reconstruction
# ------------------------------------------------------------

shield_test_vector = shield_morphology_features(
    shield_source_paths[0]
)

if shield_test_vector.shape != (135,):
    raise ValueError(
        f"Extractor returned {shield_test_vector.shape}; "
        "expected (135,)."
    )

if not np.isfinite(shield_test_vector).all():
    raise ValueError(
        "Test morphology vector contains non-finite values."
    )

print("\nEXTRACTOR CHECK")
print("-" * 72)
print(f"Test image             : {shield_source_paths[0]}")
print(f"Test-vector shape      : {shield_test_vector.shape}")
print(f"Test-vector dtype      : {shield_test_vector.dtype}")
print(f"Test-vector finite     : {np.isfinite(shield_test_vector).all()}")
print("🟢 Exact 135-D feature definition verified")

# ------------------------------------------------------------
# 4. Reconstruct in frozen RA row order
# ------------------------------------------------------------

shield_morphology_rows = []
shield_morphology_failures = []

print("\nRECONSTRUCTING 2300 × 135 MORPHOLOGY")
print("-" * 72)

for row_index, path in enumerate(
    shield_source_paths
):
    try:
        vector = shield_morphology_features(path)

        if vector.shape != (135,):
            raise ValueError(
                f"Unexpected vector shape {vector.shape}"
            )

        if not np.isfinite(vector).all():
            raise ValueError(
                "Non-finite feature value"
            )

        shield_morphology_rows.append(vector)

    except Exception as exc:
        shield_morphology_failures.append({
            "row_index": row_index,
            "path": str(path),
            "error": repr(exc),
        })

    if (
        (row_index + 1) % 250 == 0
        or row_index + 1 == 2300
    ):
        print(
            f"processed {row_index + 1}/2300"
        )

if shield_morphology_failures:
    shield_morphology_failure_table = pd.DataFrame(
        shield_morphology_failures
    )

    display(shield_morphology_failure_table)

    raise RuntimeError(
        f"Morphology reconstruction failed for "
        f"{len(shield_morphology_failures)} image(s)."
    )

shield_X_morphology = np.ascontiguousarray(
    np.vstack(shield_morphology_rows),
    dtype=np.float32,
)

# ------------------------------------------------------------
# 5. Exact canonical fingerprint
# ------------------------------------------------------------

shield_morphology_sha256 = hashlib.sha256(
    shield_X_morphology.tobytes()
).hexdigest()

print("\nCANONICAL MORPHOLOGY AUDIT")
print("-" * 72)
print(f"Shape                  : {shield_X_morphology.shape}")
print(f"Dtype                  : {shield_X_morphology.dtype}")
print(f"Finite                 : {np.isfinite(shield_X_morphology).all()}")
print(f"Failures               : {len(shield_morphology_failures)}")
print(f"Observed SHA-256       : {shield_morphology_sha256}")
print(f"Expected SHA-256       : {SHIELD_EXPECTED_MORPHOLOGY_SHA256}")

if shield_X_morphology.shape != (2300, 135):
    raise ValueError(
        f"Expected shape (2300, 135), "
        f"found {shield_X_morphology.shape}"
    )

if not np.isfinite(shield_X_morphology).all():
    raise ValueError(
        "Reconstructed morphology contains non-finite values."
    )

if (
    shield_morphology_sha256
    != SHIELD_EXPECTED_MORPHOLOGY_SHA256
):
    raise RuntimeError(
        "Morphology fingerprint mismatch. "
        "The reconstructed matrix will not be used."
    )

# Lock accepted matrix
shield_X_morphology.setflags(write=False)

print("\n" + "=" * 72)
print("🟢 EXACT CANONICAL MORPHOLOGY RESTORED")
print("🟢 Recorded SHA-256 reproduced exactly")
print("🟢 Morphology and RA use the same frozen row order")
print("🟢 135-D morphology matrix locked read-only")
print("🟢 No original artifact or image was modified")
print("=" * 72)

🛡️ CELL 5B — EXACT MORPHOLOGY RECONSTRUCTION

SOURCE-PATH CHECK
------------------------------------------------------------------------
Frozen paths            : 2300
Existing paths          : 2300
Missing paths           : 0
🟢 All frozen TIFF paths are accessible

EXTRACTOR CHECK
------------------------------------------------------------------------
Test image             : /content/drive/MyDrive/FashionAI/datasets/Clo-Sket/Clo-Sket/A-Line/1-1.tif
Test-vector shape      : (135,)
Test-vector dtype      : float32
Test-vector finite     : True
🟢 Exact 135-D feature definition verified

RECONSTRUCTING 2300 × 135 MORPHOLOGY
------------------------------------------------------------------------
processed 250/2300
processed 500/2300
processed 750/2300
processed 1000/2300
processed 1250/2300
processed 1500/2300
processed 1750/2300
processed 2000/2300
processed 2250/2300
processed 2300/2300

CANONICAL MORPHOLOGY AUDIT
-----------------------------------------------------------------------

In [ ]:
# ============================================================
# CELL 6 — ORIGINAL-SPLIT REPRODUCTION AND
#          UNSEEN-GARMENT COMPLEMENTARITY TEST
# ============================================================

from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import classification_report
from sklearn.utils.validation import check_is_fitted

print("=" * 72)
print("🛡️ CELL 6 — GROUPED COMPLEMENTARITY TEST")
print("=" * 72)

# ------------------------------------------------------------
# 1. Freeze predictor matrices
# ------------------------------------------------------------

shield_X_morphology_work = np.asarray(
    shield_X_morphology,
    dtype=np.float64,
)

shield_X_ra_work = np.asarray(
    shield_X_ra,
    dtype=np.float64,
)

shield_X_combined = np.column_stack([
    shield_X_morphology_work,
    shield_X_ra_work,
])

assert shield_X_morphology_work.shape == (2300, 135)
assert shield_X_ra_work.shape == (2300, 28)
assert shield_X_combined.shape == (2300, 163)

assert np.isfinite(shield_X_morphology_work).all()
assert np.isfinite(shield_X_ra_work).all()
assert np.isfinite(shield_X_combined).all()

# ------------------------------------------------------------
# 2. Fixed classifier probe
#
# In sklearn 1.6+, multiclass logistic regression with lbfgs
# automatically uses the multinomial loss. We intentionally
# omit the deprecated explicit multi_class argument.
# ------------------------------------------------------------

shield_classifier_template = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            penalty="l2",
            max_iter=5000,
            tol=1e-4,
            class_weight=None,
        ),
    ),
])

shield_classifier_configuration = {
    "scaler": "StandardScaler fitted inside each training fold",
    "classifier": "LogisticRegression",
    "C": 1.0,
    "solver": "lbfgs",
    "penalty": "l2",
    "max_iter": 5000,
    "tol": 1e-4,
    "class_weight": None,
    "multiclass_loss": (
        "automatic multinomial behavior under "
        "scikit-learn 1.6.1"
    ),
}

print("\nFIXED CLASSIFIER")
print("-" * 72)

for key, value in shield_classifier_configuration.items():
    print(f"{key:22s}: {value}")

# ------------------------------------------------------------
# 3. Reusable out-of-fold evaluator
# ------------------------------------------------------------

def shield_evaluate_classifier(
    X,
    y,
    splits,
    representation_name,
    split_name,
):
    y = np.asarray(y)
    n_samples = len(y)

    predictions = np.empty(
        n_samples,
        dtype=y.dtype,
    )

    test_coverage = np.zeros(
        n_samples,
        dtype=int,
    )

    fold_records = []
    fitted_models = []

    for fold_number, (
        train_index,
        test_index,
    ) in enumerate(
        splits,
        start=1,
    ):
        model = clone(
            shield_classifier_template
        )

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter(
                "always",
                ConvergenceWarning,
            )

            model.fit(
                X[train_index],
                y[train_index],
            )

        convergence_warnings = [
            warning
            for warning in caught
            if issubclass(
                warning.category,
                ConvergenceWarning,
            )
        ]

        fold_predictions = model.predict(
            X[test_index]
        )

        predictions[test_index] = (
            fold_predictions
        )

        test_coverage[test_index] += 1

        fold_records.append({
            "split_design": split_name,
            "representation": representation_name,
            "fold": fold_number,
            "train_n": len(train_index),
            "test_n": len(test_index),
            "accuracy": accuracy_score(
                y[test_index],
                fold_predictions,
            ),
            "balanced_accuracy": (
                balanced_accuracy_score(
                    y[test_index],
                    fold_predictions,
                )
            ),
            "macro_f1": f1_score(
                y[test_index],
                fold_predictions,
                average="macro",
                zero_division=0,
            ),
            "convergence_warnings": len(
                convergence_warnings
            ),
            "iterations": int(
                np.max(
                    model.named_steps[
                        "classifier"
                    ].n_iter_
                )
            ),
        })

        fitted_models.append(model)

    if not np.all(test_coverage == 1):
        raise RuntimeError(
            f"{split_name}/{representation_name}: "
            "out-of-fold coverage is not exactly one."
        )

    pooled_metrics = {
        "split_design": split_name,
        "representation": representation_name,
        "accuracy": accuracy_score(
            y,
            predictions,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y,
                predictions,
            )
        ),
        "macro_f1": f1_score(
            y,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }

    return {
        "predictions": predictions,
        "fold_metrics": pd.DataFrame(
            fold_records
        ),
        "pooled_metrics": pooled_metrics,
        "models": fitted_models,
    }

# ------------------------------------------------------------
# 4. Reproduce the original image-level experiment
# ------------------------------------------------------------

print("\nORIGINAL IMAGE-LEVEL SPLITS")
print("-" * 72)

shield_original_morphology = (
    shield_evaluate_classifier(
        X=shield_X_morphology_work,
        y=shield_labels,
        splits=shield_original_splits,
        representation_name="Morphology",
        split_name="Original StratifiedKFold",
    )
)

shield_original_combined = (
    shield_evaluate_classifier(
        X=shield_X_combined,
        y=shield_labels,
        splits=shield_original_splits,
        representation_name="Morphology + RA",
        split_name="Original StratifiedKFold",
    )
)

# ------------------------------------------------------------
# 5. Run strict unseen-garment evaluation
# ------------------------------------------------------------

print("🟢 Original-split models complete")
print("\nSTRICT UNSEEN-GARMENT SPLITS")
print("-" * 72)

shield_grouped_morphology = (
    shield_evaluate_classifier(
        X=shield_X_morphology_work,
        y=shield_labels,
        splits=shield_group_splits,
        representation_name="Morphology",
        split_name="Unseen garment identity",
    )
)

shield_grouped_combined = (
    shield_evaluate_classifier(
        X=shield_X_combined,
        y=shield_labels,
        splits=shield_group_splits,
        representation_name="Morphology + RA",
        split_name="Unseen garment identity",
    )
)

print("🟢 Grouped-split models complete")

# ------------------------------------------------------------
# 6. Build pooled result table
# ------------------------------------------------------------

shield_classification_summary = pd.DataFrame([
    shield_original_morphology[
        "pooled_metrics"
    ],
    shield_original_combined[
        "pooled_metrics"
    ],
    shield_grouped_morphology[
        "pooled_metrics"
    ],
    shield_grouped_combined[
        "pooled_metrics"
    ],
])

print("\nPOOLED OUT-OF-FOLD PERFORMANCE")
print("-" * 72)
display(
    shield_classification_summary.round(6)
)

# ------------------------------------------------------------
# 7. Compare against frozen original reference values
# ------------------------------------------------------------

shield_frozen_reference = {
    "morphology_macro_f1": 0.341348,
    "morphology_balanced_accuracy": 0.342609,
    "combined_macro_f1": 0.412332,
    "combined_balanced_accuracy": 0.415652,
}

original_m = shield_original_morphology[
    "pooled_metrics"
]

original_c = shield_original_combined[
    "pooled_metrics"
]

shield_original_reproduction = pd.DataFrame([
    {
        "quantity": "Morphology Macro-F1",
        "reproduced": original_m["macro_f1"],
        "frozen_reference": (
            shield_frozen_reference[
                "morphology_macro_f1"
            ]
        ),
    },
    {
        "quantity": "Morphology balanced accuracy",
        "reproduced": (
            original_m["balanced_accuracy"]
        ),
        "frozen_reference": (
            shield_frozen_reference[
                "morphology_balanced_accuracy"
            ]
        ),
    },
    {
        "quantity": "Combined Macro-F1",
        "reproduced": original_c["macro_f1"],
        "frozen_reference": (
            shield_frozen_reference[
                "combined_macro_f1"
            ]
        ),
    },
    {
        "quantity": "Combined balanced accuracy",
        "reproduced": (
            original_c["balanced_accuracy"]
        ),
        "frozen_reference": (
            shield_frozen_reference[
                "combined_balanced_accuracy"
            ]
        ),
    },
])

shield_original_reproduction["difference"] = (
    shield_original_reproduction["reproduced"]
    - shield_original_reproduction[
        "frozen_reference"
    ]
)

print("\nORIGINAL-RESULT REPRODUCTION")
print("-" * 72)
display(
    shield_original_reproduction.round(9)
)

# ------------------------------------------------------------
# 8. Compute paired representation improvements
# ------------------------------------------------------------

def shield_build_delta_table(
    morphology_result,
    combined_result,
    split_name,
):
    morphology_folds = (
        morphology_result["fold_metrics"]
        .set_index("fold")
    )

    combined_folds = (
        combined_result["fold_metrics"]
        .set_index("fold")
    )

    delta_table = pd.DataFrame({
        "fold": morphology_folds.index,
        "morphology_macro_f1": (
            morphology_folds["macro_f1"]
        ),
        "combined_macro_f1": (
            combined_folds["macro_f1"]
        ),
        "delta_macro_f1": (
            combined_folds["macro_f1"]
            - morphology_folds["macro_f1"]
        ),
        "morphology_balanced_accuracy": (
            morphology_folds[
                "balanced_accuracy"
            ]
        ),
        "combined_balanced_accuracy": (
            combined_folds[
                "balanced_accuracy"
            ]
        ),
        "delta_balanced_accuracy": (
            combined_folds[
                "balanced_accuracy"
            ]
            - morphology_folds[
                "balanced_accuracy"
            ]
        ),
    }).reset_index(drop=True)

    delta_table.insert(
        0,
        "split_design",
        split_name,
    )

    return delta_table

shield_original_fold_deltas = (
    shield_build_delta_table(
        shield_original_morphology,
        shield_original_combined,
        "Original StratifiedKFold",
    )
)

shield_grouped_fold_deltas = (
    shield_build_delta_table(
        shield_grouped_morphology,
        shield_grouped_combined,
        "Unseen garment identity",
    )
)

print("\nORIGINAL FOLD-WISE DELTAS")
print("-" * 72)
display(
    shield_original_fold_deltas.round(6)
)

print("\nUNSEEN-GARMENT FOLD-WISE DELTAS")
print("-" * 72)
display(
    shield_grouped_fold_deltas.round(6)
)

# ------------------------------------------------------------
# 9. Pooled unseen-garment effect
# ------------------------------------------------------------

grouped_m = shield_grouped_morphology[
    "pooled_metrics"
]

grouped_c = shield_grouped_combined[
    "pooled_metrics"
]

shield_grouped_primary_result = {
    "morphology_macro_f1": (
        grouped_m["macro_f1"]
    ),
    "combined_macro_f1": (
        grouped_c["macro_f1"]
    ),
    "delta_macro_f1": (
        grouped_c["macro_f1"]
        - grouped_m["macro_f1"]
    ),
    "morphology_balanced_accuracy": (
        grouped_m["balanced_accuracy"]
    ),
    "combined_balanced_accuracy": (
        grouped_c["balanced_accuracy"]
    ),
    "delta_balanced_accuracy": (
        grouped_c["balanced_accuracy"]
        - grouped_m["balanced_accuracy"]
    ),
    "macro_f1_folds_improved": int(
        (
            shield_grouped_fold_deltas[
                "delta_macro_f1"
            ] > 0
        ).sum()
    ),
    "balanced_accuracy_folds_improved": int(
        (
            shield_grouped_fold_deltas[
                "delta_balanced_accuracy"
            ] > 0
        ).sum()
    ),
}

print("\nPRIMARY UNSEEN-GARMENT RESULT")
print("-" * 72)

for key, value in (
    shield_grouped_primary_result.items()
):
    if isinstance(value, float):
        print(f"{key:38s}: {value:+.6f}")
    else:
        print(f"{key:38s}: {value}")

# ------------------------------------------------------------
# 10. Convergence audit
# ------------------------------------------------------------

shield_all_fold_metrics = pd.concat([
    shield_original_morphology[
        "fold_metrics"
    ],
    shield_original_combined[
        "fold_metrics"
    ],
    shield_grouped_morphology[
        "fold_metrics"
    ],
    shield_grouped_combined[
        "fold_metrics"
    ],
], ignore_index=True)

shield_total_convergence_warnings = int(
    shield_all_fold_metrics[
        "convergence_warnings"
    ].sum()
)

print("\nCONVERGENCE AUDIT")
print("-" * 72)
print(
    f"Total convergence warnings: "
    f"{shield_total_convergence_warnings}"
)
print(
    f"Maximum iterations used    : "
    f"{shield_all_fold_metrics['iterations'].max()}"
)

print("\n" + "=" * 72)
print("🟢 Original and grouped evaluations completed")
print("🟢 Scaling occurred inside every training fold")
print("🟢 Identical classifier used for both representations")
print("🟢 All predictions are out of fold")
print("🟢 No frozen feature was modified")
print("=" * 72)

🛡️ CELL 6 — GROUPED COMPLEMENTARITY TEST

FIXED CLASSIFIER
------------------------------------------------------------------------
scaler                : StandardScaler fitted inside each training fold
classifier            : LogisticRegression
C                     : 1.0
solver                : lbfgs
penalty               : l2
max_iter              : 5000
tol                   : 0.0001
class_weight          : None
multiclass_loss       : automatic multinomial behavior under scikit-learn 1.6.1

ORIGINAL IMAGE-LEVEL SPLITS
------------------------------------------------------------------------
🟢 Original-split models complete

STRICT UNSEEN-GARMENT SPLITS
------------------------------------------------------------------------
🟢 Grouped-split models complete

POOLED OUT-OF-FOLD PERFORMANCE
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,split_design,representation,accuracy,balanced_accuracy,macro_f1
0,Original StratifiedKFold,Morphology,0.342174,0.342174,0.342322
1,Original StratifiedKFold,Morphology + RA,0.415652,0.415652,0.414257
2,Unseen garment identity,Morphology,0.307826,0.307826,0.306847
3,Unseen garment identity,Morphology + RA,0.342174,0.342174,0.341445



ORIGINAL-RESULT REPRODUCTION
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

,split_design,fold,morphology_macro_f1,combined_macro_f1,delta_macro_f1,morphology_balanced_accuracy,combined_balanced_accuracy,delta_balanced_accuracy
0,Original StratifiedKFold,1,0.328726,0.417483,0.088757,0.326087,0.419565,0.093478
1,Original StratifiedKFold,2,0.343118,0.425692,0.082574,0.343478,0.428261,0.084783
2,Original StratifiedKFold,3,0.360832,0.403813,0.042980,0.363043,0.408696,0.045652
3,Original StratifiedKFold,4,0.324048,0.356170,0.032122,0.323913,0.358696,0.034783
4,Original StratifiedKFold,5,0.348930,0.458502,0.109572,0.354348,0.463043,0.108696



UNSEEN-GARMENT FOLD-WISE DELTAS
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,split_design,fold,morphology_macro_f1,combined_macro_f1,delta_macro_f1,morphology_balanced_accuracy,combined_balanced_accuracy,delta_balanced_accuracy
0,Unseen garment identity,1,0.322061,0.343336,0.021276,0.316770,0.343168,0.026398
1,Unseen garment identity,2,0.349416,0.372376,0.022960,0.356522,0.382609,0.026087
2,Unseen garment identity,3,0.281334,0.311576,0.030242,0.287414,0.317620,0.030206
3,Unseen garment identity,4,0.280342,0.315083,0.034741,0.287093,0.322311,0.035218
4,Unseen garment identity,5,0.290411,0.337506,0.047095,0.291304,0.345652,0.054348



PRIMARY UNSEEN-GARMENT RESULT
------------------------------------------------------------------------
morphology_macro_f1                   : +0.306847
combined_macro_f1                     : +0.341445
delta_macro_f1                        : +0.034598
morphology_balanced_accuracy          : +0.307826
combined_balanced_accuracy            : +0.342174
delta_balanced_accuracy               : +0.034348
macro_f1_folds_improved               : 5
balanced_accuracy_folds_improved      : 5

CONVERGENCE AUDIT
------------------------------------------------------------------------
Total convergence warnings: 0
Maximum iterations used    : 142

🟢 Original and grouped evaluations completed
🟢 Scaling occurred inside every training fold
🟢 Identical classifier used for both representations
🟢 All predictions are out of fold
🟢 No frozen feature was modified


In [ ]:
# ============================================================
# CELL 7 — METRIC-CONVENTION LOCK AND
#          PAIRED SOURCE-IDENTITY BOOTSTRAP
# ============================================================

print("=" * 72)
print("🛡️ CELL 7 — IDENTITY-AWARE UNCERTAINTY")
print("=" * 72)

SHIELD_BOOTSTRAP_REPLICATES = 5000
SHIELD_BOOTSTRAP_SEED = 20260820

shield_bootstrap_rng = np.random.default_rng(
    SHIELD_BOOTSTRAP_SEED
)

# ------------------------------------------------------------
# 1. Verify that each source identity has exactly one category
# ------------------------------------------------------------

shield_categories_per_identity = (
    shield_identity_table
    .groupby("garment_identity")["category"]
    .nunique()
)

assert (
    shield_categories_per_identity == 1
).all()

shield_identity_category_table = (
    shield_identity_table[
        [
            "garment_identity",
            "category",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "category",
            "garment_identity",
        ]
    )
    .reset_index(drop=True)
)

assert len(shield_identity_category_table) == 230

print("\nIDENTITY UNIT")
print("-" * 72)
print("Source identities       : 230")
print("Categories              : 23")
print("Identities/category     : 10")
print("Bootstrap unit          : source garment identity")
print("Bootstrap stratification: garment category")

# ------------------------------------------------------------
# 2. Lock fold-mean versus pooled metric conventions
# ------------------------------------------------------------

def shield_metric_convention_row(
    morphology_result,
    combined_result,
    split_name,
):
    morphology_folds = (
        morphology_result["fold_metrics"]
    )

    combined_folds = (
        combined_result["fold_metrics"]
    )

    morphology_pooled = (
        morphology_result["pooled_metrics"]
    )

    combined_pooled = (
        combined_result["pooled_metrics"]
    )

    return {
        "split_design": split_name,

        "fold_mean_morphology_macro_f1":
            morphology_folds[
                "macro_f1"
            ].mean(),

        "fold_mean_combined_macro_f1":
            combined_folds[
                "macro_f1"
            ].mean(),

        "fold_mean_delta_macro_f1":
            (
                combined_folds["macro_f1"]
                - morphology_folds["macro_f1"]
            ).mean(),

        "pooled_morphology_macro_f1":
            morphology_pooled["macro_f1"],

        "pooled_combined_macro_f1":
            combined_pooled["macro_f1"],

        "pooled_delta_macro_f1":
            (
                combined_pooled["macro_f1"]
                - morphology_pooled["macro_f1"]
            ),

        "fold_mean_morphology_ba":
            morphology_folds[
                "balanced_accuracy"
            ].mean(),

        "fold_mean_combined_ba":
            combined_folds[
                "balanced_accuracy"
            ].mean(),

        "fold_mean_delta_ba":
            (
                combined_folds[
                    "balanced_accuracy"
                ]
                - morphology_folds[
                    "balanced_accuracy"
                ]
            ).mean(),

        "pooled_morphology_ba":
            morphology_pooled[
                "balanced_accuracy"
            ],

        "pooled_combined_ba":
            combined_pooled[
                "balanced_accuracy"
            ],

        "pooled_delta_ba":
            (
                combined_pooled[
                    "balanced_accuracy"
                ]
                - morphology_pooled[
                    "balanced_accuracy"
                ]
            ),
    }

shield_metric_convention_table = pd.DataFrame([
    shield_metric_convention_row(
        shield_original_morphology,
        shield_original_combined,
        "Original image-level CV",
    ),
    shield_metric_convention_row(
        shield_grouped_morphology,
        shield_grouped_combined,
        "Unseen-garment CV",
    ),
])

print("\nMETRIC-CONVENTION TABLE")
print("-" * 72)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 240,
):
    display(
        shield_metric_convention_table.round(6)
    )

# Primary reporting convention:
#
# - fold means: comparison with the original frozen study;
# - pooled OOF metrics: identity-bootstrap inference.

shield_primary_reporting_convention = {
    "original_result_comparison":
        "mean of five fold-level metrics",
    "bootstrap_inference":
        "pooled out-of-fold predictions",
    "bootstrap_unit":
        "source garment identity",
    "bootstrap_stratification":
        "garment category",
}

# ------------------------------------------------------------
# 3. Prepare identity-to-row lookup
# ------------------------------------------------------------

shield_rows_by_identity = {
    garment_identity: (
        shield_identity_table.index[
            shield_identity_table[
                "garment_identity"
            ] == garment_identity
        ].to_numpy()
    )
    for garment_identity in (
        shield_identity_table[
            "garment_identity"
        ].unique()
    )
}

shield_identities_by_category = {
    category: (
        shield_identity_category_table.loc[
            shield_identity_category_table[
                "category"
            ] == category,
            "garment_identity",
        ].to_numpy()
    )
    for category in (
        shield_categories_sorted
    )
}

for category, identities in (
    shield_identities_by_category.items()
):
    assert len(identities) == 10

# ------------------------------------------------------------
# 4. Observed grouped-CV effect from pooled OOF predictions
# ------------------------------------------------------------

shield_y_true = np.asarray(
    shield_labels
)

shield_y_pred_morphology = np.asarray(
    shield_grouped_morphology[
        "predictions"
    ]
)

shield_y_pred_combined = np.asarray(
    shield_grouped_combined[
        "predictions"
    ]
)

shield_observed_bootstrap_target = {
    "morphology_macro_f1": f1_score(
        shield_y_true,
        shield_y_pred_morphology,
        average="macro",
        zero_division=0,
    ),
    "combined_macro_f1": f1_score(
        shield_y_true,
        shield_y_pred_combined,
        average="macro",
        zero_division=0,
    ),
    "morphology_ba": (
        balanced_accuracy_score(
            shield_y_true,
            shield_y_pred_morphology,
        )
    ),
    "combined_ba": (
        balanced_accuracy_score(
            shield_y_true,
            shield_y_pred_combined,
        )
    ),
}

shield_observed_bootstrap_target[
    "delta_macro_f1"
] = (
    shield_observed_bootstrap_target[
        "combined_macro_f1"
    ]
    - shield_observed_bootstrap_target[
        "morphology_macro_f1"
    ]
)

shield_observed_bootstrap_target[
    "delta_ba"
] = (
    shield_observed_bootstrap_target[
        "combined_ba"
    ]
    - shield_observed_bootstrap_target[
        "morphology_ba"
    ]
)

# ------------------------------------------------------------
# 5. Stratified paired identity bootstrap
#
# Predictions are held fixed. Complete source identities are
# resampled with replacement within each category.
# ------------------------------------------------------------

shield_bootstrap_records = []

print("\nPAIRED IDENTITY BOOTSTRAP")
print("-" * 72)

for bootstrap_index in range(
    SHIELD_BOOTSTRAP_REPLICATES
):
    sampled_row_blocks = []

    for category in shield_categories_sorted:
        category_identities = (
            shield_identities_by_category[
                category
            ]
        )

        sampled_identities = (
            shield_bootstrap_rng.choice(
                category_identities,
                size=len(category_identities),
                replace=True,
            )
        )

        for garment_identity in (
            sampled_identities
        ):
            sampled_row_blocks.append(
                shield_rows_by_identity[
                    garment_identity
                ]
            )

    sampled_rows = np.concatenate(
        sampled_row_blocks
    )

    y_boot = shield_y_true[
        sampled_rows
    ]

    morphology_boot = (
        shield_y_pred_morphology[
            sampled_rows
        ]
    )

    combined_boot = (
        shield_y_pred_combined[
            sampled_rows
        ]
    )

    morphology_macro_f1 = f1_score(
        y_boot,
        morphology_boot,
        average="macro",
        zero_division=0,
    )

    combined_macro_f1 = f1_score(
        y_boot,
        combined_boot,
        average="macro",
        zero_division=0,
    )

    morphology_ba = (
        balanced_accuracy_score(
            y_boot,
            morphology_boot,
        )
    )

    combined_ba = (
        balanced_accuracy_score(
            y_boot,
            combined_boot,
        )
    )

    shield_bootstrap_records.append({
        "bootstrap": bootstrap_index,
        "delta_macro_f1": (
            combined_macro_f1
            - morphology_macro_f1
        ),
        "delta_balanced_accuracy": (
            combined_ba
            - morphology_ba
        ),
    })

    if (
        (bootstrap_index + 1) % 1000 == 0
        or bootstrap_index + 1
        == SHIELD_BOOTSTRAP_REPLICATES
    ):
        print(
            f"completed "
            f"{bootstrap_index + 1}/"
            f"{SHIELD_BOOTSTRAP_REPLICATES}"
        )

shield_identity_bootstrap = pd.DataFrame(
    shield_bootstrap_records
)

# ------------------------------------------------------------
# 6. Summarize bootstrap distributions
# ------------------------------------------------------------

def shield_bootstrap_summary(
    values,
    observed,
    metric_name,
):
    values = np.asarray(values)

    lower, upper = np.quantile(
        values,
        [0.025, 0.975],
    )

    return {
        "metric": metric_name,
        "observed_delta": observed,
        "bootstrap_mean": values.mean(),
        "bootstrap_median": np.median(values),
        "ci_2.5_percent": lower,
        "ci_97.5_percent": upper,
        "fraction_delta_le_zero": (
            np.mean(values <= 0)
        ),
        "bootstrap_replicates": len(values),
    }

shield_identity_bootstrap_summary = pd.DataFrame([
    shield_bootstrap_summary(
        values=shield_identity_bootstrap[
            "delta_macro_f1"
        ],
        observed=(
            shield_observed_bootstrap_target[
                "delta_macro_f1"
            ]
        ),
        metric_name="Macro-F1",
    ),
    shield_bootstrap_summary(
        values=shield_identity_bootstrap[
            "delta_balanced_accuracy"
        ],
        observed=(
            shield_observed_bootstrap_target[
                "delta_ba"
            ]
        ),
        metric_name="Balanced accuracy",
    ),
])

print("\nIDENTITY-BOOTSTRAP SUMMARY")
print("-" * 72)
display(
    shield_identity_bootstrap_summary.round(6)
)

# ------------------------------------------------------------
# 7. Fold-direction audit
# ------------------------------------------------------------

shield_fold_direction_summary = {
    "macro_f1_positive_folds": int(
        (
            shield_grouped_fold_deltas[
                "delta_macro_f1"
            ] > 0
        ).sum()
    ),
    "balanced_accuracy_positive_folds": int(
        (
            shield_grouped_fold_deltas[
                "delta_balanced_accuracy"
            ] > 0
        ).sum()
    ),
    "total_folds": 5,
}

print("\nDIRECTIONAL CONSISTENCY")
print("-" * 72)
print(
    "Macro-F1 positive folds       : "
    f"{shield_fold_direction_summary['macro_f1_positive_folds']}/5"
)
print(
    "Balanced-accuracy positive folds: "
    f"{shield_fold_direction_summary['balanced_accuracy_positive_folds']}/5"
)

print("\nINTERPRETATION BOUNDARY")
print("-" * 72)
print(
    "The bootstrap quantifies uncertainty conditional on "
    "the fixed grouped out-of-fold predictions."
)
print(
    "It does not replace model-refitting permutations or "
    "repeated grouped cross-validation."
)

print("\n" + "=" * 72)
print("🟢 Metric convention explicitly locked")
print("🟢 Original fold-mean result identified")
print("🟢 Source-identity-aware uncertainty estimated")
print("🟢 Category balance preserved in every bootstrap")
print("🟢 No model or frozen feature was modified")
print("=" * 72)

🛡️ CELL 7 — IDENTITY-AWARE UNCERTAINTY

IDENTITY UNIT
------------------------------------------------------------------------
Source identities       : 230
Categories              : 23
Identities/category     : 10
Bootstrap unit          : source garment identity
Bootstrap stratification: garment category

METRIC-CONVENTION TABLE
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,split_design,fold_mean_morphology_macro_f1,fold_mean_combined_macro_f1,fold_mean_delta_macro_f1,pooled_morphology_macro_f1,pooled_combined_macro_f1,pooled_delta_macro_f1,fold_mean_morphology_ba,fold_mean_combined_ba,fold_mean_delta_ba,pooled_morphology_ba,pooled_combined_ba,pooled_delta_ba
0,Original image-level CV,0.341131,0.412332,0.071201,0.342322,0.414257,0.071936,0.342174,0.415652,0.073478,0.342174,0.415652,0.073478
1,Unseen-garment CV,0.304713,0.335976,0.031263,0.306847,0.341445,0.034598,0.307821,0.342272,0.034451,0.307826,0.342174,0.034348



PAIRED IDENTITY BOOTSTRAP
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

completed 5000/5000

IDENTITY-BOOTSTRAP SUMMARY
------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,metric,observed_delta,bootstrap_mean,bootstrap_median,ci_2.5_percent,ci_97.5_percent,fraction_delta_le_zero,bootstrap_replicates
0,Macro-F1,0.034598,0.034582,0.034395,0.015783,0.053962,0.0004,5000
1,Balanced accuracy,0.034348,0.034458,0.034348,0.015612,0.054268,0.0004,5000



DIRECTIONAL CONSISTENCY
------------------------------------------------------------------------
Macro-F1 positive folds       : 5/5
Balanced-accuracy positive folds: 5/5

INTERPRETATION BOUNDARY
------------------------------------------------------------------------
The bootstrap quantifies uncertainty conditional on the fixed grouped out-of-fold predictions.
It does not replace model-refitting permutations or repeated grouped cross-validation.

🟢 Metric convention explicitly locked
🟢 Original fold-mean result identified
🟢 Source-identity-aware uncertainty estimated
🟢 Category balance preserved in every bootstrap
🟢 No model or frozen feature was modified


In [ ]:
# ============================================================
# CELL 8 — UNSEEN-GARMENT THREE-WAY COMPARISON
#
# Morphology versus RA versus Morphology + RA
# ============================================================

from sklearn.metrics import (
    precision_recall_fscore_support,
)

# Silence only the irrelevant Jupyter datetime warning
warnings.filterwarnings(
    "ignore",
    message=(
        "datetime.datetime.utcnow\\(\\) is deprecated"
    ),
    category=DeprecationWarning,
)

print("=" * 72)
print("🛡️ CELL 8 — THREE-WAY REPRESENTATION COMPARISON")
print("=" * 72)

# ------------------------------------------------------------
# 1. Evaluate RA alone on identical grouped folds
# ------------------------------------------------------------

shield_grouped_ra_only = (
    shield_evaluate_classifier(
        X=shield_X_ra_work,
        y=shield_labels,
        splits=shield_group_splits,
        representation_name="RA only",
        split_name="Unseen garment identity",
    )
)

print("🟢 RA-only grouped evaluation complete")

# ------------------------------------------------------------
# 2. Pooled three-way comparison
# ------------------------------------------------------------

shield_grouped_three_way_summary = pd.DataFrame([
    shield_grouped_morphology[
        "pooled_metrics"
    ],
    shield_grouped_ra_only[
        "pooled_metrics"
    ],
    shield_grouped_combined[
        "pooled_metrics"
    ],
])

representation_order = [
    "Morphology",
    "RA only",
    "Morphology + RA",
]

shield_grouped_three_way_summary[
    "representation"
] = pd.Categorical(
    shield_grouped_three_way_summary[
        "representation"
    ],
    categories=representation_order,
    ordered=True,
)

shield_grouped_three_way_summary = (
    shield_grouped_three_way_summary
    .sort_values("representation")
    .reset_index(drop=True)
)

print("\nPOOLED UNSEEN-GARMENT PERFORMANCE")
print("-" * 72)
display(
    shield_grouped_three_way_summary.round(6)
)

# ------------------------------------------------------------
# 3. Fold-mean comparison
# ------------------------------------------------------------

def shield_fold_mean_summary(result):
    fold_table = result["fold_metrics"]

    return {
        "representation": (
            result["pooled_metrics"][
                "representation"
            ]
        ),
        "mean_fold_accuracy": (
            fold_table["accuracy"].mean()
        ),
        "mean_fold_balanced_accuracy": (
            fold_table[
                "balanced_accuracy"
            ].mean()
        ),
        "mean_fold_macro_f1": (
            fold_table["macro_f1"].mean()
        ),
        "sd_fold_macro_f1": (
            fold_table["macro_f1"].std(
                ddof=1
            )
        ),
        "minimum_fold_macro_f1": (
            fold_table["macro_f1"].min()
        ),
        "maximum_fold_macro_f1": (
            fold_table["macro_f1"].max()
        ),
    }

shield_grouped_fold_mean_summary = pd.DataFrame([
    shield_fold_mean_summary(
        shield_grouped_morphology
    ),
    shield_fold_mean_summary(
        shield_grouped_ra_only
    ),
    shield_fold_mean_summary(
        shield_grouped_combined
    ),
])

shield_grouped_fold_mean_summary[
    "representation"
] = pd.Categorical(
    shield_grouped_fold_mean_summary[
        "representation"
    ],
    categories=representation_order,
    ordered=True,
)

shield_grouped_fold_mean_summary = (
    shield_grouped_fold_mean_summary
    .sort_values("representation")
    .reset_index(drop=True)
)

print("\nFOLD-MEAN UNSEEN-GARMENT PERFORMANCE")
print("-" * 72)
display(
    shield_grouped_fold_mean_summary.round(6)
)

# ------------------------------------------------------------
# 4. Pairwise pooled improvements
# ------------------------------------------------------------

three_way_lookup = (
    shield_grouped_three_way_summary
    .set_index("representation")
)

shield_three_way_contrasts = pd.DataFrame([
    {
        "contrast":
            "RA only − Morphology",
        "delta_macro_f1": (
            three_way_lookup.loc[
                "RA only",
                "macro_f1",
            ]
            - three_way_lookup.loc[
                "Morphology",
                "macro_f1",
            ]
        ),
        "delta_balanced_accuracy": (
            three_way_lookup.loc[
                "RA only",
                "balanced_accuracy",
            ]
            - three_way_lookup.loc[
                "Morphology",
                "balanced_accuracy",
            ]
        ),
    },
    {
        "contrast":
            "Combined − Morphology",
        "delta_macro_f1": (
            three_way_lookup.loc[
                "Morphology + RA",
                "macro_f1",
            ]
            - three_way_lookup.loc[
                "Morphology",
                "macro_f1",
            ]
        ),
        "delta_balanced_accuracy": (
            three_way_lookup.loc[
                "Morphology + RA",
                "balanced_accuracy",
            ]
            - three_way_lookup.loc[
                "Morphology",
                "balanced_accuracy",
            ]
        ),
    },
    {
        "contrast":
            "Combined − RA only",
        "delta_macro_f1": (
            three_way_lookup.loc[
                "Morphology + RA",
                "macro_f1",
            ]
            - three_way_lookup.loc[
                "RA only",
                "macro_f1",
            ]
        ),
        "delta_balanced_accuracy": (
            three_way_lookup.loc[
                "Morphology + RA",
                "balanced_accuracy",
            ]
            - three_way_lookup.loc[
                "RA only",
                "balanced_accuracy",
            ]
        ),
    },
])

print("\nPAIRWISE POOLED CONTRASTS")
print("-" * 72)
display(
    shield_three_way_contrasts.round(6)
)

# ------------------------------------------------------------
# 5. Per-category F1 comparison
# ------------------------------------------------------------

shield_category_order = np.sort(
    np.unique(shield_labels)
)

_, _, morphology_category_f1, morphology_support = (
    precision_recall_fscore_support(
        shield_labels,
        shield_grouped_morphology[
            "predictions"
        ],
        labels=shield_category_order,
        zero_division=0,
    )
)

_, _, ra_category_f1, ra_support = (
    precision_recall_fscore_support(
        shield_labels,
        shield_grouped_ra_only[
            "predictions"
        ],
        labels=shield_category_order,
        zero_division=0,
    )
)

_, _, combined_category_f1, combined_support = (
    precision_recall_fscore_support(
        shield_labels,
        shield_grouped_combined[
            "predictions"
        ],
        labels=shield_category_order,
        zero_division=0,
    )
)

assert np.array_equal(
    morphology_support,
    ra_support,
)

assert np.array_equal(
    morphology_support,
    combined_support,
)

shield_grouped_category_results = pd.DataFrame({
    "category": shield_category_order,
    "support": morphology_support.astype(int),
    "morphology_f1": morphology_category_f1,
    "ra_only_f1": ra_category_f1,
    "combined_f1": combined_category_f1,
})

shield_grouped_category_results[
    "combined_minus_morphology"
] = (
    shield_grouped_category_results[
        "combined_f1"
    ]
    - shield_grouped_category_results[
        "morphology_f1"
    ]
)

shield_grouped_category_results[
    "combined_minus_ra"
] = (
    shield_grouped_category_results[
        "combined_f1"
    ]
    - shield_grouped_category_results[
        "ra_only_f1"
    ]
)

shield_categories_improved_vs_morphology = int(
    (
        shield_grouped_category_results[
            "combined_minus_morphology"
        ] > 0
    ).sum()
)

shield_categories_improved_vs_ra = int(
    (
        shield_grouped_category_results[
            "combined_minus_ra"
        ] > 0
    ).sum()
)

print("\nPER-CATEGORY UNSEEN-GARMENT RESULTS")
print("-" * 72)

display(
    shield_grouped_category_results
    .sort_values(
        "combined_minus_morphology",
        ascending=False,
    )
    .round(4)
)

print("\nCATEGORY-WIDE DIRECTION")
print("-" * 72)
print(
    "Combined improved over morphology: "
    f"{shield_categories_improved_vs_morphology}/23"
)
print(
    "Combined improved over RA alone  : "
    f"{shield_categories_improved_vs_ra}/23"
)

# ------------------------------------------------------------
# 6. Convergence
# ------------------------------------------------------------

shield_ra_convergence_warnings = int(
    shield_grouped_ra_only[
        "fold_metrics"
    ]["convergence_warnings"].sum()
)

print("\nCONVERGENCE")
print("-" * 72)
print(
    f"RA-only convergence warnings: "
    f"{shield_ra_convergence_warnings}"
)

print("\n" + "=" * 72)
print("🟢 Three representations compared on identical folds")
print("🟢 All test garments were unseen during training")
print("🟢 Per-category behavior quantified")
print("🟢 No frozen feature or split was modified")
print("=" * 72)

🛡️ CELL 8 — THREE-WAY REPRESENTATION COMPARISON
🟢 RA-only grouped evaluation complete

POOLED UNSEEN-GARMENT PERFORMANCE
------------------------------------------------------------------------


,split_design,representation,accuracy,balanced_accuracy,macro_f1
0,Unseen garment identity,Morphology,0.307826,0.307826,0.306847
1,Unseen garment identity,RA only,0.276087,0.276087,0.265323
2,Unseen garment identity,Morphology + RA,0.342174,0.342174,0.341445



FOLD-MEAN UNSEEN-GARMENT PERFORMANCE
------------------------------------------------------------------------


,representation,mean_fold_accuracy,mean_fold_balanced_accuracy,mean_fold_macro_f1,sd_fold_macro_f1,minimum_fold_macro_f1,maximum_fold_macro_f1
0,Morphology,0.307813,0.307821,0.304713,0.030184,0.280342,0.349416
1,RA only,0.276073,0.276137,0.260717,0.015196,0.243876,0.282863
2,Morphology + RA,0.342163,0.342272,0.335976,0.024563,0.311576,0.372376



PAIRWISE POOLED CONTRASTS
------------------------------------------------------------------------


,contrast,delta_macro_f1,delta_balanced_accuracy
0,RA only − Morphology,-0.041525,-0.031739
1,Combined − Morphology,0.034598,0.034348
2,Combined − RA only,0.076122,0.066087



PER-CATEGORY UNSEEN-GARMENT RESULTS
------------------------------------------------------------------------


,category,support,morphology_f1,ra_only_f1,combined_f1,combined_minus_morphology,combined_minus_ra
14,Sarong,100,0.2755,0.3755,0.3923,0.1168,0.0169
18,Suit,100,0.2128,0.3415,0.3269,0.1142,-0.0145
2,Blouse,100,0.5027,0.3368,0.5949,0.0922,0.2580
21,Vest,100,0.3033,0.3282,0.3838,0.0805,0.0556
16,Skinny,100,0.1111,0.1914,0.1910,0.0798,-0.0004
12,Mini,100,0.2451,0.3092,0.3085,0.0634,-0.0007
4,Circle,100,0.3707,0.2632,0.4190,0.0483,0.1559
10,Jumpsuit,100,0.3861,0.3470,0.4299,0.0438,0.0829
3,Cardigan,100,0.1415,0.1130,0.1759,0.0344,0.0629
19,T-shirt,100,0.4909,0.5094,0.5171,0.0262,0.0076



CATEGORY-WIDE DIRECTION
------------------------------------------------------------------------
Combined improved over morphology: 18/23
Combined improved over RA alone  : 19/23

CONVERGENCE
------------------------------------------------------------------------
RA-only convergence warnings: 0

🟢 Three representations compared on identical folds
🟢 All test garments were unseen during training
🟢 Per-category behavior quantified
🟢 No frozen feature or split was modified


In [ ]:
# ============================================================
# CELL 9 — WITHIN-CATEGORY TEST-TIME ALIGNMENT CONTROL
#
# Purpose:
#   Preserve garment-category membership and RA distributions,
#   but disrupt the exact morphology ↔ RA pairing within each
#   held-out test category.
#
# This is a fixed-model alignment-sensitivity analysis.
# It is not a model-refitting permutation test.
# ============================================================

print("=" * 72)
print("🛡️ CELL 9 — WITHIN-CATEGORY ALIGNMENT CONTROL")
print("=" * 72)

SHIELD_ALIGNMENT_PERMUTATIONS = 2000
SHIELD_ALIGNMENT_SEED = 20260821

shield_alignment_rng = np.random.default_rng(
    SHIELD_ALIGNMENT_SEED
)

# ------------------------------------------------------------
# 1. Observed grouped-CV reference
# ------------------------------------------------------------

shield_observed_combined_predictions = np.asarray(
    shield_grouped_combined["predictions"]
)

shield_observed_combined_macro_f1 = f1_score(
    shield_labels,
    shield_observed_combined_predictions,
    average="macro",
    zero_division=0,
)

shield_observed_combined_ba = (
    balanced_accuracy_score(
        shield_labels,
        shield_observed_combined_predictions,
    )
)

shield_observed_morphology_macro_f1 = f1_score(
    shield_labels,
    shield_grouped_morphology[
        "predictions"
    ],
    average="macro",
    zero_division=0,
)

shield_observed_morphology_ba = (
    balanced_accuracy_score(
        shield_labels,
        shield_grouped_morphology[
            "predictions"
        ],
    )
)

shield_observed_alignment_gain_macro_f1 = (
    shield_observed_combined_macro_f1
    - shield_observed_morphology_macro_f1
)

shield_observed_alignment_gain_ba = (
    shield_observed_combined_ba
    - shield_observed_morphology_ba
)

print("\nOBSERVED UNSEEN-GARMENT REFERENCE")
print("-" * 72)
print(
    f"Morphology Macro-F1     : "
    f"{shield_observed_morphology_macro_f1:.6f}"
)
print(
    f"Aligned combined Macro-F1: "
    f"{shield_observed_combined_macro_f1:.6f}"
)
print(
    f"Observed Δ Macro-F1     : "
    f"{shield_observed_alignment_gain_macro_f1:+.6f}"
)
print(
    f"Morphology BA           : "
    f"{shield_observed_morphology_ba:.6f}"
)
print(
    f"Aligned combined BA     : "
    f"{shield_observed_combined_ba:.6f}"
)
print(
    f"Observed Δ BA           : "
    f"{shield_observed_alignment_gain_ba:+.6f}"
)

# ------------------------------------------------------------
# 2. Verify fitted models correspond to grouped folds
# ------------------------------------------------------------

shield_grouped_combined_models = (
    shield_grouped_combined["models"]
)

assert len(shield_grouped_combined_models) == 5
assert len(shield_group_splits) == 5

# ------------------------------------------------------------
# 3. Within-category permutation helper
# ------------------------------------------------------------

def shield_permute_indices_within_category(
    row_indices,
    labels,
    rng,
):
    row_indices = np.asarray(row_indices)
    labels = np.asarray(labels)

    permuted_source_indices = (
        row_indices.copy()
    )

    for category in np.unique(
        labels[row_indices]
    ):
        category_positions = np.flatnonzero(
            labels[row_indices] == category
        )

        category_source_rows = (
            row_indices[category_positions]
        )

        permuted_source_indices[
            category_positions
        ] = rng.permutation(
            category_source_rows
        )

    return permuted_source_indices

# ------------------------------------------------------------
# 4. Fixed-model test-time perturbation
# ------------------------------------------------------------

shield_alignment_records = []

print("\nRUNNING WITHIN-CATEGORY ALIGNMENT PERTURBATIONS")
print("-" * 72)

for permutation_index in range(
    SHIELD_ALIGNMENT_PERMUTATIONS
):
    permuted_predictions = np.empty(
        2300,
        dtype=shield_labels.dtype,
    )

    coverage = np.zeros(
        2300,
        dtype=int,
    )

    for fold_index, (
        train_index,
        test_index,
    ) in enumerate(
        shield_group_splits
    ):
        model = (
            shield_grouped_combined_models[
                fold_index
            ]
        )

        permuted_ra_source_rows = (
            shield_permute_indices_within_category(
                row_indices=test_index,
                labels=shield_labels,
                rng=shield_alignment_rng,
            )
        )

        # Morphology remains attached to its true sketch.
        # RA comes from another held-out sketch of the
        # same category.
        permuted_test_matrix = np.column_stack([
            shield_X_morphology_work[
                test_index
            ],
            shield_X_ra_work[
                permuted_ra_source_rows
            ],
        ])

        fold_predictions = model.predict(
            permuted_test_matrix
        )

        permuted_predictions[test_index] = (
            fold_predictions
        )

        coverage[test_index] += 1

    assert np.all(coverage == 1)

    permuted_macro_f1 = f1_score(
        shield_labels,
        permuted_predictions,
        average="macro",
        zero_division=0,
    )

    permuted_ba = balanced_accuracy_score(
        shield_labels,
        permuted_predictions,
    )

    shield_alignment_records.append({
        "permutation": permutation_index,
        "permuted_combined_macro_f1":
            permuted_macro_f1,
        "permuted_combined_ba":
            permuted_ba,
        "permuted_delta_vs_morphology_macro_f1":
            (
                permuted_macro_f1
                - shield_observed_morphology_macro_f1
            ),
        "permuted_delta_vs_morphology_ba":
            (
                permuted_ba
                - shield_observed_morphology_ba
            ),
        "aligned_minus_permuted_macro_f1":
            (
                shield_observed_combined_macro_f1
                - permuted_macro_f1
            ),
        "aligned_minus_permuted_ba":
            (
                shield_observed_combined_ba
                - permuted_ba
            ),
    })

    if (
        (permutation_index + 1) % 500 == 0
        or permutation_index + 1
        == SHIELD_ALIGNMENT_PERMUTATIONS
    ):
        print(
            f"completed "
            f"{permutation_index + 1}/"
            f"{SHIELD_ALIGNMENT_PERMUTATIONS}"
        )

shield_alignment_null = pd.DataFrame(
    shield_alignment_records
)

# ------------------------------------------------------------
# 5. Empirical alignment p-values
#
# Null question:
# Can a category-preserving but incorrectly paired RA row
# equal or exceed correctly aligned combined performance?
# ------------------------------------------------------------

shield_alignment_p_macro_f1 = (
    1
    + (
        shield_alignment_null[
            "permuted_combined_macro_f1"
        ]
        >= shield_observed_combined_macro_f1
    ).sum()
) / (
    SHIELD_ALIGNMENT_PERMUTATIONS + 1
)

shield_alignment_p_ba = (
    1
    + (
        shield_alignment_null[
            "permuted_combined_ba"
        ]
        >= shield_observed_combined_ba
    ).sum()
) / (
    SHIELD_ALIGNMENT_PERMUTATIONS + 1
)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

def shield_alignment_summary_row(
    metric_name,
    observed_combined,
    observed_morphology,
    permuted_values,
    empirical_p,
):
    permuted_values = np.asarray(
        permuted_values
    )

    lower, upper = np.quantile(
        permuted_values,
        [0.025, 0.975],
    )

    return {
        "metric": metric_name,
        "morphology_only": observed_morphology,
        "aligned_combined": observed_combined,
        "observed_gain_vs_morphology": (
            observed_combined
            - observed_morphology
        ),
        "permuted_mean": (
            permuted_values.mean()
        ),
        "permuted_median": (
            np.median(permuted_values)
        ),
        "permuted_2.5_percent": lower,
        "permuted_97.5_percent": upper,
        "aligned_minus_permuted_mean": (
            observed_combined
            - permuted_values.mean()
        ),
        "empirical_alignment_p": (
            empirical_p
        ),
        "permutations": len(
            permuted_values
        ),
    }

shield_alignment_summary = pd.DataFrame([
    shield_alignment_summary_row(
        metric_name="Macro-F1",
        observed_combined=(
            shield_observed_combined_macro_f1
        ),
        observed_morphology=(
            shield_observed_morphology_macro_f1
        ),
        permuted_values=(
            shield_alignment_null[
                "permuted_combined_macro_f1"
            ]
        ),
        empirical_p=(
            shield_alignment_p_macro_f1
        ),
    ),
    shield_alignment_summary_row(
        metric_name="Balanced accuracy",
        observed_combined=(
            shield_observed_combined_ba
        ),
        observed_morphology=(
            shield_observed_morphology_ba
        ),
        permuted_values=(
            shield_alignment_null[
                "permuted_combined_ba"
            ]
        ),
        empirical_p=shield_alignment_p_ba,
    ),
])

print("\nWITHIN-CATEGORY ALIGNMENT SUMMARY")
print("-" * 72)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 220,
):
    display(
        shield_alignment_summary.round(6)
    )

print("\nINTERPRETATION BOUNDARY")
print("-" * 72)
print(
    "This analysis preserves category membership and "
    "perturbs only held-out morphology–RA pairing."
)
print(
    "It tests fixed-model sensitivity to correct "
    "test-time alignment."
)
print(
    "It is not a substitute for a full model-refitting "
    "permutation test."
)

print("\n" + "=" * 72)
print("🟢 Category-level RA distributions preserved")
print("🟢 Exact held-out sketch alignment disrupted")
print("🟢 Source-grouped folds preserved")
print("🟢 Fitted models and frozen features unchanged")
print("=" * 72)

🛡️ CELL 9 — WITHIN-CATEGORY ALIGNMENT CONTROL

OBSERVED UNSEEN-GARMENT REFERENCE
------------------------------------------------------------------------
Morphology Macro-F1     : 0.306847
Aligned combined Macro-F1: 0.341445
Observed Δ Macro-F1     : +0.034598
Morphology BA           : 0.307826
Aligned combined BA     : 0.342174
Observed Δ BA           : +0.034348

RUNNING WITHIN-CATEGORY ALIGNMENT PERTURBATIONS
------------------------------------------------------------------------
completed 500/2000
completed 1000/2000
completed 1500/2000
completed 2000/2000

WITHIN-CATEGORY ALIGNMENT SUMMARY
------------------------------------------------------------------------


,metric,morphology_only,aligned_combined,observed_gain_vs_morphology,permuted_mean,permuted_median,permuted_2.5_percent,permuted_97.5_percent,aligned_minus_permuted_mean,empirical_alignment_p,permutations
0,Macro-F1,0.306847,0.341445,0.034598,0.335293,0.335451,0.324199,0.345944,0.006153,0.141929,2000
1,Balanced accuracy,0.307826,0.342174,0.034348,0.335332,0.335652,0.323913,0.346087,0.006842,0.122939,2000



INTERPRETATION BOUNDARY
------------------------------------------------------------------------
This analysis preserves category membership and perturbs only held-out morphology–RA pairing.
It tests fixed-model sensitivity to correct test-time alignment.
It is not a substitute for a full model-refitting permutation test.

🟢 Category-level RA distributions preserved
🟢 Exact held-out sketch alignment disrupted
🟢 Source-grouped folds preserved
🟢 Fitted models and frozen features unchanged


In [ ]:
# ============================================================
# CELL 10 — REPEATED EXACT-BALANCED GROUPED CV
#
# Tests whether the unseen-garment complementarity result is
# robust to different allocations of source identities to folds.
# ============================================================

import gc

print("=" * 72)
print("🛡️ CELL 10 — REPEATED GROUPED-CV ROBUSTNESS")
print("=" * 72)

SHIELD_GROUPED_REPEATS = 10
SHIELD_REPEAT_BASE_SEED = 20260820

# ------------------------------------------------------------
# 1. Exact balanced grouped-split constructor
# ------------------------------------------------------------

def shield_make_balanced_group_splits(seed):
    split_rng = np.random.default_rng(seed)

    test_groups_by_fold = [
        set() for _ in range(5)
    ]

    for category in shield_categories_sorted:
        category_groups = np.sort(
            shield_identity_table.loc[
                shield_identity_table[
                    "category"
                ] == category,
                "garment_identity",
            ].unique()
        )

        if len(category_groups) != 10:
            raise ValueError(
                f"{category}: expected 10 identities, "
                f"found {len(category_groups)}"
            )

        shuffled_groups = (
            split_rng.permutation(
                category_groups
            )
        )

        fold_chunks = np.array_split(
            shuffled_groups,
            5,
        )

        for fold_index, chunk in enumerate(
            fold_chunks
        ):
            if len(chunk) != 2:
                raise RuntimeError(
                    "Balanced identity allocation failed."
                )

            test_groups_by_fold[
                fold_index
            ].update(chunk.tolist())

    all_rows = np.arange(2300)
    splits = []

    identity_test_coverage = Counter()

    for test_group_set in test_groups_by_fold:
        test_mask = np.isin(
            shield_groups,
            list(test_group_set),
        )

        test_index = all_rows[test_mask]
        train_index = all_rows[~test_mask]

        train_groups = set(
            shield_groups[train_index]
        )

        test_groups = set(
            shield_groups[test_index]
        )

        assert len(train_groups) == 184
        assert len(test_groups) == 46
        assert not train_groups.intersection(
            test_groups
        )

        test_category_identity_counts = (
            shield_identity_table.iloc[
                test_index
            ]
            .groupby("category")[
                "garment_identity"
            ]
            .nunique()
        )

        assert len(
            test_category_identity_counts
        ) == 23

        assert (
            test_category_identity_counts == 2
        ).all()

        for garment_identity in test_groups:
            identity_test_coverage[
                garment_identity
            ] += 1

        splits.append(
            (
                train_index,
                test_index,
            )
        )

    assert len(identity_test_coverage) == 230

    assert all(
        count == 1
        for count in identity_test_coverage.values()
    )

    return splits

# ------------------------------------------------------------
# 2. Run repeated grouped evaluations
# ------------------------------------------------------------

shield_repeated_grouped_records = []
shield_repeated_grouped_fold_records = []

print("\nRUNNING REPEATED GROUPED CV")
print("-" * 72)

for repeat_index in range(
    SHIELD_GROUPED_REPEATS
):
    repeat_number = repeat_index + 1
    repeat_seed = (
        SHIELD_REPEAT_BASE_SEED
        + repeat_index
    )

    repeat_splits = (
        shield_make_balanced_group_splits(
            seed=repeat_seed
        )
    )

    morphology_result = (
        shield_evaluate_classifier(
            X=shield_X_morphology_work,
            y=shield_labels,
            splits=repeat_splits,
            representation_name="Morphology",
            split_name=(
                f"Repeated grouped CV "
                f"{repeat_number}"
            ),
        )
    )

    combined_result = (
        shield_evaluate_classifier(
            X=shield_X_combined,
            y=shield_labels,
            splits=repeat_splits,
            representation_name=(
                "Morphology + RA"
            ),
            split_name=(
                f"Repeated grouped CV "
                f"{repeat_number}"
            ),
        )
    )

    morphology_pooled = (
        morphology_result[
            "pooled_metrics"
        ]
    )

    combined_pooled = (
        combined_result[
            "pooled_metrics"
        ]
    )

    morphology_folds = (
        morphology_result[
            "fold_metrics"
        ]
        .set_index("fold")
    )

    combined_folds = (
        combined_result[
            "fold_metrics"
        ]
        .set_index("fold")
    )

    fold_delta_macro_f1 = (
        combined_folds["macro_f1"]
        - morphology_folds["macro_f1"]
    )

    fold_delta_ba = (
        combined_folds[
            "balanced_accuracy"
        ]
        - morphology_folds[
            "balanced_accuracy"
        ]
    )

    shield_repeated_grouped_records.append({
        "repeat": repeat_number,
        "seed": repeat_seed,

        "morphology_pooled_macro_f1":
            morphology_pooled[
                "macro_f1"
            ],

        "combined_pooled_macro_f1":
            combined_pooled[
                "macro_f1"
            ],

        "pooled_delta_macro_f1":
            (
                combined_pooled[
                    "macro_f1"
                ]
                - morphology_pooled[
                    "macro_f1"
                ]
            ),

        "morphology_pooled_ba":
            morphology_pooled[
                "balanced_accuracy"
            ],

        "combined_pooled_ba":
            combined_pooled[
                "balanced_accuracy"
            ],

        "pooled_delta_ba":
            (
                combined_pooled[
                    "balanced_accuracy"
                ]
                - morphology_pooled[
                    "balanced_accuracy"
                ]
            ),

        "mean_fold_delta_macro_f1":
            fold_delta_macro_f1.mean(),

        "mean_fold_delta_ba":
            fold_delta_ba.mean(),

        "positive_macro_f1_folds":
            int(
                (
                    fold_delta_macro_f1 > 0
                ).sum()
            ),

        "positive_ba_folds":
            int(
                (
                    fold_delta_ba > 0
                ).sum()
            ),

        "convergence_warnings":
            int(
                morphology_result[
                    "fold_metrics"
                ][
                    "convergence_warnings"
                ].sum()
                +
                combined_result[
                    "fold_metrics"
                ][
                    "convergence_warnings"
                ].sum()
            ),
    })

    for fold_number in range(1, 6):
        shield_repeated_grouped_fold_records.append({
            "repeat": repeat_number,
            "seed": repeat_seed,
            "fold": fold_number,
            "delta_macro_f1": (
                fold_delta_macro_f1.loc[
                    fold_number
                ]
            ),
            "delta_balanced_accuracy": (
                fold_delta_ba.loc[
                    fold_number
                ]
            ),
        })

    # Release fitted estimators before the next repeat
    del morphology_result
    del combined_result
    gc.collect()

    print(
        f"repeat {repeat_number:2d}/"
        f"{SHIELD_GROUPED_REPEATS} complete | "
        f"ΔF1="
        f"{shield_repeated_grouped_records[-1]['pooled_delta_macro_f1']:+.4f} | "
        f"ΔBA="
        f"{shield_repeated_grouped_records[-1]['pooled_delta_ba']:+.4f}"
    )

shield_repeated_grouped_results = pd.DataFrame(
    shield_repeated_grouped_records
)

shield_repeated_grouped_fold_results = (
    pd.DataFrame(
        shield_repeated_grouped_fold_records
    )
)

# ------------------------------------------------------------
# 3. Summarize split-assignment robustness
# ------------------------------------------------------------

def shield_repeat_summary(
    column,
    metric_name,
):
    values = (
        shield_repeated_grouped_results[
            column
        ].to_numpy()
    )

    return {
        "metric": metric_name,
        "mean_delta": values.mean(),
        "sd_across_splits": values.std(
            ddof=1
        ),
        "minimum_delta": values.min(),
        "median_delta": np.median(values),
        "maximum_delta": values.max(),
        "positive_repeats": int(
            (values > 0).sum()
        ),
        "total_repeats": len(values),
    }

shield_repeated_grouped_summary = pd.DataFrame([
    shield_repeat_summary(
        "pooled_delta_macro_f1",
        "Macro-F1",
    ),
    shield_repeat_summary(
        "pooled_delta_ba",
        "Balanced accuracy",
    ),
])

print("\nREPEATED GROUPED-CV RESULTS")
print("-" * 72)
display(
    shield_repeated_grouped_results.round(6)
)

print("\nSPLIT-ROBUSTNESS SUMMARY")
print("-" * 72)
display(
    shield_repeated_grouped_summary.round(6)
)

shield_all_repeated_fold_positive = {
    "macro_f1": int(
        (
            shield_repeated_grouped_fold_results[
                "delta_macro_f1"
            ] > 0
        ).sum()
    ),
    "balanced_accuracy": int(
        (
            shield_repeated_grouped_fold_results[
                "delta_balanced_accuracy"
            ] > 0
        ).sum()
    ),
    "total_fold_comparisons": len(
        shield_repeated_grouped_fold_results
    ),
}

print("\nALL REPEATED FOLD COMPARISONS")
print("-" * 72)
print(
    "Positive Macro-F1 fold effects: "
    f"{shield_all_repeated_fold_positive['macro_f1']}/"
    f"{shield_all_repeated_fold_positive['total_fold_comparisons']}"
)
print(
    "Positive BA fold effects      : "
    f"{shield_all_repeated_fold_positive['balanced_accuracy']}/"
    f"{shield_all_repeated_fold_positive['total_fold_comparisons']}"
)
print(
    "Total convergence warnings    : "
    f"{shield_repeated_grouped_results['convergence_warnings'].sum()}"
)

print("\nINTERPRETATION BOUNDARY")
print("-" * 72)
print(
    "Variation across repeats measures sensitivity to "
    "identity-to-fold allocation."
)
print(
    "It is descriptive robustness evidence, not an "
    "independent confidence interval."
)

print("\n" + "=" * 72)
print("🟢 Ten exact-balanced grouped partitions evaluated")
print("🟢 Every test fold contained all 23 categories")
print("🟢 No source identity crossed train and test")
print("🟢 Morphology and combined models used identical folds")
print("=" * 72)

🛡️ CELL 10 — REPEATED GROUPED-CV ROBUSTNESS

RUNNING REPEATED GROUPED CV
------------------------------------------------------------------------
repeat  1/10 complete | ΔF1=+0.0346 | ΔBA=+0.0343
repeat  2/10 complete | ΔF1=+0.0336 | ΔBA=+0.0313
repeat  3/10 complete | ΔF1=+0.0354 | ΔBA=+0.0330
repeat  4/10 complete | ΔF1=+0.0528 | ΔBA=+0.0509
repeat  5/10 complete | ΔF1=+0.0437 | ΔBA=+0.0404
repeat  6/10 complete | ΔF1=+0.0401 | ΔBA=+0.0396
repeat  7/10 complete | ΔF1=+0.0470 | ΔBA=+0.0452
repeat  8/10 complete | ΔF1=+0.0529 | ΔBA=+0.0517
repeat  9/10 complete | ΔF1=+0.0434 | ΔBA=+0.0430
repeat 10/10 complete | ΔF1=+0.0395 | ΔBA=+0.0365

REPEATED GROUPED-CV RESULTS
------------------------------------------------------------------------


,repeat,seed,morphology_pooled_macro_f1,combined_pooled_macro_f1,pooled_delta_macro_f1,morphology_pooled_ba,combined_pooled_ba,pooled_delta_ba,mean_fold_delta_macro_f1,mean_fold_delta_ba,positive_macro_f1_folds,positive_ba_folds,convergence_warnings
0,1,20260820,0.306847,0.341445,0.034598,0.307826,0.342174,0.034348,0.031263,0.034451,5,5,0
1,2,20260821,0.313910,0.347484,0.033574,0.315217,0.346522,0.031304,0.029268,0.031346,5,5,0
2,3,20260822,0.308628,0.344077,0.035449,0.309565,0.342609,0.033043,0.033188,0.033108,4,4,0
3,4,20260823,0.298983,0.351735,0.052752,0.300435,0.351304,0.050870,0.050252,0.050901,5,5,0
4,5,20260824,0.299648,0.343379,0.043731,0.302609,0.343043,0.040435,0.043520,0.040458,5,5,0
5,6,20260825,0.307112,0.347248,0.040136,0.308261,0.347826,0.039565,0.042196,0.039843,5,5,0
6,7,20260826,0.305941,0.352957,0.047016,0.306957,0.352174,0.045217,0.045657,0.045290,5,5,0
7,8,20260827,0.299075,0.351956,0.052882,0.299130,0.350870,0.051739,0.052348,0.051791,5,5,0
8,9,20260828,0.317837,0.361226,0.043390,0.319130,0.362174,0.043043,0.042899,0.043154,5,5,0
9,10,20260829,0.304125,0.343642,0.039517,0.305652,0.342174,0.036522,0.036808,0.036673,5,5,0



SPLIT-ROBUSTNESS SUMMARY
------------------------------------------------------------------------


,metric,mean_delta,sd_across_splits,minimum_delta,median_delta,maximum_delta,positive_repeats,total_repeats
0,Macro-F1,0.042304,0.007004,0.033574,0.041763,0.052882,10,10
1,Balanced accuracy,0.040609,0.007127,0.031304,0.040000,0.051739,10,10



ALL REPEATED FOLD COMPARISONS
------------------------------------------------------------------------
Positive Macro-F1 fold effects: 49/50
Positive BA fold effects      : 49/50
Total convergence warnings    : 0

INTERPRETATION BOUNDARY
------------------------------------------------------------------------
Variation across repeats measures sensitivity to identity-to-fold allocation.
It is descriptive robustness evidence, not an independent confidence interval.

🟢 Ten exact-balanced grouped partitions evaluated
🟢 Every test fold contained all 23 categories
🟢 No source identity crossed train and test
🟢 Morphology and combined models used identical folds


In [ ]:
# ============================================================
# CELL 11 — EXACT RADIAL–ANGULAR TARGET RECONSTRUCTION
#
# No model is fitted.
# No category label is used.
# ============================================================

print("=" * 72)
print("🛡️ CELL 11 — RADIAL–ANGULAR TARGET LOCK")
print("=" * 72)

# ------------------------------------------------------------
# 1. Recover frozen radial coordinates and fields
# ------------------------------------------------------------

shield_radial_centers_full = np.asarray(
    shield_post25_backup[
        "radial_centers"
    ],
    dtype=np.float64,
)

shield_f2_peak_magnitude = np.asarray(
    shield_post25_backup[
        "F2_peak_magnitude"
    ],
    dtype=np.float64,
)

shield_f2_peak_radius = np.asarray(
    shield_post25_backup[
        "F2_peak_radius"
    ],
    dtype=np.float64,
)

shield_r2_observed_field = np.asarray(
    shield_post25_backup[
        "R2_obs"
    ],
    dtype=np.float64,
)

shield_mu2_observed_deg = np.asarray(
    shield_post25_backup[
        "mu2_obs_deg"
    ],
    dtype=np.float64,
)

shield_mu2_learned_deg = np.asarray(
    shield_post25_backup[
        "mu2_hat_field_deg"
    ],
    dtype=np.float64,
)

print("\nFROZEN INPUT SHAPES")
print("-" * 72)
print(
    f"Full radial centers     : "
    f"{shield_radial_centers_full.shape}"
)
print(
    f"F2 peak magnitude       : "
    f"{shield_f2_peak_magnitude.shape}"
)
print(
    f"F2 peak radius          : "
    f"{shield_f2_peak_radius.shape}"
)
print(
    f"Observed R2 field       : "
    f"{shield_r2_observed_field.shape}"
)
print(
    f"Observed mu2 field      : "
    f"{shield_mu2_observed_deg.shape}"
)
print(
    f"Learned mu2 field       : "
    f"{shield_mu2_learned_deg.shape}"
)

# ------------------------------------------------------------
# 2. Recover the locked 25-shell radial domain
# ------------------------------------------------------------

shield_circular_domain_mask = (
    (shield_radial_centers_full >= 3.5)
    &
    (shield_radial_centers_full <= 27.5)
)

shield_circular_radial_centers = (
    shield_radial_centers_full[
        shield_circular_domain_mask
    ]
)

if shield_circular_radial_centers.shape != (25,):
    raise ValueError(
        "Expected 25 circular-domain radial centers, "
        f"found {shield_circular_radial_centers.shape}"
    )

if not np.allclose(
    shield_circular_radial_centers,
    np.arange(3.5, 28.5, 1.0),
):
    raise ValueError(
        "Recovered circular radial centers do not match "
        "the expected 3.5 → 27.5 shell sequence."
    )

print("\nLOCKED CIRCULAR DOMAIN")
print("-" * 72)
print(
    f"First center            : "
    f"{shield_circular_radial_centers[0]:.1f}"
)
print(
    f"Last center             : "
    f"{shield_circular_radial_centers[-1]:.1f}"
)
print(
    f"Number of shells        : "
    f"{len(shield_circular_radial_centers)}"
)

# ------------------------------------------------------------
# 3. Match each F2 peak to its circular shell
# ------------------------------------------------------------

shield_peak_shell_indices = np.argmin(
    np.abs(
        shield_f2_peak_radius[:, None]
        - shield_circular_radial_centers[
            None, :
        ]
    ),
    axis=1,
)

shield_matched_peak_radii = (
    shield_circular_radial_centers[
        shield_peak_shell_indices
    ]
)

shield_peak_shell_mismatch = np.abs(
    shield_f2_peak_radius
    - shield_matched_peak_radii
)

shield_maximum_shell_mismatch = float(
    shield_peak_shell_mismatch.max()
)

print("\nF2-PEAK TO CIRCULAR-SHELL MATCH")
print("-" * 72)
print(
    f"Maximum mismatch        : "
    f"{shield_maximum_shell_mismatch:.12f}"
)

if not np.allclose(
    shield_peak_shell_mismatch,
    0.0,
    atol=1e-12,
):
    raise ValueError(
        "At least one F2 peak does not match the "
        "locked circular shell exactly."
    )

# ------------------------------------------------------------
# 4. Extract sketch-level R2 at the matched peak shell
# ------------------------------------------------------------

shield_row_indices = np.arange(2300)

shield_r2_at_f2_peak = (
    shield_r2_observed_field[
        shield_row_indices,
        shield_peak_shell_indices,
    ]
)

# ------------------------------------------------------------
# 5. Compute axial angular recovery error
#
# Axial angles are equivalent modulo 180 degrees.
# The resulting error is constrained to [0, 90].
# ------------------------------------------------------------

shield_mu2_observed_at_peak = (
    shield_mu2_observed_deg[
        shield_row_indices,
        shield_peak_shell_indices,
    ]
)

shield_mu2_learned_at_peak = (
    shield_mu2_learned_deg[
        shield_row_indices,
        shield_peak_shell_indices,
    ]
)

shield_axial_error_at_f2_peak = np.abs(
    (
        (
            shield_mu2_observed_at_peak
            - shield_mu2_learned_at_peak
            + 90.0
        )
        % 180.0
    )
    - 90.0
)

if (
    shield_axial_error_at_f2_peak.min()
    < 0.0
    or
    shield_axial_error_at_f2_peak.max()
    > 90.0
):
    raise ValueError(
        "Axial error is outside [0, 90] degrees."
    )

# ------------------------------------------------------------
# 6. Assemble and verify the four frozen targets
# ------------------------------------------------------------

shield_ra_targets = {
    "F2_peak_magnitude":
        shield_f2_peak_magnitude.copy(),

    "F2_peak_radius":
        shield_f2_peak_radius.copy(),

    "R2_at_F2_peak":
        shield_r2_at_f2_peak.copy(),

    "axial_error_at_F2_peak":
        shield_axial_error_at_f2_peak.copy(),
}

shield_expected_target_medians = {
    "F2_peak_magnitude": 0.039141,
    "F2_peak_radius": 21.500000,
    "R2_at_F2_peak": 0.436421,
    "axial_error_at_F2_peak": 6.132369,
}

shield_target_audit_records = []

for target_name, target_values in (
    shield_ra_targets.items()
):
    target_values = np.asarray(
        target_values,
        dtype=np.float64,
    )

    if target_values.shape != (2300,):
        raise ValueError(
            f"{target_name}: expected shape (2300,), "
            f"found {target_values.shape}"
        )

    if not np.isfinite(
        target_values
    ).all():
        raise ValueError(
            f"{target_name} contains non-finite values."
        )

    observed_median = float(
        np.median(target_values)
    )

    expected_median = (
        shield_expected_target_medians[
            target_name
        ]
    )

    shield_target_audit_records.append({
        "target": target_name,
        "minimum": target_values.min(),
        "median": observed_median,
        "mean": target_values.mean(),
        "maximum": target_values.max(),
        "expected_median": expected_median,
        "median_difference": (
            observed_median
            - expected_median
        ),
    })

    target_values.setflags(write=False)

shield_ra_target_audit = pd.DataFrame(
    shield_target_audit_records
)

print("\nFROZEN TARGET AUDIT")
print("-" * 72)

display(
    shield_ra_target_audit.round(9)
)

# The published medians were rounded to six decimals.
shield_target_median_match = np.all(
    np.abs(
        shield_ra_target_audit[
            "median_difference"
        ]
    ) < 1e-6
)

if not shield_target_median_match:
    raise RuntimeError(
        "At least one reconstructed target median does "
        "not reproduce the frozen six-decimal result."
    )

# ------------------------------------------------------------
# 7. Lock target vectors
# ------------------------------------------------------------

for target_values in (
    shield_ra_targets.values()
):
    target_values.setflags(
        write=False
    )

print("\n" + "=" * 72)
print("🟢 Four frozen RA targets reconstructed")
print("🟢 Exact F2-to-circular-shell matching verified")
print("🟢 All target vectors contain 2300 finite values")
print("🟢 All four frozen medians reproduced")
print("🟢 No category label entered target construction")
print("🟢 No predictive model was fitted")
print("=" * 72)

🛡️ CELL 11 — RADIAL–ANGULAR TARGET LOCK

FROZEN INPUT SHAPES
------------------------------------------------------------------------
Full radial centers     : (72,)
F2 peak magnitude       : (2300,)
F2 peak radius          : (2300,)
Observed R2 field       : (2300, 25)
Observed mu2 field      : (2300, 25)
Learned mu2 field       : (2300, 25)

LOCKED CIRCULAR DOMAIN
------------------------------------------------------------------------
First center            : 3.5
Last center             : 27.5
Number of shells        : 25

F2-PEAK TO CIRCULAR-SHELL MATCH
------------------------------------------------------------------------
Maximum mismatch        : 0.000000000000

FROZEN TARGET AUDIT
------------------------------------------------------------------------


,target,minimum,median,mean,maximum,expected_median,median_difference
0,F2_peak_magnitude,0.003810,0.039141,0.040096,0.144782,0.039141,2.170000e-07
1,F2_peak_radius,3.500000,21.500000,20.860870,27.500000,21.500000,0.000000e+00
2,R2_at_F2_peak,0.001144,0.436421,0.430006,0.847681,0.436421,1.610000e-07
3,axial_error_at_F2_peak,0.005167,6.132369,21.277244,89.957353,6.132369,9.900000e-08



🟢 Four frozen RA targets reconstructed
🟢 Exact F2-to-circular-shell matching verified
🟢 All target vectors contain 2300 finite values
🟢 All four frozen medians reproduced
🟢 No category label entered target construction
🟢 No predictive model was fitted


In [ ]:
# ============================================================
# CELL 12A — ORIGINAL REGRESSION-ESTIMATOR REPRODUCTION
#
# Tests the likely frozen estimator:
# StandardScaler + ordinary least-squares LinearRegression.
#
# Grouped validation will run only after reproduction.
# ============================================================

from sklearn.linear_model import LinearRegression

print("=" * 72)
print("🛡️ CELL 12A — REGRESSION ESTIMATOR REPRODUCTION")
print("=" * 72)

# ------------------------------------------------------------
# 1. Candidate frozen regression specification
# ------------------------------------------------------------

shield_regression_template = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "regressor",
        LinearRegression(
            fit_intercept=True,
        ),
    ),
])

shield_regression_configuration = {
    "predictors": "135-D frozen morphology",
    "scaler": "StandardScaler inside each training fold",
    "regressor": "ordinary least-squares LinearRegression",
    "fit_intercept": True,
    "positive_constraint": False,
    "feature_selection": False,
    "hyperparameter_search": False,
    "category_labels_used": False,
}

print("\nCANDIDATE ESTIMATOR")
print("-" * 72)

for key, value in (
    shield_regression_configuration.items()
):
    print(f"{key:25s}: {value}")

# ------------------------------------------------------------
# 2. Reusable OOF regression evaluator
# ------------------------------------------------------------

def shield_evaluate_regression(
    X,
    y,
    splits,
    target_name,
    split_name,
    model_template,
    retain_models=False,
):
    X = np.asarray(X)
    y = np.asarray(
        y,
        dtype=np.float64,
    )

    predictions = np.full(
        y.shape,
        np.nan,
        dtype=np.float64,
    )

    coverage = np.zeros(
        len(y),
        dtype=int,
    )

    fold_records = []
    fitted_models = []

    for fold_number, (
        train_index,
        test_index,
    ) in enumerate(
        splits,
        start=1,
    ):
        model = clone(
            model_template
        )

        model.fit(
            X[train_index],
            y[train_index],
        )

        fold_predictions = model.predict(
            X[test_index]
        )

        predictions[test_index] = (
            fold_predictions
        )

        coverage[test_index] += 1

        fold_spearman = spearmanr(
            y[test_index],
            fold_predictions,
        ).statistic

        fold_records.append({
            "target": target_name,
            "split_design": split_name,
            "fold": fold_number,
            "train_n": len(train_index),
            "test_n": len(test_index),
            "r2": r2_score(
                y[test_index],
                fold_predictions,
            ),
            "mae": mean_absolute_error(
                y[test_index],
                fold_predictions,
            ),
            "rmse": np.sqrt(
                mean_squared_error(
                    y[test_index],
                    fold_predictions,
                )
            ),
            "spearman_rho": fold_spearman,
        })

        if retain_models:
            fitted_models.append(model)

    if not np.all(coverage == 1):
        raise RuntimeError(
            f"{target_name}: OOF coverage is not one."
        )

    if not np.isfinite(
        predictions
    ).all():
        raise RuntimeError(
            f"{target_name}: predictions are non-finite."
        )

    pooled_spearman = spearmanr(
        y,
        predictions,
    ).statistic

    pooled_metrics = {
        "target": target_name,
        "split_design": split_name,
        "r2": r2_score(
            y,
            predictions,
        ),
        "mae": mean_absolute_error(
            y,
            predictions,
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y,
                predictions,
            )
        ),
        "spearman_rho": pooled_spearman,
    }

    return {
        "predictions": predictions,
        "pooled_metrics": pooled_metrics,
        "fold_metrics": pd.DataFrame(
            fold_records
        ),
        "models": fitted_models,
    }

# ------------------------------------------------------------
# 3. Frozen original reference
# ------------------------------------------------------------

shield_frozen_recovery_reference = {
    "F2_peak_magnitude": {
        "r2": 0.2961,
        "mae": 0.01315,
        "rmse": 0.01710,
        "spearman_rho": 0.6415,
    },
    "F2_peak_radius": {
        "r2": 0.0594,
        "mae": 4.0152,
        "rmse": 5.0096,
        "spearman_rho": 0.3417,
    },
    "R2_at_F2_peak": {
        "r2": 0.2170,
        "mae": 0.1258,
        "rmse": 0.1599,
        "spearman_rho": 0.5377,
    },
    "axial_error_at_F2_peak": {
        "r2": 0.1979,
        "mae": 20.1515,
        "rmse": 26.4623,
        "spearman_rho": 0.4400,
    },
}

# ------------------------------------------------------------
# 4. Reproduce original image-level recovery
# ------------------------------------------------------------

shield_original_recovery_results = {}
shield_original_recovery_rows = []

print("\nRUNNING ORIGINAL-SPLIT RECOVERY")
print("-" * 72)

for target_name, target_values in (
    shield_ra_targets.items()
):
    result = shield_evaluate_regression(
        X=shield_X_morphology_work,
        y=target_values,
        splits=shield_original_splits,
        target_name=target_name,
        split_name="Original StratifiedKFold",
        model_template=(
            shield_regression_template
        ),
        retain_models=False,
    )

    shield_original_recovery_results[
        target_name
    ] = result

    reproduced = result[
        "pooled_metrics"
    ]

    reference = (
        shield_frozen_recovery_reference[
            target_name
        ]
    )

    shield_original_recovery_rows.append({
        "target": target_name,

        "reproduced_r2":
            reproduced["r2"],
        "reference_r2":
            reference["r2"],
        "difference_r2":
            reproduced["r2"]
            - reference["r2"],

        "reproduced_mae":
            reproduced["mae"],
        "reference_mae":
            reference["mae"],
        "difference_mae":
            reproduced["mae"]
            - reference["mae"],

        "reproduced_rmse":
            reproduced["rmse"],
        "reference_rmse":
            reference["rmse"],
        "difference_rmse":
            reproduced["rmse"]
            - reference["rmse"],

        "reproduced_spearman":
            reproduced["spearman_rho"],
        "reference_spearman":
            reference["spearman_rho"],
        "difference_spearman":
            reproduced["spearman_rho"]
            - reference["spearman_rho"],
    })

    print(
        f"{target_name:28s} | "
        f"R²={reproduced['r2']:+.6f} | "
        f"MAE={reproduced['mae']:.6f} | "
        f"RMSE={reproduced['rmse']:.6f} | "
        f"rho={reproduced['spearman_rho']:+.6f}"
    )

shield_original_recovery_reproduction = (
    pd.DataFrame(
        shield_original_recovery_rows
    )
)

print("\nORIGINAL RECOVERY REPRODUCTION TABLE")
print("-" * 72)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 260,
):
    display(
        shield_original_recovery_reproduction.round(
            8
        )
    )

# ------------------------------------------------------------
# 5. Determine whether the estimator reproduces the frozen
#    four-decimal manuscript values
# ------------------------------------------------------------

shield_regression_reproduction_checks = []

for row in (
    shield_original_recovery_reproduction
    .itertuples(index=False)
):
    checks = {
        "target": row.target,

        "r2_matches_4dp": (
            round(row.reproduced_r2, 4)
            == round(row.reference_r2, 4)
        ),

        "mae_matches_reported_precision": (
            np.isclose(
                row.reproduced_mae,
                row.reference_mae,
                atol=5e-5,
            )
        ),

        "rmse_matches_reported_precision": (
            np.isclose(
                row.reproduced_rmse,
                row.reference_rmse,
                atol=5e-5,
            )
        ),

        "spearman_matches_4dp": (
            round(
                row.reproduced_spearman,
                4,
            )
            == round(
                row.reference_spearman,
                4,
            )
        ),
    }

    checks["all_metrics_match"] = all(
        value
        for key, value in checks.items()
        if key != "target"
    )

    shield_regression_reproduction_checks.append(
        checks
    )

shield_regression_reproduction_check_table = (
    pd.DataFrame(
        shield_regression_reproduction_checks
    )
)

print("\nREPRODUCTION CHECK")
print("-" * 72)
display(
    shield_regression_reproduction_check_table
)

shield_regression_estimator_reproduced = bool(
    shield_regression_reproduction_check_table[
        "all_metrics_match"
    ].all()
)

print(
    "\nEstimator reproduces all frozen metrics: "
    f"{shield_regression_estimator_reproduced}"
)

if shield_regression_estimator_reproduced:
    print(
        "🟢 Original estimator identified as "
        "StandardScaler + LinearRegression"
    )
else:
    print(
        "🟡 Estimator not yet identified exactly; "
        "do not run grouped recovery yet"
    )

print("\n" + "=" * 72)
print("🟢 Original folds and exact targets used")
print("🟢 No grouped result was fitted prematurely")
print("🟢 No category label entered regression")
print("🟢 No target-specific tuning was performed")
print("=" * 72)

🛡️ CELL 12A — REGRESSION ESTIMATOR REPRODUCTION

CANDIDATE ESTIMATOR
------------------------------------------------------------------------
predictors               : 135-D frozen morphology
scaler                   : StandardScaler inside each training fold
regressor                : ordinary least-squares LinearRegression
fit_intercept            : True
positive_constraint      : False
feature_selection        : False
hyperparameter_search    : False
category_labels_used     : False

RUNNING ORIGINAL-SPLIT RECOVERY
------------------------------------------------------------------------
F2_peak_magnitude            | R²=+0.314817 | MAE=0.012966 | RMSE=0.016872 | rho=+0.645210
F2_peak_radius               | R²=+0.051300 | MAE=3.995300 | RMSE=5.031136 | rho=+0.351723
R2_at_F2_peak                | R²=+0.231837 | MAE=0.125377 | RMSE=0.158361 | rho=+0.536785
axial_error_at_F2_peak       | R²=+0.196591 | MAE=20.054119 | RMSE=26.484385 | rho=+0.457398

ORIGINAL RECOVERY REPRODUCTION TABL

,target,reproduced_r2,reference_r2,difference_r2,reproduced_mae,reference_mae,difference_mae,reproduced_rmse,reference_rmse,difference_rmse,reproduced_spearman,reference_spearman,difference_spearman
0,F2_peak_magnitude,0.314817,0.2961,0.018717,0.012966,0.01315,-0.000184,0.016872,0.0171,-0.000228,0.645210,0.6415,0.003710
1,F2_peak_radius,0.051300,0.0594,-0.008100,3.995300,4.01520,-0.019900,5.031136,5.0096,0.021536,0.351723,0.3417,0.010023
2,R2_at_F2_peak,0.231837,0.2170,0.014837,0.125377,0.12580,-0.000423,0.158361,0.1599,-0.001539,0.536785,0.5377,-0.000915
3,axial_error_at_F2_peak,0.196591,0.1979,-0.001309,20.054119,20.15150,-0.097381,26.484385,26.4623,0.022085,0.457398,0.4400,0.017398



REPRODUCTION CHECK
------------------------------------------------------------------------


,target,r2_matches_4dp,mae_matches_reported_precision,rmse_matches_reported_precision,spearman_matches_4dp,all_metrics_match
0,F2_peak_magnitude,False,False,False,False,False
1,F2_peak_radius,False,False,False,False,False
2,R2_at_F2_peak,False,False,False,False,False
3,axial_error_at_F2_peak,False,False,False,False,False



Estimator reproduces all frozen metrics: False
🟡 Estimator not yet identified exactly; do not run grouped recovery yet

🟢 Original folds and exact targets used
🟢 No grouped result was fitted prematurely
🟢 No category label entered regression
🟢 No target-specific tuning was performed


In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

shield_recovery_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

shield_recovery_model = Pipeline([
    ("standardization", StandardScaler()),
    ("ridge", Ridge(alpha=1.0)),
])

In [ ]:
# ============================================================
# CELL 12B-0 — CANONICAL MORPHOLOGY ALIAS
# ============================================================

import hashlib

print("=" * 72)
print("🛡️ CELL 12B-0 — CANONICAL MORPHOLOGY ALIAS")
print("=" * 72)

SHIELD_EXPECTED_MORPHOLOGY_SHA256 = (
    "66ae04156ee3fbf3f2605f382a16fc41"
    "cf19af34b50e59dd43f6c9427d96b2ee"
)

# Exact canonical object reconstructed and validated in Cell 5B
assert "shield_X_morphology" in globals()

shield_X_raw = shield_X_morphology

assert isinstance(shield_X_raw, np.ndarray)
assert shield_X_raw.shape == (2300, 135)
assert shield_X_raw.dtype == np.float32
assert np.isfinite(shield_X_raw).all()

shield_resolved_hash = hashlib.sha256(
    np.ascontiguousarray(shield_X_raw).tobytes()
).hexdigest()

assert (
    shield_resolved_hash
    == SHIELD_EXPECTED_MORPHOLOGY_SHA256
)

# Preserve canonical matrix as read-only
shield_X_raw.setflags(write=False)

print("\nRESOLVED MORPHOLOGY")
print("-" * 72)
print("Original object name    : shield_X_morphology")
print("Canonical alias         : shield_X_raw")
print(f"Shape                   : {shield_X_raw.shape}")
print(f"Dtype                   : {shield_X_raw.dtype}")
print(f"Finite                  : {np.isfinite(shield_X_raw).all()}")
print(f"SHA-256                 : {shield_resolved_hash}")
print(f"Writeable               : {shield_X_raw.flags.writeable}")

print("\n" + "=" * 72)
print("🟢 Exact frozen float32 morphology selected directly")
print("🟢 Float64 working and temporary copies excluded")
print("🟢 No morphology value was modified")
print("=" * 72)

🛡️ CELL 12B-0 — CANONICAL MORPHOLOGY ALIAS

RESOLVED MORPHOLOGY
------------------------------------------------------------------------
Original object name    : shield_X_morphology
Canonical alias         : shield_X_raw
Shape                   : (2300, 135)
Dtype                   : float32
Finite                  : True
SHA-256                 : 66ae04156ee3fbf3f2605f382a16fc41cf19af34b50e59dd43f6c9427d96b2ee
Writeable               : False

🟢 Exact frozen float32 morphology selected directly
🟢 Float64 working and temporary copies excluded
🟢 No morphology value was modified


In [ ]:
# ============================================================
# CELL 12B — EXACT HISTORICAL RIDGE/KFOLD REPRODUCTION
# ============================================================

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)
from scipy.stats import pearsonr, spearmanr

print("=" * 72)
print("🛡️ CELL 12B — EXACT HISTORICAL RIDGE/KFOLD REPRODUCTION")
print("=" * 72)

# ------------------------------------------------------------
# 1. Lock the exact morphology predictors
# ------------------------------------------------------------

assert "shield_X_raw" in globals()

# Match the historical notebook exactly:
# X = np.asarray(X_raw, dtype=float)

shield_recovery_X = np.asarray(
    shield_X_raw,
    dtype=float,
)

assert shield_recovery_X.dtype == np.float64


assert np.isfinite(shield_recovery_X).all()

shield_recovery_X.setflags(write=False)

# ------------------------------------------------------------
# 2. Lock the exact Cell 11 targets
# ------------------------------------------------------------

assert "shield_ra_targets" in globals()

shield_required_target_keys = {
    "F2_peak_magnitude",
    "F2_peak_radius",
    "R2_at_F2_peak",
    "axial_error_at_F2_peak",
}

assert shield_required_target_keys.issubset(
    shield_ra_targets.keys()
)

# The historical notebook called the fourth target
# "axial_error". Cell 11 uses the more descriptive name
# "axial_error_at_F2_peak".

shield_recovery_targets = {
    "F2_peak_magnitude": np.asarray(
        shield_ra_targets["F2_peak_magnitude"],
        dtype=float,
    ).copy(),

    "F2_peak_radius": np.asarray(
        shield_ra_targets["F2_peak_radius"],
        dtype=float,
    ).copy(),

    "R2_at_F2_peak": np.asarray(
        shield_ra_targets["R2_at_F2_peak"],
        dtype=float,
    ).copy(),

    "axial_error": np.asarray(
        shield_ra_targets["axial_error_at_F2_peak"],
        dtype=float,
    ).copy(),
}

for shield_target_name, shield_target_values in (
    shield_recovery_targets.items()
):
    assert shield_target_values.shape == (2300,)
    assert np.isfinite(shield_target_values).all()
    shield_target_values.setflags(write=False)

# ------------------------------------------------------------
# 3. Reconstruct the exact historical estimator and CV
# ------------------------------------------------------------

shield_recovery_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

shield_recovery_model = Pipeline([
    (
        "standardization",
        StandardScaler(),
    ),
    (
        "ridge",
        Ridge(alpha=1.0),
    ),
])

print("\nESTIMATOR SPECIFICATION")
print("-" * 72)
print("Predictors              : 135-D frozen morphology")
print("Scaler                  : StandardScaler within each fold")
print("Regressor               : Ridge(alpha=1.0)")
print("Cross-validation        : KFold(n_splits=5, shuffle=True)")
print("Random state            : 42")
print("Prediction aggregation  : pooled out-of-fold predictions")
print("Hyperparameter search   : none")
print("Category labels used    : False")

# ------------------------------------------------------------
# 4. Historical reference values
# ------------------------------------------------------------

shield_historical_reference = {
    "F2_peak_magnitude": {
        "r2": 0.296073,
        "mae": 0.013148,
        "rmse": 0.017101,
        "spearman": 0.641500,
    },
    "F2_peak_radius": {
        "r2": 0.059397,
        "mae": 4.015234,
        "rmse": 5.009620,
        "spearman": 0.341674,
    },
    "R2_at_F2_peak": {
        "r2": 0.217002,
        "mae": 0.125806,
        "rmse": 0.159882,
        "spearman": 0.537740,
    },
    "axial_error": {
        "r2": 0.197931,
        "mae": 20.151463,
        "rmse": 26.462284,
        "spearman": 0.439962,
    },
}

# ------------------------------------------------------------
# 5. Generate exact historical OOF predictions
# ------------------------------------------------------------

shield_recovery_predictions = {}
shield_recovery_records = []

print("\nRUNNING HISTORICAL RIDGE/KFOLD RECOVERY")
print("-" * 72)

for shield_target_name, shield_y in (
    shield_recovery_targets.items()
):

    shield_y_pred = cross_val_predict(
        estimator=shield_recovery_model,
        X=shield_recovery_X,
        y=shield_y,
        cv=shield_recovery_cv,
        method="predict",
        n_jobs=None,
    )

    shield_y_pred = np.asarray(
        shield_y_pred,
        dtype=float,
    )

    assert shield_y_pred.shape == (2300,)
    assert np.isfinite(shield_y_pred).all()

    shield_r2 = r2_score(
        shield_y,
        shield_y_pred,
    )

    shield_mae = mean_absolute_error(
        shield_y,
        shield_y_pred,
    )

    shield_rmse = np.sqrt(
        mean_squared_error(
            shield_y,
            shield_y_pred,
        )
    )

    shield_pearson_r, shield_pearson_p = pearsonr(
        shield_y,
        shield_y_pred,
    )

    shield_spearman_rho, shield_spearman_p = spearmanr(
        shield_y,
        shield_y_pred,
    )

    shield_reference = shield_historical_reference[
        shield_target_name
    ]

    shield_recovery_predictions[
        shield_target_name
    ] = shield_y_pred

    shield_recovery_records.append({
        "target": shield_target_name,

        "reproduced_r2":
            float(shield_r2),
        "reference_r2":
            shield_reference["r2"],
        "difference_r2":
            float(shield_r2)
            - shield_reference["r2"],

        "reproduced_mae":
            float(shield_mae),
        "reference_mae":
            shield_reference["mae"],
        "difference_mae":
            float(shield_mae)
            - shield_reference["mae"],

        "reproduced_rmse":
            float(shield_rmse),
        "reference_rmse":
            shield_reference["rmse"],
        "difference_rmse":
            float(shield_rmse)
            - shield_reference["rmse"],

        "reproduced_pearson":
            float(shield_pearson_r),
        "pearson_p":
            float(shield_pearson_p),

        "reproduced_spearman":
            float(shield_spearman_rho),
        "reference_spearman":
            shield_reference["spearman"],
        "difference_spearman":
            float(shield_spearman_rho)
            - shield_reference["spearman"],
        "spearman_p":
            float(shield_spearman_p),
    })

    print(
        f"{shield_target_name:28s} | "
        f"R²={shield_r2:+.6f} | "
        f"MAE={shield_mae:.6f} | "
        f"RMSE={shield_rmse:.6f} | "
        f"rho={shield_spearman_rho:+.6f}"
    )

shield_recovery_results = pd.DataFrame(
    shield_recovery_records
)

# ------------------------------------------------------------
# 6. Reproduction audit at reported precision
# ------------------------------------------------------------

shield_reproduction_check = pd.DataFrame({
    "target":
        shield_recovery_results["target"],

    "r2_matches_6dp":
        np.isclose(
            shield_recovery_results["reproduced_r2"],
            shield_recovery_results["reference_r2"],
            atol=5e-7,
            rtol=0.0,
        ),

    "mae_matches_6dp":
        np.isclose(
            shield_recovery_results["reproduced_mae"],
            shield_recovery_results["reference_mae"],
            atol=5e-7,
            rtol=0.0,
        ),

    "rmse_matches_6dp":
        np.isclose(
            shield_recovery_results["reproduced_rmse"],
            shield_recovery_results["reference_rmse"],
            atol=5e-7,
            rtol=0.0,
        ),

    "spearman_matches_6dp":
        np.isclose(
            shield_recovery_results["reproduced_spearman"],
            shield_recovery_results["reference_spearman"],
            atol=5e-7,
            rtol=0.0,
        ),
})

shield_reproduction_check["all_metrics_match"] = (
    shield_reproduction_check[
        [
            "r2_matches_6dp",
            "mae_matches_6dp",
            "rmse_matches_6dp",
            "spearman_matches_6dp",
        ]
    ].all(axis=1)
)

shield_estimator_reproduced = bool(
    shield_reproduction_check[
        "all_metrics_match"
    ].all()
)

print("\nEXACT REPRODUCTION TABLE")
print("-" * 72)

display(
    shield_recovery_results[
        [
            "target",
            "reproduced_r2",
            "reference_r2",
            "difference_r2",
            "reproduced_mae",
            "reference_mae",
            "difference_mae",
            "reproduced_rmse",
            "reference_rmse",
            "difference_rmse",
            "reproduced_spearman",
            "reference_spearman",
            "difference_spearman",
        ]
    ]
)

print("\nREPRODUCTION CHECK")
print("-" * 72)

display(shield_reproduction_check)

print(
    "\nEstimator reproduces all historical metrics:",
    shield_estimator_reproduced,
)

if not shield_estimator_reproduced:
    raise RuntimeError(
        "Historical Ridge/KFold results were not reproduced "
        "at the reported six-decimal precision. Do not "
        "proceed to grouped recovery."
    )

# ------------------------------------------------------------
# 7. Lock OOF predictions
# ------------------------------------------------------------

for shield_prediction in (
    shield_recovery_predictions.values()
):
    shield_prediction.setflags(write=False)

print("\n" + "=" * 72)
print("🟢 Historical Ridge(alpha=1.0) estimator reproduced")
print("🟢 Historical shuffled KFold(random_state=42) reproduced")
print("🟢 Scaling remained inside every training fold")
print("🟢 All predictions were strictly out of fold")
print("🟢 No category label entered regression")
print("🟢 Ready for source-grouped recovery analysis")
print("=" * 72)

🛡️ CELL 12B — EXACT HISTORICAL RIDGE/KFOLD REPRODUCTION

ESTIMATOR SPECIFICATION
------------------------------------------------------------------------
Predictors              : 135-D frozen morphology
Scaler                  : StandardScaler within each fold
Regressor               : Ridge(alpha=1.0)
Cross-validation        : KFold(n_splits=5, shuffle=True)
Random state            : 42
Prediction aggregation  : pooled out-of-fold predictions
Hyperparameter search   : none
Category labels used    : False

RUNNING HISTORICAL RIDGE/KFOLD RECOVERY
------------------------------------------------------------------------
F2_peak_magnitude            | R²=+0.296073 | MAE=0.013148 | RMSE=0.017101 | rho=+0.641500
F2_peak_radius               | R²=+0.059397 | MAE=4.015234 | RMSE=5.009620 | rho=+0.341674
R2_at_F2_peak                | R²=+0.217002 | MAE=0.125806 | RMSE=0.159882 | rho=+0.537740
axial_error                  | R²=+0.197931 | MAE=20.151463 | RMSE=26.462284 | rho=+0.439962

EXACT R

,target,reproduced_r2,reference_r2,difference_r2,reproduced_mae,reference_mae,difference_mae,reproduced_rmse,reference_rmse,difference_rmse,reproduced_spearman,reference_spearman,difference_spearman
0,F2_peak_magnitude,0.296073,0.296073,6.768866e-08,0.013148,0.013148,3.714476e-08,0.017101,0.017101,3.554531e-07,0.641500,0.641500,4.766542e-07
1,F2_peak_radius,0.059397,0.059397,4.915008e-07,4.015234,4.015234,2.738593e-07,5.009620,5.009620,-4.106095e-07,0.341674,0.341674,4.339519e-07
2,R2_at_F2_peak,0.217002,0.217002,2.885636e-07,0.125806,0.125806,-4.027301e-07,0.159882,0.159882,3.756792e-07,0.537740,0.537740,9.146069e-08
3,axial_error,0.197931,0.197931,9.287301e-08,20.151463,20.151463,1.396144e-07,26.462284,26.462284,-5.514026e-08,0.439962,0.439962,2.148363e-07



REPRODUCTION CHECK
------------------------------------------------------------------------


,target,r2_matches_6dp,mae_matches_6dp,rmse_matches_6dp,spearman_matches_6dp,all_metrics_match
0,F2_peak_magnitude,True,True,True,True,True
1,F2_peak_radius,True,True,True,True,True
2,R2_at_F2_peak,True,True,True,True,True
3,axial_error,True,True,True,True,True



Estimator reproduces all historical metrics: True

🟢 Historical Ridge(alpha=1.0) estimator reproduced
🟢 Historical shuffled KFold(random_state=42) reproduced
🟢 Scaling remained inside every training fold
🟢 All predictions were strictly out of fold
🟢 No category label entered regression
🟢 Ready for source-grouped recovery analysis


In [ ]:
# ============================================================
# CELL 13 — SOURCE-GROUPED MORPHOLOGY-TO-RA RECOVERY
# ============================================================

from sklearn.base import clone
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)
from scipy.stats import pearsonr, spearmanr

print("=" * 72)
print("🛡️ CELL 13 — SOURCE-GROUPED MORPHOLOGY-TO-RA RECOVERY")
print("=" * 72)

# ------------------------------------------------------------
# 1. Prerequisite audit
# ------------------------------------------------------------

shield_required_objects = [
    "shield_recovery_X",
    "shield_recovery_targets",
    "shield_recovery_model",
    "shield_recovery_results",
    "shield_group_splits",
    "shield_groups",
    "shield_labels",
]

for shield_required_name in shield_required_objects:
    if shield_required_name not in globals():
        raise RuntimeError(
            f"Required object missing: {shield_required_name}"
        )

assert shield_recovery_X.shape == (2300, 135)
assert shield_recovery_X.dtype == np.float64
assert np.isfinite(shield_recovery_X).all()

assert len(shield_group_splits) == 5
assert len(shield_groups) == 2300
assert len(shield_labels) == 2300

# ------------------------------------------------------------
# 2. Re-audit grouped splits
# ------------------------------------------------------------

shield_grouped_test_counts = np.zeros(
    2300,
    dtype=int,
)

print("\nGROUPED-FOLD INTEGRITY")
print("-" * 72)

for shield_fold_number, (
    shield_train_index,
    shield_test_index,
) in enumerate(
    shield_group_splits,
    start=1,
):

    shield_train_index = np.asarray(
        shield_train_index,
        dtype=int,
    )

    shield_test_index = np.asarray(
        shield_test_index,
        dtype=int,
    )

    shield_train_groups = set(
        shield_groups[shield_train_index]
    )

    shield_test_groups = set(
        shield_groups[shield_test_index]
    )

    shield_overlap = (
        shield_train_groups
        .intersection(shield_test_groups)
    )

    shield_test_categories = np.unique(
        shield_labels[shield_test_index]
    )

    shield_grouped_test_counts[
        shield_test_index
    ] += 1

    print(
        f"Fold {shield_fold_number}: "
        f"train={len(shield_train_index):4d}, "
        f"test={len(shield_test_index):3d}, "
        f"train identities={len(shield_train_groups):3d}, "
        f"test identities={len(shield_test_groups):2d}, "
        f"overlap={len(shield_overlap)}, "
        f"categories={len(shield_test_categories)}"
    )

    assert len(shield_overlap) == 0
    assert len(shield_test_categories) == 23

assert np.all(shield_grouped_test_counts == 1)

print("🟢 Every row appears in exactly one grouped test fold")

# ------------------------------------------------------------
# 3. Fixed grouped out-of-fold prediction
# ------------------------------------------------------------

shield_grouped_recovery_predictions = {}
shield_grouped_recovery_fold_records = []
shield_grouped_recovery_records = []

print("\nRUNNING SOURCE-GROUPED RECOVERY")
print("-" * 72)

for shield_target_name, shield_y in (
    shield_recovery_targets.items()
):

    shield_grouped_y_pred = np.full(
        shape=len(shield_y),
        fill_value=np.nan,
        dtype=float,
    )

    for shield_fold_number, (
        shield_train_index,
        shield_test_index,
    ) in enumerate(
        shield_group_splits,
        start=1,
    ):

        shield_train_index = np.asarray(
            shield_train_index,
            dtype=int,
        )

        shield_test_index = np.asarray(
            shield_test_index,
            dtype=int,
        )

        shield_fold_model = clone(
            shield_recovery_model
        )

        shield_fold_model.fit(
            shield_recovery_X[shield_train_index],
            shield_y[shield_train_index],
        )

        shield_fold_prediction = (
            shield_fold_model.predict(
                shield_recovery_X[shield_test_index]
            )
        )

        shield_grouped_y_pred[
            shield_test_index
        ] = shield_fold_prediction

        shield_fold_r2 = r2_score(
            shield_y[shield_test_index],
            shield_fold_prediction,
        )

        shield_fold_mae = mean_absolute_error(
            shield_y[shield_test_index],
            shield_fold_prediction,
        )

        shield_fold_rmse = np.sqrt(
            mean_squared_error(
                shield_y[shield_test_index],
                shield_fold_prediction,
            )
        )

        shield_fold_spearman, _ = spearmanr(
            shield_y[shield_test_index],
            shield_fold_prediction,
        )

        shield_grouped_recovery_fold_records.append({
            "target": shield_target_name,
            "fold": shield_fold_number,
            "train_rows": len(shield_train_index),
            "test_rows": len(shield_test_index),
            "r2": float(shield_fold_r2),
            "mae": float(shield_fold_mae),
            "rmse": float(shield_fold_rmse),
            "spearman": float(shield_fold_spearman),
        })

    assert np.isfinite(
        shield_grouped_y_pred
    ).all()

    shield_grouped_r2 = r2_score(
        shield_y,
        shield_grouped_y_pred,
    )

    shield_grouped_mae = mean_absolute_error(
        shield_y,
        shield_grouped_y_pred,
    )

    shield_grouped_rmse = np.sqrt(
        mean_squared_error(
            shield_y,
            shield_grouped_y_pred,
        )
    )

    (
        shield_grouped_pearson,
        shield_grouped_pearson_p,
    ) = pearsonr(
        shield_y,
        shield_grouped_y_pred,
    )

    (
        shield_grouped_spearman,
        shield_grouped_spearman_p,
    ) = spearmanr(
        shield_y,
        shield_grouped_y_pred,
    )

    shield_historical_row = (
        shield_recovery_results
        .loc[
            shield_recovery_results["target"]
            == shield_target_name
        ]
        .iloc[0]
    )

    shield_grouped_recovery_predictions[
        shield_target_name
    ] = shield_grouped_y_pred

    shield_grouped_recovery_records.append({
        "target":
            shield_target_name,

        "historical_image_level_r2":
            float(
                shield_historical_row[
                    "reproduced_r2"
                ]
            ),

        "grouped_unseen_identity_r2":
            float(shield_grouped_r2),

        "r2_change_grouped_minus_historical":
            float(
                shield_grouped_r2
                - shield_historical_row[
                    "reproduced_r2"
                ]
            ),

        "historical_image_level_mae":
            float(
                shield_historical_row[
                    "reproduced_mae"
                ]
            ),

        "grouped_unseen_identity_mae":
            float(shield_grouped_mae),

        "mae_change_grouped_minus_historical":
            float(
                shield_grouped_mae
                - shield_historical_row[
                    "reproduced_mae"
                ]
            ),

        "historical_image_level_rmse":
            float(
                shield_historical_row[
                    "reproduced_rmse"
                ]
            ),

        "grouped_unseen_identity_rmse":
            float(shield_grouped_rmse),

        "rmse_change_grouped_minus_historical":
            float(
                shield_grouped_rmse
                - shield_historical_row[
                    "reproduced_rmse"
                ]
            ),

        "historical_image_level_spearman":
            float(
                shield_historical_row[
                    "reproduced_spearman"
                ]
            ),

        "grouped_unseen_identity_spearman":
            float(shield_grouped_spearman),

        "spearman_change_grouped_minus_historical":
            float(
                shield_grouped_spearman
                - shield_historical_row[
                    "reproduced_spearman"
                ]
            ),

        "grouped_pearson":
            float(shield_grouped_pearson),

        "grouped_pearson_p":
            float(shield_grouped_pearson_p),

        "grouped_spearman_p":
            float(shield_grouped_spearman_p),
    })

    print(
        f"{shield_target_name:28s} | "
        f"R²={shield_grouped_r2:+.6f} | "
        f"MAE={shield_grouped_mae:.6f} | "
        f"RMSE={shield_grouped_rmse:.6f} | "
        f"rho={shield_grouped_spearman:+.6f}"
    )

shield_grouped_recovery_results = pd.DataFrame(
    shield_grouped_recovery_records
)

shield_grouped_recovery_fold_results = pd.DataFrame(
    shield_grouped_recovery_fold_records
)

# ------------------------------------------------------------
# 4. Main comparison table
# ------------------------------------------------------------

print("\nHISTORICAL VS UNSEEN-IDENTITY RECOVERY")
print("-" * 72)

display(
    shield_grouped_recovery_results[
        [
            "target",
            "historical_image_level_r2",
            "grouped_unseen_identity_r2",
            "r2_change_grouped_minus_historical",
            "historical_image_level_mae",
            "grouped_unseen_identity_mae",
            "historical_image_level_rmse",
            "grouped_unseen_identity_rmse",
            "historical_image_level_spearman",
            "grouped_unseen_identity_spearman",
            "spearman_change_grouped_minus_historical",
        ]
    ]
)

# ------------------------------------------------------------
# 5. Fold-level stability
# ------------------------------------------------------------

shield_grouped_fold_summary = (
    shield_grouped_recovery_fold_results
    .groupby("target", as_index=False)
    .agg(
        mean_fold_r2=("r2", "mean"),
        minimum_fold_r2=("r2", "min"),
        maximum_fold_r2=("r2", "max"),
        positive_r2_folds=(
            "r2",
            lambda shield_values: int(
                np.sum(
                    np.asarray(shield_values) > 0
                )
            ),
        ),
        mean_fold_spearman=("spearman", "mean"),
        minimum_fold_spearman=("spearman", "min"),
        maximum_fold_spearman=("spearman", "max"),
        positive_spearman_folds=(
            "spearman",
            lambda shield_values: int(
                np.sum(
                    np.asarray(shield_values) > 0
                )
            ),
        ),
    )
)

print("\nGROUPED FOLD STABILITY")
print("-" * 72)

display(shield_grouped_fold_summary)

# ------------------------------------------------------------
# 6. Lock results
# ------------------------------------------------------------

for shield_prediction in (
    shield_grouped_recovery_predictions.values()
):
    shield_prediction.setflags(write=False)

print("\nINTERPRETATION BOUNDARY")
print("-" * 72)
print(
    "These results measure morphology-to-RA recovery for "
    "previously unseen source-garment identities."
)
print(
    "The estimator and preprocessing are fixed from the "
    "historical analysis; only the CV design changed."
)
print(
    "The axial-error target is a scalar disagreement magnitude, "
    "not direct circular-orientation recovery."
)

print("\n" + "=" * 72)
print("🟢 Identical fixed Ridge estimator retained")
print("🟢 Scaling fitted inside every grouped training fold")
print("🟢 No source garment crossed training and testing")
print("🟢 Every row received one grouped OOF prediction")
print("🟢 Historical and grouped recovery directly compared")
print("🟢 No category label entered regression")
print("=" * 72)

🛡️ CELL 13 — SOURCE-GROUPED MORPHOLOGY-TO-RA RECOVERY

GROUPED-FOLD INTEGRITY
------------------------------------------------------------------------
Fold 1: train=1839, test=461, train identities=184, test identities=46, overlap=0, categories=23
Fold 2: train=1840, test=460, train identities=184, test identities=46, overlap=0, categories=23
Fold 3: train=1841, test=459, train identities=184, test identities=46, overlap=0, categories=23
Fold 4: train=1840, test=460, train identities=184, test identities=46, overlap=0, categories=23
Fold 5: train=1840, test=460, train identities=184, test identities=46, overlap=0, categories=23
🟢 Every row appears in exactly one grouped test fold

RUNNING SOURCE-GROUPED RECOVERY
------------------------------------------------------------------------
F2_peak_magnitude            | R²=+0.302221 | MAE=0.013196 | RMSE=0.017027 | rho=+0.631055
F2_peak_radius               | R²=+0.014269 | MAE=4.080491 | RMSE=5.128389 | rho=+0.324874
R2_at_F2_peak          

,target,historical_image_level_r2,grouped_unseen_identity_r2,r2_change_grouped_minus_historical,historical_image_level_mae,grouped_unseen_identity_mae,historical_image_level_rmse,grouped_unseen_identity_rmse,historical_image_level_spearman,grouped_unseen_identity_spearman,spearman_change_grouped_minus_historical
0,F2_peak_magnitude,0.296073,0.302221,0.006148,0.013148,0.013196,0.017101,0.017027,0.641500,0.631055,-0.010445
1,F2_peak_radius,0.059397,0.014269,-0.045129,4.015234,4.080491,5.009620,5.128389,0.341674,0.324874,-0.016800
2,R2_at_F2_peak,0.217002,0.190971,-0.026032,0.125806,0.127284,0.159882,0.162518,0.537740,0.521587,-0.016153
3,axial_error,0.197931,0.206346,0.008415,20.151463,20.041103,26.462284,26.323096,0.439962,0.442901,0.002939



GROUPED FOLD STABILITY
------------------------------------------------------------------------


,target,mean_fold_r2,minimum_fold_r2,maximum_fold_r2,positive_r2_folds,mean_fold_spearman,minimum_fold_spearman,maximum_fold_spearman,positive_spearman_folds
0,F2_peak_magnitude,0.302248,0.293562,0.331021,5,0.631811,0.603840,0.654837,5
1,F2_peak_radius,0.009498,-0.065499,0.063080,3,0.328111,0.244861,0.379067,5
2,R2_at_F2_peak,0.190611,0.119192,0.224770,5,0.520671,0.497437,0.554715,5
3,axial_error,0.205264,0.101959,0.267321,5,0.442411,0.379785,0.495222,5



INTERPRETATION BOUNDARY
------------------------------------------------------------------------
These results measure morphology-to-RA recovery for previously unseen source-garment identities.
The estimator and preprocessing are fixed from the historical analysis; only the CV design changed.
The axial-error target is a scalar disagreement magnitude, not direct circular-orientation recovery.

🟢 Identical fixed Ridge estimator retained
🟢 Scaling fitted inside every grouped training fold
🟢 No source garment crossed training and testing
🟢 Every row received one grouped OOF prediction
🟢 Historical and grouped recovery directly compared
🟢 No category label entered regression


In [ ]:
# ============================================================
# CELL 14 — IDENTITY-AWARE BOOTSTRAP FOR GROUPED RECOVERY
# ============================================================

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)
from scipy.stats import spearmanr

print("=" * 72)
print("🛡️ CELL 14 — IDENTITY-AWARE BOOTSTRAP FOR GROUPED RECOVERY")
print("=" * 72)

# ------------------------------------------------------------
# 1. Configuration and prerequisites
# ------------------------------------------------------------

SHIELD_RECOVERY_BOOTSTRAPS = 5000
SHIELD_RECOVERY_BOOTSTRAP_SEED = 20260820

shield_required_objects = [
    "shield_recovery_targets",
    "shield_grouped_recovery_predictions",
    "shield_grouped_recovery_results",
    "shield_groups",
    "shield_labels",
]

for shield_required_name in shield_required_objects:
    if shield_required_name not in globals():
        raise RuntimeError(
            f"Required object missing: {shield_required_name}"
        )

shield_bootstrap_groups = np.asarray(
    shield_groups,
    dtype=str,
)

shield_bootstrap_labels = np.asarray(
    shield_labels,
)

assert shield_bootstrap_groups.shape == (2300,)
assert shield_bootstrap_labels.shape == (2300,)

# ------------------------------------------------------------
# 2. Construct category-stratified identity map
# ------------------------------------------------------------

shield_identity_to_indices = {}
shield_identity_to_category = {}

for shield_identity in np.unique(
    shield_bootstrap_groups
):

    shield_identity_indices = np.flatnonzero(
        shield_bootstrap_groups == shield_identity
    )

    shield_identity_categories = np.unique(
        shield_bootstrap_labels[
            shield_identity_indices
        ]
    )

    assert len(shield_identity_categories) == 1

    shield_identity_to_indices[
        shield_identity
    ] = shield_identity_indices

    shield_identity_to_category[
        shield_identity
    ] = shield_identity_categories[0]

shield_categories = np.unique(
    shield_bootstrap_labels
)

shield_category_to_identities = {}

for shield_category in shield_categories:

    shield_category_identities = np.asarray([
        shield_identity
        for shield_identity, shield_identity_category
        in shield_identity_to_category.items()
        if shield_identity_category == shield_category
    ])

    shield_category_to_identities[
        shield_category
    ] = shield_category_identities

    assert len(shield_category_identities) == 10

assert len(shield_categories) == 23
assert len(shield_identity_to_indices) == 230

print("\nBOOTSTRAP DESIGN")
print("-" * 72)
print(f"Rows                         : {len(shield_bootstrap_groups)}")
print(f"Categories                   : {len(shield_categories)}")
print(f"Source identities            : {len(shield_identity_to_indices)}")
print("Identities per category      : 10")
print(f"Bootstrap replicates         : {SHIELD_RECOVERY_BOOTSTRAPS}")
print(f"Random seed                  : {SHIELD_RECOVERY_BOOTSTRAP_SEED}")
print("Sampling unit                : complete source identity")
print("Stratification               : within category")
print("Predictions                  : fixed grouped OOF predictions")
print("Model refitting              : False")

# ------------------------------------------------------------
# 3. Freeze observed metrics
# ------------------------------------------------------------

shield_observed_recovery = {}

for shield_target_name, shield_y in (
    shield_recovery_targets.items()
):

    shield_y_pred = (
        shield_grouped_recovery_predictions[
            shield_target_name
        ]
    )

    shield_observed_recovery[
        shield_target_name
    ] = {
        "r2": float(
            r2_score(
                shield_y,
                shield_y_pred,
            )
        ),
        "mae": float(
            mean_absolute_error(
                shield_y,
                shield_y_pred,
            )
        ),
        "rmse": float(
            np.sqrt(
                mean_squared_error(
                    shield_y,
                    shield_y_pred,
                )
            )
        ),
        "spearman": float(
            spearmanr(
                shield_y,
                shield_y_pred,
            ).statistic
        ),
    }

# ------------------------------------------------------------
# 4. Generate identity-aware bootstrap samples
# ------------------------------------------------------------

shield_bootstrap_rng = np.random.default_rng(
    SHIELD_RECOVERY_BOOTSTRAP_SEED
)

shield_recovery_bootstrap_distributions = {
    shield_target_name: {
        "r2": np.empty(
            SHIELD_RECOVERY_BOOTSTRAPS,
            dtype=float,
        ),
        "mae": np.empty(
            SHIELD_RECOVERY_BOOTSTRAPS,
            dtype=float,
        ),
        "rmse": np.empty(
            SHIELD_RECOVERY_BOOTSTRAPS,
            dtype=float,
        ),
        "spearman": np.empty(
            SHIELD_RECOVERY_BOOTSTRAPS,
            dtype=float,
        ),
    }
    for shield_target_name
    in shield_recovery_targets
}

print("\nRUNNING IDENTITY-AWARE BOOTSTRAP")
print("-" * 72)

for shield_bootstrap_number in range(
    SHIELD_RECOVERY_BOOTSTRAPS
):

    shield_sampled_indices = []

    for shield_category in shield_categories:

        shield_available_identities = (
            shield_category_to_identities[
                shield_category
            ]
        )

        shield_sampled_identities = (
            shield_bootstrap_rng.choice(
                shield_available_identities,
                size=len(shield_available_identities),
                replace=True,
            )
        )

        for shield_sampled_identity in (
            shield_sampled_identities
        ):
            shield_sampled_indices.append(
                shield_identity_to_indices[
                    shield_sampled_identity
                ]
            )

    shield_sampled_indices = np.concatenate(
        shield_sampled_indices
    )

    for shield_target_name, shield_y in (
        shield_recovery_targets.items()
    ):

        shield_y_pred = (
            shield_grouped_recovery_predictions[
                shield_target_name
            ]
        )

        shield_sampled_y = shield_y[
            shield_sampled_indices
        ]

        shield_sampled_prediction = shield_y_pred[
            shield_sampled_indices
        ]

        shield_distribution = (
            shield_recovery_bootstrap_distributions[
                shield_target_name
            ]
        )

        shield_distribution["r2"][
            shield_bootstrap_number
        ] = r2_score(
            shield_sampled_y,
            shield_sampled_prediction,
        )

        shield_distribution["mae"][
            shield_bootstrap_number
        ] = mean_absolute_error(
            shield_sampled_y,
            shield_sampled_prediction,
        )

        shield_distribution["rmse"][
            shield_bootstrap_number
        ] = np.sqrt(
            mean_squared_error(
                shield_sampled_y,
                shield_sampled_prediction,
            )
        )

        shield_distribution["spearman"][
            shield_bootstrap_number
        ] = spearmanr(
            shield_sampled_y,
            shield_sampled_prediction,
        ).statistic

    if (
        (shield_bootstrap_number + 1) % 1000 == 0
        or
        (shield_bootstrap_number + 1)
        == SHIELD_RECOVERY_BOOTSTRAPS
    ):
        print(
            f"completed "
            f"{shield_bootstrap_number + 1}/"
            f"{SHIELD_RECOVERY_BOOTSTRAPS}"
        )

# ------------------------------------------------------------
# 5. Summarize conditional bootstrap intervals
# ------------------------------------------------------------

shield_recovery_bootstrap_records = []

for shield_target_name in (
    shield_recovery_targets
):

    shield_observed = shield_observed_recovery[
        shield_target_name
    ]

    shield_distribution = (
        shield_recovery_bootstrap_distributions[
            shield_target_name
        ]
    )

    shield_r2_distribution = (
        shield_distribution["r2"]
    )

    shield_mae_distribution = (
        shield_distribution["mae"]
    )

    shield_rmse_distribution = (
        shield_distribution["rmse"]
    )

    shield_spearman_distribution = (
        shield_distribution["spearman"]
    )

    shield_recovery_bootstrap_records.append({
        "target":
            shield_target_name,

        "observed_r2":
            shield_observed["r2"],
        "r2_ci_low":
            float(
                np.percentile(
                    shield_r2_distribution,
                    2.5,
                )
            ),
        "r2_ci_high":
            float(
                np.percentile(
                    shield_r2_distribution,
                    97.5,
                )
            ),
        "r2_fraction_le_zero":
            float(
                np.mean(
                    shield_r2_distribution <= 0.0
                )
            ),

        "observed_mae":
            shield_observed["mae"],
        "mae_ci_low":
            float(
                np.percentile(
                    shield_mae_distribution,
                    2.5,
                )
            ),
        "mae_ci_high":
            float(
                np.percentile(
                    shield_mae_distribution,
                    97.5,
                )
            ),

        "observed_rmse":
            shield_observed["rmse"],
        "rmse_ci_low":
            float(
                np.percentile(
                    shield_rmse_distribution,
                    2.5,
                )
            ),
        "rmse_ci_high":
            float(
                np.percentile(
                    shield_rmse_distribution,
                    97.5,
                )
            ),

        "observed_spearman":
            shield_observed["spearman"],
        "spearman_ci_low":
            float(
                np.percentile(
                    shield_spearman_distribution,
                    2.5,
                )
            ),
        "spearman_ci_high":
            float(
                np.percentile(
                    shield_spearman_distribution,
                    97.5,
                )
            ),
        "spearman_fraction_le_zero":
            float(
                np.mean(
                    shield_spearman_distribution
                    <= 0.0
                )
            ),

        "bootstrap_replicates":
            SHIELD_RECOVERY_BOOTSTRAPS,
    })

shield_recovery_bootstrap_summary = pd.DataFrame(
    shield_recovery_bootstrap_records
)

# ------------------------------------------------------------
# 6. Display inferential summary
# ------------------------------------------------------------

print("\nIDENTITY-AWARE RECOVERY INTERVALS")
print("-" * 72)

display(
    shield_recovery_bootstrap_summary[
        [
            "target",
            "observed_r2",
            "r2_ci_low",
            "r2_ci_high",
            "r2_fraction_le_zero",
            "observed_spearman",
            "spearman_ci_low",
            "spearman_ci_high",
            "spearman_fraction_le_zero",
            "observed_mae",
            "mae_ci_low",
            "mae_ci_high",
            "observed_rmse",
            "rmse_ci_low",
            "rmse_ci_high",
        ]
    ]
)

print("\nSIGNAL SUMMARY")
print("-" * 72)

for shield_record in (
    shield_recovery_bootstrap_records
):

    shield_r2_supported = (
        shield_record["r2_ci_low"] > 0.0
    )

    shield_spearman_supported = (
        shield_record["spearman_ci_low"] > 0.0
    )

    print(
        f"{shield_record['target']:28s} | "
        f"R² CI excludes zero="
        f"{shield_r2_supported!s:5s} | "
        f"rho CI excludes zero="
        f"{shield_spearman_supported!s:5s}"
    )

# ------------------------------------------------------------
# 7. Lock bootstrap distributions
# ------------------------------------------------------------

for shield_target_distributions in (
    shield_recovery_bootstrap_distributions.values()
):
    for shield_metric_distribution in (
        shield_target_distributions.values()
    ):
        shield_metric_distribution.setflags(
            write=False
        )

print("\nINTERPRETATION BOUNDARY")
print("-" * 72)
print(
    "Intervals resample complete source identities within each "
    "category while holding grouped OOF predictions fixed."
)
print(
    "They quantify uncertainty across sampled garment identities "
    "conditional on the fitted fold-specific models."
)
print(
    "They are not model-refitting permutation tests and should "
    "not be presented as independent hypothesis-test p-values."
)

print("\n" + "=" * 72)
print("🟢 Complete source identities resampled as clusters")
print("🟢 Category composition preserved in every replicate")
print("🟢 Grouped OOF predictions remained fixed")
print("🟢 Four recovery targets evaluated identically")
print("🟢 No frozen feature or prediction was modified")
print("=" * 72)

🛡️ CELL 14 — IDENTITY-AWARE BOOTSTRAP FOR GROUPED RECOVERY

BOOTSTRAP DESIGN
------------------------------------------------------------------------
Rows                         : 2300
Categories                   : 23
Source identities            : 230
Identities per category      : 10
Bootstrap replicates         : 5000
Random seed                  : 20260820
Sampling unit                : complete source identity
Stratification               : within category
Predictions                  : fixed grouped OOF predictions
Model refitting              : False

RUNNING IDENTITY-AWARE BOOTSTRAP
------------------------------------------------------------------------
completed 1000/5000
completed 2000/5000
completed 3000/5000
completed 4000/5000
completed 5000/5000

IDENTITY-AWARE RECOVERY INTERVALS
------------------------------------------------------------------------


,target,observed_r2,r2_ci_low,r2_ci_high,r2_fraction_le_zero,observed_spearman,spearman_ci_low,spearman_ci_high,spearman_fraction_le_zero,observed_mae,mae_ci_low,mae_ci_high,observed_rmse,rmse_ci_low,rmse_ci_high
0,F2_peak_magnitude,0.302221,0.266905,0.333870,0.0000,0.631055,0.602427,0.658446,0.0,0.013196,0.012721,0.013686,0.017027,0.016362,0.017707
1,F2_peak_radius,0.014269,-0.042301,0.066573,0.2974,0.324874,0.284080,0.366127,0.0,4.080491,3.939184,4.216400,5.128389,4.940123,5.304208
2,R2_at_F2_peak,0.190971,0.121399,0.253892,0.0000,0.521587,0.479772,0.563474,0.0,0.127284,0.122482,0.132113,0.162518,0.155809,0.169390
3,axial_error,0.206346,0.148195,0.260074,0.0000,0.442901,0.396301,0.488413,0.0,20.041103,19.303544,20.797660,26.323096,25.389361,27.251001



SIGNAL SUMMARY
------------------------------------------------------------------------
F2_peak_magnitude            | R² CI excludes zero=True  | rho CI excludes zero=True 
F2_peak_radius               | R² CI excludes zero=False | rho CI excludes zero=True 
R2_at_F2_peak                | R² CI excludes zero=True  | rho CI excludes zero=True 
axial_error                  | R² CI excludes zero=True  | rho CI excludes zero=True 

INTERPRETATION BOUNDARY
------------------------------------------------------------------------
Intervals resample complete source identities within each category while holding grouped OOF predictions fixed.
They quantify uncertainty across sampled garment identities conditional on the fitted fold-specific models.
They are not model-refitting permutation tests and should not be presented as independent hypothesis-test p-values.

🟢 Complete source identities resampled as clusters
🟢 Category composition preserved in every replicate
🟢 Grouped OOF predictions rem

In [ ]:
# ============================================================
# CELL 15 — DIRECT AXIAL-ORIENTATION RECOVERY
#           DOUBLED-ANGLE GROUPED-CV SENSITIVITY
# ============================================================

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

print("=" * 72)
print("🛡️ CELL 15 — DIRECT AXIAL-ORIENTATION RECOVERY")
print("=" * 72)

# ------------------------------------------------------------
# 1. Prerequisites
# ------------------------------------------------------------

shield_required_objects = [
    "shield_recovery_X",
    "shield_group_splits",
    "shield_groups",
    "shield_labels",
    "shield_mu2_observed_at_peak",
    "shield_r2_at_f2_peak",
]

for shield_required_name in shield_required_objects:
    if shield_required_name not in globals():
        raise RuntimeError(
            f"Required object missing: {shield_required_name}"
        )

shield_orientation_X = np.asarray(
    shield_recovery_X,
    dtype=float,
)

shield_observed_axial_deg = np.asarray(
    shield_mu2_observed_at_peak,
    dtype=float,
)

shield_observed_r2_strength = np.asarray(
    shield_r2_at_f2_peak,
    dtype=float,
)

assert shield_orientation_X.shape == (2300, 135)
assert shield_observed_axial_deg.shape == (2300,)
assert shield_observed_r2_strength.shape == (2300,)

assert np.isfinite(shield_orientation_X).all()
assert np.isfinite(shield_observed_axial_deg).all()
assert np.isfinite(shield_observed_r2_strength).all()

assert np.all(shield_observed_axial_deg >= 0.0)
assert np.all(shield_observed_axial_deg < 180.0)

assert np.all(shield_observed_r2_strength >= 0.0)
assert np.all(shield_observed_r2_strength <= 1.0)

# ------------------------------------------------------------
# 2. Axial doubled-angle targets
# ------------------------------------------------------------

shield_observed_axial_rad = np.deg2rad(
    shield_observed_axial_deg
)

shield_axial_cos2 = np.cos(
    2.0 * shield_observed_axial_rad
)

shield_axial_sin2 = np.sin(
    2.0 * shield_observed_axial_rad
)

assert np.isfinite(shield_axial_cos2).all()
assert np.isfinite(shield_axial_sin2).all()

# ------------------------------------------------------------
# 3. Fixed component-wise Ridge estimator
# ------------------------------------------------------------

shield_axial_component_model = Pipeline([
    (
        "standardization",
        StandardScaler(),
    ),
    (
        "ridge",
        Ridge(alpha=1.0),
    ),
])

print("\nANALYSIS SPECIFICATION")
print("-" * 72)
print("Predictors              : 135-D frozen morphology")
print("Targets                 : cos(2α), sin(2α)")
print("Estimator               : Ridge(alpha=1.0) per component")
print("Scaling                 : inside each training fold")
print("Evaluation              : source-identity-grouped CV")
print("Primary population      : all 2300 sketches")
print("Primary error domain    : 0° → 90° axial error")
print("Category labels used    : False")
print("Hyperparameter search   : none")

# ------------------------------------------------------------
# 4. Circular helper functions
# ------------------------------------------------------------

def shield_components_to_axial_degrees(
    shield_cos_component,
    shield_sin_component,
):
    """
    Convert doubled-angle components to an axial direction
    in [0, 180) degrees.
    """

    shield_angle = 0.5 * np.arctan2(
        shield_sin_component,
        shield_cos_component,
    )

    return np.mod(
        np.rad2deg(shield_angle),
        180.0,
    )


def shield_axial_angular_error(
    shield_true_deg,
    shield_predicted_deg,
):
    """
    Shortest unsigned axial difference in [0, 90] degrees.
    """

    shield_difference = np.abs(
        shield_true_deg - shield_predicted_deg
    )

    shield_difference = np.mod(
        shield_difference,
        180.0,
    )

    return np.minimum(
        shield_difference,
        180.0 - shield_difference,
    )


def shield_orientation_metrics(
    shield_error_deg,
    shield_r2_weights,
):
    """
    Summarize axial errors. Full-sample unweighted metrics
    remain primary; the R2-weighted mean is supplementary.
    """

    shield_error_deg = np.asarray(
        shield_error_deg,
        dtype=float,
    )

    shield_r2_weights = np.asarray(
        shield_r2_weights,
        dtype=float,
    )

    shield_weight_sum = np.sum(
        shield_r2_weights
    )

    if shield_weight_sum > 0.0:
        shield_weighted_mean_error = np.average(
            shield_error_deg,
            weights=shield_r2_weights,
        )
    else:
        shield_weighted_mean_error = np.nan

    return {
        "n": int(len(shield_error_deg)),
        "mean_error_deg": float(
            np.mean(shield_error_deg)
        ),
        "median_error_deg": float(
            np.median(shield_error_deg)
        ),
        "q25_error_deg": float(
            np.percentile(
                shield_error_deg,
                25.0,
            )
        ),
        "q75_error_deg": float(
            np.percentile(
                shield_error_deg,
                75.0,
            )
        ),
        "within_10_deg": float(
            np.mean(shield_error_deg <= 10.0)
        ),
        "within_15_deg": float(
            np.mean(shield_error_deg <= 15.0)
        ),
        "within_30_deg": float(
            np.mean(shield_error_deg <= 30.0)
        ),
        "mean_axial_agreement": float(
            np.mean(
                np.cos(
                    np.deg2rad(
                        2.0 * shield_error_deg
                    )
                )
            )
        ),
        "r2_weighted_mean_error_deg": float(
            shield_weighted_mean_error
        ),
    }

# ------------------------------------------------------------
# 5. Source-grouped OOF prediction
# ------------------------------------------------------------

shield_axial_cos2_oof = np.full(
    2300,
    np.nan,
    dtype=float,
)

shield_axial_sin2_oof = np.full(
    2300,
    np.nan,
    dtype=float,
)

shield_axial_baseline_cos2_oof = np.full(
    2300,
    np.nan,
    dtype=float,
)

shield_axial_baseline_sin2_oof = np.full(
    2300,
    np.nan,
    dtype=float,
)

shield_axial_fold_records = []

print("\nRUNNING GROUPED DOUBLED-ANGLE RECOVERY")
print("-" * 72)

for shield_fold_number, (
    shield_train_index,
    shield_test_index,
) in enumerate(
    shield_group_splits,
    start=1,
):

    shield_train_index = np.asarray(
        shield_train_index,
        dtype=int,
    )

    shield_test_index = np.asarray(
        shield_test_index,
        dtype=int,
    )

    # Confirm zero source-identity overlap
    shield_train_groups = set(
        shield_groups[shield_train_index]
    )

    shield_test_groups = set(
        shield_groups[shield_test_index]
    )

    assert len(
        shield_train_groups.intersection(
            shield_test_groups
        )
    ) == 0

    # Fit independent doubled-angle components
    shield_cos_model = clone(
        shield_axial_component_model
    )

    shield_sin_model = clone(
        shield_axial_component_model
    )

    shield_cos_model.fit(
        shield_orientation_X[shield_train_index],
        shield_axial_cos2[shield_train_index],
    )

    shield_sin_model.fit(
        shield_orientation_X[shield_train_index],
        shield_axial_sin2[shield_train_index],
    )

    shield_axial_cos2_oof[
        shield_test_index
    ] = shield_cos_model.predict(
        shield_orientation_X[shield_test_index]
    )

    shield_axial_sin2_oof[
        shield_test_index
    ] = shield_sin_model.predict(
        shield_orientation_X[shield_test_index]
    )

    # Training-fold-only mean axial-direction baseline
    shield_train_mean_cos2 = np.mean(
        shield_axial_cos2[shield_train_index]
    )

    shield_train_mean_sin2 = np.mean(
        shield_axial_sin2[shield_train_index]
    )

    shield_axial_baseline_cos2_oof[
        shield_test_index
    ] = shield_train_mean_cos2

    shield_axial_baseline_sin2_oof[
        shield_test_index
    ] = shield_train_mean_sin2

    # Temporary fold predictions for audit
    shield_fold_predicted_deg = (
        shield_components_to_axial_degrees(
            shield_axial_cos2_oof[
                shield_test_index
            ],
            shield_axial_sin2_oof[
                shield_test_index
            ],
        )
    )

    shield_fold_baseline_deg = (
        shield_components_to_axial_degrees(
            shield_axial_baseline_cos2_oof[
                shield_test_index
            ],
            shield_axial_baseline_sin2_oof[
                shield_test_index
            ],
        )
    )

    shield_fold_model_error = (
        shield_axial_angular_error(
            shield_observed_axial_deg[
                shield_test_index
            ],
            shield_fold_predicted_deg,
        )
    )

    shield_fold_baseline_error = (
        shield_axial_angular_error(
            shield_observed_axial_deg[
                shield_test_index
            ],
            shield_fold_baseline_deg,
        )
    )

    shield_axial_fold_records.append({
        "fold":
            shield_fold_number,
        "train_rows":
            len(shield_train_index),
        "test_rows":
            len(shield_test_index),
        "median_model_error_deg":
            float(
                np.median(
                    shield_fold_model_error
                )
            ),
        "median_baseline_error_deg":
            float(
                np.median(
                    shield_fold_baseline_error
                )
            ),
        "median_improvement_deg":
            float(
                np.median(
                    shield_fold_baseline_error
                )
                - np.median(
                    shield_fold_model_error
                )
            ),
        "mean_model_error_deg":
            float(
                np.mean(
                    shield_fold_model_error
                )
            ),
        "mean_baseline_error_deg":
            float(
                np.mean(
                    shield_fold_baseline_error
                )
            ),
        "mean_improvement_deg":
            float(
                np.mean(
                    shield_fold_baseline_error
                )
                - np.mean(
                    shield_fold_model_error
                )
            ),
    })

    print(
        f"Fold {shield_fold_number}: "
        f"model median="
        f"{np.median(shield_fold_model_error):.3f}°, "
        f"baseline median="
        f"{np.median(shield_fold_baseline_error):.3f}°, "
        f"Δ="
        f"{np.median(shield_fold_baseline_error) - np.median(shield_fold_model_error):+.3f}°"
    )

# ------------------------------------------------------------
# 6. Validate and normalize predicted components
# ------------------------------------------------------------

assert np.isfinite(shield_axial_cos2_oof).all()
assert np.isfinite(shield_axial_sin2_oof).all()

assert np.isfinite(
    shield_axial_baseline_cos2_oof
).all()

assert np.isfinite(
    shield_axial_baseline_sin2_oof
).all()

shield_axial_prediction_norm = np.hypot(
    shield_axial_cos2_oof,
    shield_axial_sin2_oof,
)

shield_axial_baseline_norm = np.hypot(
    shield_axial_baseline_cos2_oof,
    shield_axial_baseline_sin2_oof,
)

# atan2 is sufficient for angle reconstruction, but explicitly
# identify near-zero model vectors because their direction is unstable.

SHIELD_AXIAL_NORM_EPSILON = 1e-12

shield_near_zero_prediction_count = int(
    np.sum(
        shield_axial_prediction_norm
        <= SHIELD_AXIAL_NORM_EPSILON
    )
)

shield_near_zero_baseline_count = int(
    np.sum(
        shield_axial_baseline_norm
        <= SHIELD_AXIAL_NORM_EPSILON
    )
)

if shield_near_zero_prediction_count > 0:
    raise RuntimeError(
        "One or more doubled-angle model predictions have "
        "near-zero vector norm."
    )

if shield_near_zero_baseline_count > 0:
    raise RuntimeError(
        "The training-fold mean-direction baseline has "
        "near-zero vector norm."
    )

shield_axial_cos2_unit_oof = (
    shield_axial_cos2_oof
    / shield_axial_prediction_norm
)

shield_axial_sin2_unit_oof = (
    shield_axial_sin2_oof
    / shield_axial_prediction_norm
)

shield_axial_baseline_cos2_unit_oof = (
    shield_axial_baseline_cos2_oof
    / shield_axial_baseline_norm
)

shield_axial_baseline_sin2_unit_oof = (
    shield_axial_baseline_sin2_oof
    / shield_axial_baseline_norm
)

# ------------------------------------------------------------
# 7. Reconstruct held-out axial orientations
# ------------------------------------------------------------

shield_axial_predicted_deg = (
    shield_components_to_axial_degrees(
        shield_axial_cos2_unit_oof,
        shield_axial_sin2_unit_oof,
    )
)

shield_axial_baseline_predicted_deg = (
    shield_components_to_axial_degrees(
        shield_axial_baseline_cos2_unit_oof,
        shield_axial_baseline_sin2_unit_oof,
    )
)

shield_axial_model_error_deg = (
    shield_axial_angular_error(
        shield_observed_axial_deg,
        shield_axial_predicted_deg,
    )
)

shield_axial_baseline_error_deg = (
    shield_axial_angular_error(
        shield_observed_axial_deg,
        shield_axial_baseline_predicted_deg,
    )
)

assert np.all(
    shield_axial_model_error_deg >= 0.0
)

assert np.all(
    shield_axial_model_error_deg <= 90.0
)

assert np.all(
    shield_axial_baseline_error_deg >= 0.0
)

assert np.all(
    shield_axial_baseline_error_deg <= 90.0
)

# ------------------------------------------------------------
# 8. Primary full-sample results
# ------------------------------------------------------------

shield_axial_model_metrics = (
    shield_orientation_metrics(
        shield_axial_model_error_deg,
        shield_observed_r2_strength,
    )
)

shield_axial_baseline_metrics = (
    shield_orientation_metrics(
        shield_axial_baseline_error_deg,
        shield_observed_r2_strength,
    )
)

shield_axial_primary_results = pd.DataFrame([
    {
        "method": "Morphology doubled-angle Ridge",
        **shield_axial_model_metrics,
    },
    {
        "method": "Training-fold mean direction",
        **shield_axial_baseline_metrics,
    },
])

print("\nPRIMARY FULL-SAMPLE RESULTS")
print("-" * 72)

display(shield_axial_primary_results)

print("\nMODEL ADVANTAGE OVER BASELINE")
print("-" * 72)
print(
    "Mean-error reduction     : "
    f"{shield_axial_baseline_metrics['mean_error_deg'] - shield_axial_model_metrics['mean_error_deg']:+.6f}°"
)
print(
    "Median-error reduction   : "
    f"{shield_axial_baseline_metrics['median_error_deg'] - shield_axial_model_metrics['median_error_deg']:+.6f}°"
)
print(
    "R2-weighted reduction    : "
    f"{shield_axial_baseline_metrics['r2_weighted_mean_error_deg'] - shield_axial_model_metrics['r2_weighted_mean_error_deg']:+.6f}°"
)
print(
    "Axial-agreement increase : "
    f"{shield_axial_model_metrics['mean_axial_agreement'] - shield_axial_baseline_metrics['mean_axial_agreement']:+.6f}"
)

# ------------------------------------------------------------
# 9. R2-strength sensitivity strata
# ------------------------------------------------------------

shield_axial_strength_records = []

shield_axial_strength_strata = [
    ("All sketches", np.ones(2300, dtype=bool)),
    (
        "Observed R2 >= 0.20",
        shield_observed_r2_strength >= 0.20,
    ),
    (
        "Observed R2 >= 0.40",
        shield_observed_r2_strength >= 0.40,
    ),
]

for (
    shield_stratum_name,
    shield_stratum_mask,
) in shield_axial_strength_strata:

    shield_stratum_model_metrics = (
        shield_orientation_metrics(
            shield_axial_model_error_deg[
                shield_stratum_mask
            ],
            shield_observed_r2_strength[
                shield_stratum_mask
            ],
        )
    )

    shield_stratum_baseline_metrics = (
        shield_orientation_metrics(
            shield_axial_baseline_error_deg[
                shield_stratum_mask
            ],
            shield_observed_r2_strength[
                shield_stratum_mask
            ],
        )
    )

    shield_axial_strength_records.append({
        "stratum":
            shield_stratum_name,
        "n":
            int(np.sum(shield_stratum_mask)),

        "model_mean_error_deg":
            shield_stratum_model_metrics[
                "mean_error_deg"
            ],
        "baseline_mean_error_deg":
            shield_stratum_baseline_metrics[
                "mean_error_deg"
            ],
        "mean_error_reduction_deg":
            shield_stratum_baseline_metrics[
                "mean_error_deg"
            ]
            - shield_stratum_model_metrics[
                "mean_error_deg"
            ],

        "model_median_error_deg":
            shield_stratum_model_metrics[
                "median_error_deg"
            ],
        "baseline_median_error_deg":
            shield_stratum_baseline_metrics[
                "median_error_deg"
            ],
        "median_error_reduction_deg":
            shield_stratum_baseline_metrics[
                "median_error_deg"
            ]
            - shield_stratum_model_metrics[
                "median_error_deg"
            ],

        "model_within_15_deg":
            shield_stratum_model_metrics[
                "within_15_deg"
            ],
        "baseline_within_15_deg":
            shield_stratum_baseline_metrics[
                "within_15_deg"
            ],

        "model_mean_axial_agreement":
            shield_stratum_model_metrics[
                "mean_axial_agreement"
            ],
        "baseline_mean_axial_agreement":
            shield_stratum_baseline_metrics[
                "mean_axial_agreement"
            ],
    })

shield_axial_strength_summary = pd.DataFrame(
    shield_axial_strength_records
)

print("\nORIENTATION-STRENGTH SENSITIVITY")
print("-" * 72)

display(shield_axial_strength_summary)

# ------------------------------------------------------------
# 10. Fold stability
# ------------------------------------------------------------

shield_axial_fold_results = pd.DataFrame(
    shield_axial_fold_records
)

print("\nFOLD-LEVEL RESULTS")
print("-" * 72)

display(shield_axial_fold_results)

print("\nPREDICTED VECTOR-NORM AUDIT")
print("-" * 72)
print(
    f"Minimum model norm       : "
    f"{shield_axial_prediction_norm.min():.12f}"
)
print(
    f"Median model norm        : "
    f"{np.median(shield_axial_prediction_norm):.6f}"
)
print(
    f"Maximum model norm       : "
    f"{shield_axial_prediction_norm.max():.6f}"
)
print(
    f"Near-zero model vectors  : "
    f"{shield_near_zero_prediction_count}"
)

# ------------------------------------------------------------
# 11. Lock primary OOF objects
# ------------------------------------------------------------

shield_axial_cos2_oof.setflags(write=False)
shield_axial_sin2_oof.setflags(write=False)

shield_axial_predicted_deg.setflags(
    write=False
)

shield_axial_baseline_predicted_deg.setflags(
    write=False
)

shield_axial_model_error_deg.setflags(
    write=False
)

shield_axial_baseline_error_deg.setflags(
    write=False
)

print("\nINTERPRETATION BOUNDARY")
print("-" * 72)
print(
    "This is direct recovery of observed axial orientation "
    "using a doubled-angle representation."
)
print(
    "It is distinct from predicting the scalar magnitude of "
    "observed–learned orientation disagreement."
)
print(
    "The full 2300-sketch analysis is primary. R2 thresholds "
    "are supplementary sensitivity strata, not exclusions "
    "from the primary population."
)
print(
    "This component-wise Ridge analysis respects axial "
    "periodicity but is not a full probabilistic circular model."
)

print("\n" + "=" * 72)
print("🟢 Observed axial orientation encoded as cos(2α), sin(2α)")
print("🟢 Source-grouped folds preserved")
print("🟢 Fold-local scaling and Ridge fitting preserved")
print("🟢 Training-only mean-direction baseline constructed")
print("🟢 Full-sample and R2-strength results reported separately")
print("🟢 No category label entered model fitting")
print("=" * 72)

🛡️ CELL 15 — DIRECT AXIAL-ORIENTATION RECOVERY

ANALYSIS SPECIFICATION
------------------------------------------------------------------------
Predictors              : 135-D frozen morphology
Targets                 : cos(2α), sin(2α)
Estimator               : Ridge(alpha=1.0) per component
Scaling                 : inside each training fold
Evaluation              : source-identity-grouped CV
Primary population      : all 2300 sketches
Primary error domain    : 0° → 90° axial error
Category labels used    : False
Hyperparameter search   : none

RUNNING GROUPED DOUBLED-ANGLE RECOVERY
------------------------------------------------------------------------
Fold 1: model median=7.617°, baseline median=5.523°, Δ=-2.094°
Fold 2: model median=6.831°, baseline median=5.359°, Δ=-1.472°
Fold 3: model median=7.442°, baseline median=6.034°, Δ=-1.408°
Fold 4: model median=7.075°, baseline median=6.111°, Δ=-0.964°
Fold 5: model median=8.947°, baseline median=7.811°, Δ=-1.136°

PRIMARY FULL-SAMPL

,method,n,mean_error_deg,median_error_deg,q25_error_deg,q75_error_deg,within_10_deg,within_15_deg,within_30_deg,mean_axial_agreement,r2_weighted_mean_error_deg
0,Morphology doubled-angle Ridge,2300,20.011741,7.485617,2.911384,22.641830,0.575652,0.680000,0.784783,0.617136,15.734644
1,Training-fold mean direction,2300,21.459638,6.136701,2.173836,23.468162,0.626957,0.694348,0.766957,0.568979,16.758891



MODEL ADVANTAGE OVER BASELINE
------------------------------------------------------------------------
Mean-error reduction     : +1.447897°
Median-error reduction   : -1.348916°
R2-weighted reduction    : +1.024246°
Axial-agreement increase : +0.048157

ORIENTATION-STRENGTH SENSITIVITY
------------------------------------------------------------------------


,stratum,n,model_mean_error_deg,baseline_mean_error_deg,mean_error_reduction_deg,model_median_error_deg,baseline_median_error_deg,median_error_reduction_deg,model_within_15_deg,baseline_within_15_deg,model_mean_axial_agreement,baseline_mean_axial_agreement
0,All sketches,2300,20.011741,21.459638,1.447897,7.485617,6.136701,-1.348916,0.680000,0.694348,0.617136,0.568979
1,Observed R2 >= 0.20,2015,17.545936,18.914817,1.368880,6.397100,5.105746,-1.291354,0.731514,0.747395,0.675891,0.627658
2,Observed R2 >= 0.40,1315,12.256745,12.519628,0.262883,4.754000,3.415950,-1.338051,0.821293,0.855513,0.801461,0.778848



FOLD-LEVEL RESULTS
------------------------------------------------------------------------


,fold,train_rows,test_rows,median_model_error_deg,median_baseline_error_deg,median_improvement_deg,mean_model_error_deg,mean_baseline_error_deg,mean_improvement_deg
0,1,1839,461,7.617115,5.522966,-2.094150,19.798351,21.431766,1.633415
1,2,1840,460,6.830665,5.358519,-1.472146,18.073669,19.774792,1.701122
2,3,1841,459,7.441797,6.034041,-1.407756,20.357308,20.759478,0.402170
3,4,1840,460,7.074770,6.110650,-0.964120,19.262534,22.608772,3.346237
4,5,1840,460,8.947125,7.811331,-1.135794,22.568059,22.721922,0.153863



PREDICTED VECTOR-NORM AUDIT
------------------------------------------------------------------------
Minimum model norm       : 0.006218140592
Median model norm        : 0.610617
Maximum model norm       : 2.769042
Near-zero model vectors  : 0

INTERPRETATION BOUNDARY
------------------------------------------------------------------------
This is direct recovery of observed axial orientation using a doubled-angle representation.
It is distinct from predicting the scalar magnitude of observed–learned orientation disagreement.
The full 2300-sketch analysis is primary. R2 thresholds are supplementary sensitivity strata, not exclusions from the primary population.
This component-wise Ridge analysis respects axial periodicity but is not a full probabilistic circular model.

🟢 Observed axial orientation encoded as cos(2α), sin(2α)
🟢 Source-grouped folds preserved
🟢 Fold-local scaling and Ridge fitting preserved
🟢 Training-only mean-direction baseline constructed
🟢 Full-sample and R2-streng

In [ ]:
# ============================================================
# CELL 16 — PAIRED IDENTITY-AWARE BOOTSTRAP
#           FOR DIRECT AXIAL-ORIENTATION RECOVERY
# ============================================================

print("=" * 72)
print("🛡️ CELL 16 — PAIRED AXIAL-ORIENTATION BOOTSTRAP")
print("=" * 72)

# ------------------------------------------------------------
# 1. Configuration and prerequisites
# ------------------------------------------------------------

SHIELD_AXIAL_BOOTSTRAPS = 5000
SHIELD_AXIAL_BOOTSTRAP_SEED = 20260820

shield_required_objects = [
    "shield_axial_model_error_deg",
    "shield_axial_baseline_error_deg",
    "shield_observed_r2_strength",
    "shield_groups",
    "shield_labels",
]

for shield_required_name in shield_required_objects:
    if shield_required_name not in globals():
        raise RuntimeError(
            f"Required object missing: {shield_required_name}"
        )

shield_axial_model_error = np.asarray(
    shield_axial_model_error_deg,
    dtype=float,
)

shield_axial_baseline_error = np.asarray(
    shield_axial_baseline_error_deg,
    dtype=float,
)

shield_axial_strength = np.asarray(
    shield_observed_r2_strength,
    dtype=float,
)

shield_axial_bootstrap_groups = np.asarray(
    shield_groups,
    dtype=str,
)

shield_axial_bootstrap_labels = np.asarray(
    shield_labels,
)

for shield_array in [
    shield_axial_model_error,
    shield_axial_baseline_error,
    shield_axial_strength,
    shield_axial_bootstrap_groups,
    shield_axial_bootstrap_labels,
]:
    assert shield_array.shape == (2300,)

assert np.isfinite(shield_axial_model_error).all()
assert np.isfinite(shield_axial_baseline_error).all()
assert np.isfinite(shield_axial_strength).all()

assert np.all(
    (shield_axial_model_error >= 0.0)
    & (shield_axial_model_error <= 90.0)
)

assert np.all(
    (shield_axial_baseline_error >= 0.0)
    & (shield_axial_baseline_error <= 90.0)
)

# ------------------------------------------------------------
# 2. Reconstruct category-stratified identity map
# ------------------------------------------------------------

shield_axial_identity_to_indices = {}
shield_axial_identity_to_category = {}

for shield_identity in np.unique(
    shield_axial_bootstrap_groups
):

    shield_identity_indices = np.flatnonzero(
        shield_axial_bootstrap_groups
        == shield_identity
    )

    shield_identity_categories = np.unique(
        shield_axial_bootstrap_labels[
            shield_identity_indices
        ]
    )

    assert len(shield_identity_categories) == 1

    shield_axial_identity_to_indices[
        shield_identity
    ] = shield_identity_indices

    shield_axial_identity_to_category[
        shield_identity
    ] = shield_identity_categories[0]

shield_axial_categories = np.unique(
    shield_axial_bootstrap_labels
)

shield_axial_category_to_identities = {}

for shield_category in shield_axial_categories:

    shield_category_identities = np.asarray([
        shield_identity
        for shield_identity, shield_identity_category
        in shield_axial_identity_to_category.items()
        if shield_identity_category == shield_category
    ])

    assert len(shield_category_identities) == 10

    shield_axial_category_to_identities[
        shield_category
    ] = shield_category_identities

assert len(shield_axial_categories) == 23
assert len(shield_axial_identity_to_indices) == 230

print("\nBOOTSTRAP DESIGN")
print("-" * 72)
print("Rows                         : 2300")
print("Categories                   : 23")
print("Source identities            : 230")
print("Identities per category      : 10")
print(f"Bootstrap replicates         : {SHIELD_AXIAL_BOOTSTRAPS}")
print(f"Random seed                  : {SHIELD_AXIAL_BOOTSTRAP_SEED}")
print("Sampling unit                : complete source identity")
print("Stratification               : within category")
print("Comparison                   : paired model vs baseline")
print("Predictions                  : fixed grouped OOF")
print("Model refitting              : False")

# ------------------------------------------------------------
# 3. Paired-effect function
#    Positive values always favor the morphology model
# ------------------------------------------------------------

def shield_axial_paired_effects(
    shield_model_error,
    shield_baseline_error,
    shield_strength_weights,
):
    shield_model_error = np.asarray(
        shield_model_error,
        dtype=float,
    )

    shield_baseline_error = np.asarray(
        shield_baseline_error,
        dtype=float,
    )

    shield_strength_weights = np.asarray(
        shield_strength_weights,
        dtype=float,
    )

    shield_weight_sum = np.sum(
        shield_strength_weights
    )

    if shield_weight_sum > 0.0:
        shield_model_weighted_mean = np.average(
            shield_model_error,
            weights=shield_strength_weights,
        )

        shield_baseline_weighted_mean = np.average(
            shield_baseline_error,
            weights=shield_strength_weights,
        )

        shield_weighted_mean_reduction = (
            shield_baseline_weighted_mean
            - shield_model_weighted_mean
        )
    else:
        shield_weighted_mean_reduction = np.nan

    shield_model_agreement = np.mean(
        np.cos(
            np.deg2rad(
                2.0 * shield_model_error
            )
        )
    )

    shield_baseline_agreement = np.mean(
        np.cos(
            np.deg2rad(
                2.0 * shield_baseline_error
            )
        )
    )

    return {
        # Positive reduction means lower error for morphology
        "mean_error_reduction_deg": float(
            np.mean(shield_baseline_error)
            - np.mean(shield_model_error)
        ),

        "median_error_reduction_deg": float(
            np.median(shield_baseline_error)
            - np.median(shield_model_error)
        ),

        "r2_weighted_mean_error_reduction_deg": float(
            shield_weighted_mean_reduction
        ),

        # Positive increase means more model predictions
        # fall within the specified tolerance
        "within_10_increase": float(
            np.mean(shield_model_error <= 10.0)
            - np.mean(shield_baseline_error <= 10.0)
        ),

        "within_15_increase": float(
            np.mean(shield_model_error <= 15.0)
            - np.mean(shield_baseline_error <= 15.0)
        ),

        "within_30_increase": float(
            np.mean(shield_model_error <= 30.0)
            - np.mean(shield_baseline_error <= 30.0)
        ),

        # Positive increase means better axial agreement
        "axial_agreement_increase": float(
            shield_model_agreement
            - shield_baseline_agreement
        ),
    }

# ------------------------------------------------------------
# 4. Observed paired effects
# ------------------------------------------------------------

shield_axial_observed_effects = (
    shield_axial_paired_effects(
        shield_axial_model_error,
        shield_axial_baseline_error,
        shield_axial_strength,
    )
)

print("\nOBSERVED PAIRED EFFECTS")
print("-" * 72)
print("Positive values favor morphology.")

for (
    shield_metric_name,
    shield_metric_value,
) in shield_axial_observed_effects.items():
    print(
        f"{shield_metric_name:42s} "
        f"{shield_metric_value:+.6f}"
    )

# ------------------------------------------------------------
# 5. Paired identity-aware bootstrap
# ------------------------------------------------------------

shield_axial_bootstrap_rng = np.random.default_rng(
    SHIELD_AXIAL_BOOTSTRAP_SEED
)

shield_axial_bootstrap_distributions = {
    shield_metric_name: np.empty(
        SHIELD_AXIAL_BOOTSTRAPS,
        dtype=float,
    )
    for shield_metric_name
    in shield_axial_observed_effects
}

print("\nRUNNING PAIRED IDENTITY BOOTSTRAP")
print("-" * 72)

for shield_bootstrap_number in range(
    SHIELD_AXIAL_BOOTSTRAPS
):

    shield_sampled_indices = []

    for shield_category in (
        shield_axial_categories
    ):

        shield_available_identities = (
            shield_axial_category_to_identities[
                shield_category
            ]
        )

        shield_sampled_identities = (
            shield_axial_bootstrap_rng.choice(
                shield_available_identities,
                size=len(shield_available_identities),
                replace=True,
            )
        )

        for shield_sampled_identity in (
            shield_sampled_identities
        ):
            shield_sampled_indices.append(
                shield_axial_identity_to_indices[
                    shield_sampled_identity
                ]
            )

    shield_sampled_indices = np.concatenate(
        shield_sampled_indices
    )

    shield_bootstrap_effects = (
        shield_axial_paired_effects(
            shield_axial_model_error[
                shield_sampled_indices
            ],
            shield_axial_baseline_error[
                shield_sampled_indices
            ],
            shield_axial_strength[
                shield_sampled_indices
            ],
        )
    )

    for (
        shield_metric_name,
        shield_metric_value,
    ) in shield_bootstrap_effects.items():

        shield_axial_bootstrap_distributions[
            shield_metric_name
        ][
            shield_bootstrap_number
        ] = shield_metric_value

    if (
        (shield_bootstrap_number + 1) % 1000 == 0
        or
        (shield_bootstrap_number + 1)
        == SHIELD_AXIAL_BOOTSTRAPS
    ):
        print(
            f"completed "
            f"{shield_bootstrap_number + 1}/"
            f"{SHIELD_AXIAL_BOOTSTRAPS}"
        )

# ------------------------------------------------------------
# 6. Bootstrap summary
# ------------------------------------------------------------

shield_axial_bootstrap_records = []

for (
    shield_metric_name,
    shield_observed_value,
) in shield_axial_observed_effects.items():

    shield_distribution = (
        shield_axial_bootstrap_distributions[
            shield_metric_name
        ]
    )

    shield_axial_bootstrap_records.append({
        "metric":
            shield_metric_name,

        "observed_effect":
            shield_observed_value,

        "bootstrap_mean":
            float(
                np.mean(shield_distribution)
            ),

        "ci_2.5_percent":
            float(
                np.percentile(
                    shield_distribution,
                    2.5,
                )
            ),

        "ci_97.5_percent":
            float(
                np.percentile(
                    shield_distribution,
                    97.5,
                )
            ),

        "fraction_le_zero":
            float(
                np.mean(
                    shield_distribution <= 0.0
                )
            ),

        "positive_ci":
            bool(
                np.percentile(
                    shield_distribution,
                    2.5,
                ) > 0.0
            ),

        "negative_ci":
            bool(
                np.percentile(
                    shield_distribution,
                    97.5,
                ) < 0.0
            ),

        "bootstrap_replicates":
            SHIELD_AXIAL_BOOTSTRAPS,
    })

shield_axial_bootstrap_summary = pd.DataFrame(
    shield_axial_bootstrap_records
)

print("\nPAIRED IDENTITY-AWARE INTERVALS")
print("-" * 72)
print("Positive effects favor morphology; negative effects favor baseline.")

display(shield_axial_bootstrap_summary)

# ------------------------------------------------------------
# 7. Directional interpretation audit
# ------------------------------------------------------------

print("\nEFFECT DIRECTION SUMMARY")
print("-" * 72)

for shield_record in (
    shield_axial_bootstrap_records
):

    if shield_record["positive_ci"]:
        shield_effect_conclusion = (
            "stable morphology advantage"
        )
    elif shield_record["negative_ci"]:
        shield_effect_conclusion = (
            "stable baseline advantage"
        )
    else:
        shield_effect_conclusion = (
            "interval includes zero"
        )

    print(
        f"{shield_record['metric']:42s} "
        f"{shield_effect_conclusion}"
    )

# ------------------------------------------------------------
# 8. Lock bootstrap distributions
# ------------------------------------------------------------

for shield_distribution in (
    shield_axial_bootstrap_distributions.values()
):
    shield_distribution.setflags(
        write=False
    )

print("\nINTERPRETATION BOUNDARY")
print("-" * 72)
print(
    "Complete source identities were resampled within category, "
    "and model and baseline errors remained paired."
)
print(
    "Intervals are conditional on the fixed grouped OOF "
    "predictions and do not include model-refitting variation."
)
print(
    "The analysis evaluates several complementary loss summaries; "
    "it is not a single confirmatory hypothesis test."
)

print("\n" + "=" * 72)
print("🟢 Complete identities resampled as paired clusters")
print("🟢 Category composition preserved")
print("🟢 Model and baseline evaluated on identical samples")
print("🟢 Positive effects consistently defined as model-favorable")
print("🟢 No model, feature, target, or prediction was modified")
print("=" * 72)

🛡️ CELL 16 — PAIRED AXIAL-ORIENTATION BOOTSTRAP

BOOTSTRAP DESIGN
------------------------------------------------------------------------
Rows                         : 2300
Categories                   : 23
Source identities            : 230
Identities per category      : 10
Bootstrap replicates         : 5000
Random seed                  : 20260820
Sampling unit                : complete source identity
Stratification               : within category
Comparison                   : paired model vs baseline
Predictions                  : fixed grouped OOF
Model refitting              : False

OBSERVED PAIRED EFFECTS
------------------------------------------------------------------------
Positive values favor morphology.
mean_error_reduction_deg                   +1.447897
median_error_reduction_deg                 -1.348916
r2_weighted_mean_error_reduction_deg       +1.024246
within_10_increase                         -0.051304
within_15_increase                         -0.014348
with

,metric,observed_effect,bootstrap_mean,ci_2.5_percent,ci_97.5_percent,fraction_le_zero,positive_ci,negative_ci,bootstrap_replicates
0,mean_error_reduction_deg,1.447897,1.459367,0.425764,2.525118,0.0024,True,False,5000
1,median_error_reduction_deg,-1.348916,-1.332023,-1.811536,-0.889670,1.0000,False,True,5000
2,r2_weighted_mean_error_reduction_deg,1.024246,1.039112,-0.113484,2.231822,0.0394,False,False,5000
3,within_10_increase,-0.051304,-0.051155,-0.069285,-0.033101,1.0000,False,True,5000
4,within_15_increase,-0.014348,-0.014196,-0.030435,0.002606,0.9572,False,False,5000
5,within_30_increase,0.017826,0.017964,0.004778,0.031767,0.0046,True,False,5000
6,axial_agreement_increase,0.048157,0.048449,0.023101,0.075019,0.0000,True,False,5000



EFFECT DIRECTION SUMMARY
------------------------------------------------------------------------
mean_error_reduction_deg                   stable morphology advantage
median_error_reduction_deg                 stable baseline advantage
r2_weighted_mean_error_reduction_deg       interval includes zero
within_10_increase                         stable baseline advantage
within_15_increase                         interval includes zero
within_30_increase                         stable morphology advantage
axial_agreement_increase                   stable morphology advantage

INTERPRETATION BOUNDARY
------------------------------------------------------------------------
Complete source identities were resampled within category, and model and baseline errors remained paired.
Intervals are conditional on the fixed grouped OOF predictions and do not include model-refitting variation.
The analysis evaluates several complementary loss summaries; it is not a single confirmatory hypothesis t

In [ ]:
# ============================================================
# CELL 17A — VALIDATED RESULT-OBJECT INVENTORY
# ============================================================

print("=" * 72)
print("🛡️ CELL 17A — VALIDATED RESULT-OBJECT INVENTORY")
print("=" * 72)

shield_result_keywords = (
    "result",
    "summary",
    "bootstrap",
    "repeated",
    "grouped",
    "alignment",
    "classification",
    "recovery",
    "axial",
)

print("\nDATAFRAME OBJECTS")
print("-" * 72)

for shield_name in sorted(globals()):

    shield_name_lower = shield_name.lower()

    if not any(
        shield_keyword in shield_name_lower
        for shield_keyword in shield_result_keywords
    ):
        continue

    shield_value = globals()[shield_name]

    if isinstance(shield_value, pd.DataFrame):

        print(
            f"\n{shield_name}"
            f"\n  shape   : {shield_value.shape}"
            f"\n  columns : {list(shield_value.columns)}"
        )

print("\nDICTIONARY OBJECTS")
print("-" * 72)

for shield_name in sorted(globals()):

    shield_name_lower = shield_name.lower()

    if not any(
        shield_keyword in shield_name_lower
        for shield_keyword in shield_result_keywords
    ):
        continue

    shield_value = globals()[shield_name]

    if isinstance(shield_value, dict):

        print(
            f"\n{shield_name}"
            f"\n  keys : {list(shield_value.keys())}"
        )

print("\nSCALAR RESULT OBJECTS")
print("-" * 72)

for shield_name in sorted(globals()):

    shield_name_lower = shield_name.lower()

    if not any(
        shield_keyword in shield_name_lower
        for shield_keyword in shield_result_keywords
    ):
        continue

    shield_value = globals()[shield_name]

    if isinstance(
        shield_value,
        (int, float, np.integer, np.floating, bool),
    ):
        print(
            f"{shield_name:55s} "
            f"{shield_value}"
        )

print("\n" + "=" * 72)
print("🟢 Existing result objects inventoried")
print("🟢 No result, prediction, feature, or model was modified")
print("=" * 72)

🛡️ CELL 17A — VALIDATED RESULT-OBJECT INVENTORY

DATAFRAME OBJECTS
------------------------------------------------------------------------

shield_alignment_null
  shape   : (2000, 7)
  columns : ['permutation', 'permuted_combined_macro_f1', 'permuted_combined_ba', 'permuted_delta_vs_morphology_macro_f1', 'permuted_delta_vs_morphology_ba', 'aligned_minus_permuted_macro_f1', 'aligned_minus_permuted_ba']

shield_alignment_summary
  shape   : (2, 11)
  columns : ['metric', 'morphology_only', 'aligned_combined', 'observed_gain_vs_morphology', 'permuted_mean', 'permuted_median', 'permuted_2.5_percent', 'permuted_97.5_percent', 'aligned_minus_permuted_mean', 'empirical_alignment_p', 'permutations']

shield_axial_bootstrap_summary
  shape   : (7, 9)
  columns : ['metric', 'observed_effect', 'bootstrap_mean', 'ci_2.5_percent', 'ci_97.5_percent', 'fraction_le_zero', 'positive_ci', 'negative_ci', 'bootstrap_replicates']

shield_axial_fold_results
  shape   : (5, 9)
  columns : ['fold', 'train_r

In [ ]:
# ============================================================
# CELL 17B — FINAL RESULTS CONSOLIDATION AND FREEZE
# ============================================================

import os
import json
import pickle
import hashlib
import platform
import sklearn
from datetime import datetime, timezone

print("=" * 72)
print("🛡️ CELL 17B — FINAL RESULTS CONSOLIDATION AND FREEZE")
print("=" * 72)

# ------------------------------------------------------------
# 1. Verify required validated result objects
# ------------------------------------------------------------

shield_final_required_objects = [
    "shield_classification_summary",
    "shield_grouped_three_way_summary",
    "shield_grouped_category_results",
    "shield_identity_bootstrap_summary",
    "shield_repeated_grouped_summary",
    "shield_alignment_summary",
    "shield_grouped_recovery_results",
    "shield_recovery_bootstrap_summary",
    "shield_axial_primary_results",
    "shield_axial_bootstrap_summary",
    "shield_group_fold_audit",
]

for shield_required_name in (
    shield_final_required_objects
):
    if shield_required_name not in globals():
        raise RuntimeError(
            f"Required validated object missing: "
            f"{shield_required_name}"
        )

# ------------------------------------------------------------
# 2. Canonical classification table
# ------------------------------------------------------------

shield_final_classification = pd.concat(
    [
        shield_classification_summary.copy(),
        shield_grouped_three_way_summary.copy(),
    ],
    ignore_index=True,
)

shield_final_classification = (
    shield_final_classification
    .drop_duplicates(
        subset=[
            "split_design",
            "representation",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

shield_final_classification = (
    shield_final_classification[
        [
            "split_design",
            "representation",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
)

# ------------------------------------------------------------
# 3. Primary grouped integration result
# ------------------------------------------------------------

shield_final_primary_integration = pd.DataFrame([
    {
        "evaluation":
            "Source-identity-grouped",

        "morphology_macro_f1":
            float(
                shield_grouped_primary_result[
                    "morphology_macro_f1"
                ]
            ),

        "combined_macro_f1":
            float(
                shield_grouped_primary_result[
                    "combined_macro_f1"
                ]
            ),

        "delta_macro_f1":
            float(
                shield_grouped_primary_result[
                    "delta_macro_f1"
                ]
            ),

        "morphology_balanced_accuracy":
            float(
                shield_grouped_primary_result[
                    "morphology_balanced_accuracy"
                ]
            ),

        "combined_balanced_accuracy":
            float(
                shield_grouped_primary_result[
                    "combined_balanced_accuracy"
                ]
            ),

        "delta_balanced_accuracy":
            float(
                shield_grouped_primary_result[
                    "delta_balanced_accuracy"
                ]
            ),

        "positive_macro_f1_folds":
            int(
                shield_grouped_primary_result[
                    "macro_f1_folds_improved"
                ]
            ),

        "positive_balanced_accuracy_folds":
            int(
                shield_grouped_primary_result[
                    "balanced_accuracy_folds_improved"
                ]
            ),

        "total_folds":
            5,

        "categories_combined_above_morphology":
            int(
                (
                    shield_grouped_category_results[
                        "combined_minus_morphology"
                    ] > 0.0
                ).sum()
            ),

        "categories_combined_above_ra":
            int(
                (
                    shield_grouped_category_results[
                        "combined_minus_ra"
                    ] > 0.0
                ).sum()
            ),

        "total_categories":
            int(
                len(
                    shield_grouped_category_results
                )
            ),
    }
])

# ------------------------------------------------------------
# 4. Integration uncertainty and robustness
# ------------------------------------------------------------

shield_final_integration_uncertainty = (
    shield_identity_bootstrap_summary.copy()
)

shield_final_split_robustness = (
    shield_repeated_grouped_summary.copy()
)

shield_final_alignment_control = (
    shield_alignment_summary.copy()
)

# ------------------------------------------------------------
# 5. Morphology-to-RA recovery with bootstrap intervals
# ------------------------------------------------------------

shield_final_recovery = (
    shield_grouped_recovery_results
    .merge(
        shield_recovery_bootstrap_summary,
        on="target",
        how="inner",
        validate="one_to_one",
    )
)

assert len(shield_final_recovery) == 4

shield_final_recovery = (
    shield_final_recovery[
        [
            "target",

            "historical_image_level_r2",
            "grouped_unseen_identity_r2",
            "r2_ci_low",
            "r2_ci_high",
            "r2_fraction_le_zero",

            "historical_image_level_mae",
            "grouped_unseen_identity_mae",
            "mae_ci_low",
            "mae_ci_high",

            "historical_image_level_rmse",
            "grouped_unseen_identity_rmse",
            "rmse_ci_low",
            "rmse_ci_high",

            "historical_image_level_spearman",
            "grouped_unseen_identity_spearman",
            "spearman_ci_low",
            "spearman_ci_high",
            "spearman_fraction_le_zero",

            "bootstrap_replicates",
        ]
    ]
)

# ------------------------------------------------------------
# 6. Supplementary direct axial-orientation result
# ------------------------------------------------------------

shield_final_axial_primary = (
    shield_axial_primary_results.copy()
)

shield_final_axial_paired = (
    shield_axial_bootstrap_summary.copy()
)

# ------------------------------------------------------------
# 7. Claim ledger
# ------------------------------------------------------------

shield_final_claim_ledger = pd.DataFrame([
    {
        "claim_id": "C1",
        "status": "Supported",
        "placement": "Primary result",
        "claim": (
            "Morphology and radial-angular descriptors provide "
            "complementary information for garment-category "
            "recognition under unseen source-identity evaluation."
        ),
        "boundary": (
            "Claim applies to this dataset, fixed representations, "
            "classifier, and source-grouped evaluation design."
        ),
    },
    {
        "claim_id": "C2",
        "status": "Supported",
        "placement": "Primary robustness",
        "claim": (
            "The grouped classification advantage is stable across "
            "identity-to-fold allocations."
        ),
        "boundary": (
            "Repeated grouped-CV variation is descriptive robustness "
            "evidence, not an independent confidence interval."
        ),
    },
    {
        "claim_id": "C3",
        "status": "Supported",
        "placement": "Primary uncertainty",
        "claim": (
            "The grouped integration gain remains positive under "
            "category-stratified source-identity bootstrap resampling."
        ),
        "boundary": (
            "Intervals condition on fixed grouped out-of-fold "
            "predictions and do not include model-refitting variation."
        ),
    },
    {
        "claim_id": "C4",
        "status": "Not supported",
        "placement": "Control / limitation",
        "claim": (
            "The classification gain specifically depends on exact "
            "held-out sketch-level morphology–RA alignment."
        ),
        "boundary": (
            "Within-category alignment perturbation did not provide "
            "clear evidence for this stronger mechanism."
        ),
    },
    {
        "claim_id": "C5",
        "status": "Supported",
        "placement": "Secondary result",
        "claim": (
            "Morphology contains identity-generalizable information "
            "about F2 magnitude, angular coherence at the F2 peak, "
            "and observed–learned axial-disagreement magnitude."
        ),
        "boundary": (
            "Recovery uses a fixed linear Ridge information probe; "
            "it does not establish causal or complete reconstruction."
        ),
    },
    {
        "claim_id": "C6",
        "status": "Qualified",
        "placement": "Secondary limitation",
        "claim": (
            "Morphology recovers the precise radial location of the "
            "F2 peak."
        ),
        "boundary": (
            "Rank association is positive, but grouped R-squared "
            "uncertainty includes zero; exact-value recovery is weak."
        ),
    },
    {
        "claim_id": "C7",
        "status": "Mixed",
        "placement": "Supplementary sensitivity",
        "claim": (
            "Morphology directly recovers observed axial orientation "
            "beyond a training-fold mean-direction reference."
        ),
        "boundary": (
            "Morphology improves mean and tail-sensitive measures but "
            "is worse for median and tight-threshold accuracy."
        ),
    },
])

# ------------------------------------------------------------
# 8. Assemble final table collection
# ------------------------------------------------------------

shield_final_tables = {
    "classification":
        shield_final_classification,

    "primary_grouped_integration":
        shield_final_primary_integration,

    "integration_identity_bootstrap":
        shield_final_integration_uncertainty,

    "repeated_grouped_cv":
        shield_final_split_robustness,

    "alignment_control":
        shield_final_alignment_control,

    "category_level_classification":
        shield_grouped_category_results.copy(),

    "grouped_recovery":
        shield_final_recovery,

    "axial_orientation_primary":
        shield_final_axial_primary,

    "axial_orientation_paired_bootstrap":
        shield_final_axial_paired,

    "claim_ledger":
        shield_final_claim_ledger,

    "grouped_fold_design":
        shield_group_fold_audit.copy(),
}

# ------------------------------------------------------------
# 9. Hash every canonical table
# ------------------------------------------------------------

def shield_hash_dataframe(shield_dataframe):
    shield_csv_bytes = (
        shield_dataframe
        .to_csv(
            index=False,
            lineterminator="\n",
            float_format="%.12g",
        )
        .encode("utf-8")
    )

    return hashlib.sha256(
        shield_csv_bytes
    ).hexdigest()


shield_final_table_manifest_records = []

for (
    shield_table_name,
    shield_table,
) in shield_final_tables.items():

    assert isinstance(
        shield_table,
        pd.DataFrame,
    )

    shield_final_table_manifest_records.append({
        "table":
            shield_table_name,
        "rows":
            int(shield_table.shape[0]),
        "columns":
            int(shield_table.shape[1]),
        "sha256":
            shield_hash_dataframe(
                shield_table
            ),
    })

shield_final_table_manifest = pd.DataFrame(
    shield_final_table_manifest_records
)

# ------------------------------------------------------------
# 10. Metadata
# ------------------------------------------------------------

shield_final_metadata = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "analysis":
        "CLO-SKET Final Validation Shield",

    "rows":
        2300,

    "categories":
        23,

    "source_identities":
        230,

    "grouped_folds":
        5,

    "grouped_fold_rule":
        "two source identities per category per test fold",

    "morphology_shape":
        [2300, 135],

    "morphology_dtype":
        str(shield_X_raw.dtype),

    "morphology_sha256":
        SHIELD_EXPECTED_MORPHOLOGY_SHA256,

    "radial_angular_shape":
        [2300, 28],

    "random_state":
        SHIELD_RANDOM_STATE,

    "python_version":
        platform.python_version(),

    "numpy_version":
        np.__version__,

    "pandas_version":
        pd.__version__,

    "sklearn_version":
        sklearn.__version__,

    "primary_claim":
        (
            "Morphology plus radial-angular descriptors improve "
            "garment-category recognition for unseen source "
            "garment identities."
        ),
}

# ------------------------------------------------------------
# 11. Save canonical CSV tables and compact pickle package
# ------------------------------------------------------------

shield_final_output_directory = (
    "/content/CLO_SKET_Final_Validation_Shield_Results"
)

os.makedirs(
    shield_final_output_directory,
    exist_ok=True,
)

for (
    shield_table_name,
    shield_table,
) in shield_final_tables.items():

    shield_table.to_csv(
        os.path.join(
            shield_final_output_directory,
            f"{shield_table_name}.csv",
        ),
        index=False,
        float_format="%.12g",
        lineterminator="\n",
    )

shield_final_table_manifest.to_csv(
    os.path.join(
        shield_final_output_directory,
        "table_manifest.csv",
    ),
    index=False,
    lineterminator="\n",
)

with open(
    os.path.join(
        shield_final_output_directory,
        "metadata.json",
    ),
    "w",
    encoding="utf-8",
) as shield_metadata_file:

    json.dump(
        shield_final_metadata,
        shield_metadata_file,
        indent=2,
        sort_keys=True,
    )

shield_final_package = {
    "metadata":
        shield_final_metadata,

    "tables":
        shield_final_tables,

    "table_manifest":
        shield_final_table_manifest,
}

shield_final_pickle_path = os.path.join(
    shield_final_output_directory,
    "CLO_SKET_Final_Validation_Shield_Results.pkl",
)

with open(
    shield_final_pickle_path,
    "wb",
) as shield_pickle_file:

    pickle.dump(
        shield_final_package,
        shield_pickle_file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

# ------------------------------------------------------------
# 12. Hash the complete package
# ------------------------------------------------------------

with open(
    shield_final_pickle_path,
    "rb",
) as shield_pickle_file:
    shield_final_package_sha256 = hashlib.sha256(
        shield_pickle_file.read()
    ).hexdigest()

with open(
    os.path.join(
        shield_final_output_directory,
        "package_sha256.txt",
    ),
    "w",
    encoding="utf-8",
) as shield_hash_file:
    shield_hash_file.write(
        shield_final_package_sha256 + "\n"
    )

# ------------------------------------------------------------
# 13. Final display
# ------------------------------------------------------------

print("\nPRIMARY GROUPED INTEGRATION")
print("-" * 72)
display(shield_final_primary_integration)

print("\nGROUPED RECOVERY")
print("-" * 72)
display(shield_final_recovery)

print("\nCLAIM LEDGER")
print("-" * 72)
display(shield_final_claim_ledger)

print("\nTABLE MANIFEST")
print("-" * 72)
display(shield_final_table_manifest)

print("\nFINAL PACKAGE")
print("-" * 72)
print(
    f"Directory : {shield_final_output_directory}"
)
print(
    f"Pickle    : {shield_final_pickle_path}"
)
print(
    f"SHA-256  : {shield_final_package_sha256}"
)

print("\n" + "=" * 72)
print("🟢 Validated results consolidated")
print("🟢 Primary and supplementary findings separated")
print("🟢 Claim boundaries recorded")
print("🟢 Every canonical table individually hashed")
print("🟢 Compact final results package saved")
print("🟢 No model was refitted")
print("🟢 No frozen feature or prediction was modified")
print("=" * 72)

🛡️ CELL 17B — FINAL RESULTS CONSOLIDATION AND FREEZE

PRIMARY GROUPED INTEGRATION
------------------------------------------------------------------------


,evaluation,morphology_macro_f1,combined_macro_f1,delta_macro_f1,morphology_balanced_accuracy,combined_balanced_accuracy,delta_balanced_accuracy,positive_macro_f1_folds,positive_balanced_accuracy_folds,total_folds,categories_combined_above_morphology,categories_combined_above_ra,total_categories
0,Source-identity-grouped,0.306847,0.341445,0.034598,0.307826,0.342174,0.034348,5,5,5,18,19,23



GROUPED RECOVERY
------------------------------------------------------------------------


,target,historical_image_level_r2,grouped_unseen_identity_r2,r2_ci_low,r2_ci_high,r2_fraction_le_zero,historical_image_level_mae,grouped_unseen_identity_mae,mae_ci_low,mae_ci_high,historical_image_level_rmse,grouped_unseen_identity_rmse,rmse_ci_low,rmse_ci_high,historical_image_level_spearman,grouped_unseen_identity_spearman,spearman_ci_low,spearman_ci_high,spearman_fraction_le_zero,bootstrap_replicates
0,F2_peak_magnitude,0.296073,0.302221,0.266905,0.333870,0.0000,0.013148,0.013196,0.012721,0.013686,0.017101,0.017027,0.016362,0.017707,0.641500,0.631055,0.602427,0.658446,0.0,5000
1,F2_peak_radius,0.059397,0.014269,-0.042301,0.066573,0.2974,4.015234,4.080491,3.939184,4.216400,5.009620,5.128389,4.940123,5.304208,0.341674,0.324874,0.284080,0.366127,0.0,5000
2,R2_at_F2_peak,0.217002,0.190971,0.121399,0.253892,0.0000,0.125806,0.127284,0.122482,0.132113,0.159882,0.162518,0.155809,0.169390,0.537740,0.521587,0.479772,0.563474,0.0,5000
3,axial_error,0.197931,0.206346,0.148195,0.260074,0.0000,20.151463,20.041103,19.303544,20.797660,26.462284,26.323096,25.389361,27.251001,0.439962,0.442901,0.396301,0.488413,0.0,5000



CLAIM LEDGER
------------------------------------------------------------------------


,claim_id,status,placement,claim,boundary
0,C1,Supported,Primary result,Morphology and radial-angular descriptors prov...,"Claim applies to this dataset, fixed represent..."
1,C2,Supported,Primary robustness,The grouped classification advantage is stable...,Repeated grouped-CV variation is descriptive r...
2,C3,Supported,Primary uncertainty,The grouped integration gain remains positive ...,Intervals condition on fixed grouped out-of-fo...
3,C4,Not supported,Control / limitation,The classification gain specifically depends o...,Within-category alignment perturbation did not...
4,C5,Supported,Secondary result,Morphology contains identity-generalizable inf...,Recovery uses a fixed linear Ridge information...
5,C6,Qualified,Secondary limitation,Morphology recovers the precise radial locatio...,"Rank association is positive, but grouped R-sq..."
6,C7,Mixed,Supplementary sensitivity,Morphology directly recovers observed axial or...,Morphology improves mean and tail-sensitive me...



TABLE MANIFEST
------------------------------------------------------------------------


,table,rows,columns,sha256
0,classification,5,5,dd2349b4ba19a30efdd05a8fd4a5b4667517e0a6b289a4...
1,primary_grouped_integration,1,13,e214bf7e01406a8df33bb530397f67413203ea7090fb10...
2,integration_identity_bootstrap,2,8,9fff3734a64d838b89851912b2e947cc99102bc8729eb4...
3,repeated_grouped_cv,2,8,89d19332ea5883d1a6c222e1741b7d0cacc3bc30765fad...
4,alignment_control,2,11,e34bb0ca5d411aff0679b9cbfaeda206f16223a1c98d7f...
5,category_level_classification,23,7,0ae8aed252e7076ebac3c51c66e9e957d75f3aa2bde7fb...
6,grouped_recovery,4,20,4e04ad9cefe2eb440a5344d09a4a10434f69a34711b064...
7,axial_orientation_primary,2,11,87b29d048ba7228488160d531b8bfe7eb411d7e30a876e...
8,axial_orientation_paired_bootstrap,7,9,93fdb0e6770a156412816c80823dad923f2bdee621feda...
9,claim_ledger,7,5,497b225a7bcb36281c064f6d0b49db2d9b4bb675b30abd...



FINAL PACKAGE
------------------------------------------------------------------------
Directory : /content/CLO_SKET_Final_Validation_Shield_Results
Pickle    : /content/CLO_SKET_Final_Validation_Shield_Results/CLO_SKET_Final_Validation_Shield_Results.pkl
SHA-256  : c174f33bd40950bc4daa66d39ee0290869b79f758eac77f3ae9303847ed9a2df

🟢 Validated results consolidated
🟢 Primary and supplementary findings separated
🟢 Claim boundaries recorded
🟢 Every canonical table individually hashed
🟢 Compact final results package saved
🟢 No model was refitted
🟢 No frozen feature or prediction was modified


In [ ]:
# ============================================================
# CELL 18 — FINAL ARCHIVE AND GOOGLE DRIVE BACKUP
# ============================================================

import os
import shutil
import hashlib
from datetime import datetime, timezone

print("=" * 72)
print("🛡️ CELL 18 — FINAL ARCHIVE AND GOOGLE DRIVE BACKUP")
print("=" * 72)

# ------------------------------------------------------------
# 1. Verify source package
# ------------------------------------------------------------

shield_source_directory = (
    "/content/CLO_SKET_Final_Validation_Shield_Results"
)

shield_source_pickle = os.path.join(
    shield_source_directory,
    "CLO_SKET_Final_Validation_Shield_Results.pkl",
)

assert os.path.isdir(shield_source_directory)
assert os.path.isfile(shield_source_pickle)

# ------------------------------------------------------------
# 2. Verify Google Drive
# ------------------------------------------------------------

shield_drive_root = "/content/drive/MyDrive"

if not os.path.isdir(shield_drive_root):
    raise RuntimeError(
        "Google Drive is not mounted. Mount Drive and rerun Cell 18."
    )

shield_drive_destination = os.path.join(
    shield_drive_root,
    "FashionAI",
    "CLO_SKET_Final_Validation_Shield",
)

os.makedirs(
    shield_drive_destination,
    exist_ok=True,
)

# ------------------------------------------------------------
# 3. Create ZIP archive in /content
# ------------------------------------------------------------

shield_archive_base = (
    "/content/CLO_SKET_Final_Validation_Shield_Results"
)

shield_local_zip = (
    shield_archive_base + ".zip"
)

if os.path.isfile(shield_local_zip):
    os.remove(shield_local_zip)

shutil.make_archive(
    base_name=shield_archive_base,
    format="zip",
    root_dir="/content",
    base_dir=os.path.basename(
        shield_source_directory
    ),
)

assert os.path.isfile(shield_local_zip)

# ------------------------------------------------------------
# 4. Hash local archive
# ------------------------------------------------------------

def shield_hash_file(
    shield_path,
    shield_chunk_size=1024 * 1024,
):
    shield_hasher = hashlib.sha256()

    with open(
        shield_path,
        "rb",
    ) as shield_file:

        while True:
            shield_chunk = shield_file.read(
                shield_chunk_size
            )

            if not shield_chunk:
                break

            shield_hasher.update(
                shield_chunk
            )

    return shield_hasher.hexdigest()


shield_local_zip_sha256 = shield_hash_file(
    shield_local_zip
)

shield_local_zip_size = os.path.getsize(
    shield_local_zip
)

# ------------------------------------------------------------
# 5. Copy directory and ZIP to Drive
# ------------------------------------------------------------

shield_drive_results_directory = os.path.join(
    shield_drive_destination,
    "CLO_SKET_Final_Validation_Shield_Results",
)

shield_drive_zip = os.path.join(
    shield_drive_destination,
    os.path.basename(shield_local_zip),
)

# Replace only the designated validation-results copy
if os.path.isdir(
    shield_drive_results_directory
):
    shutil.rmtree(
        shield_drive_results_directory
    )

shutil.copytree(
    shield_source_directory,
    shield_drive_results_directory,
)

shutil.copy2(
    shield_local_zip,
    shield_drive_zip,
)

# ------------------------------------------------------------
# 6. Verify Drive copies independently
# ------------------------------------------------------------

shield_drive_zip_sha256 = shield_hash_file(
    shield_drive_zip
)

assert (
    shield_drive_zip_sha256
    == shield_local_zip_sha256
)

shield_drive_pickle = os.path.join(
    shield_drive_results_directory,
    "CLO_SKET_Final_Validation_Shield_Results.pkl",
)

assert os.path.isfile(shield_drive_pickle)

shield_local_pickle_sha256 = shield_hash_file(
    shield_source_pickle
)

shield_drive_pickle_sha256 = shield_hash_file(
    shield_drive_pickle
)

assert (
    shield_drive_pickle_sha256
    == shield_local_pickle_sha256
)

# ------------------------------------------------------------
# 7. Write backup receipt
# ------------------------------------------------------------

shield_backup_receipt_path = os.path.join(
    shield_drive_destination,
    "CLO_SKET_Final_Validation_Shield_Backup_Receipt.txt",
)

shield_backup_utc = datetime.now(
    timezone.utc
).isoformat()

with open(
    shield_backup_receipt_path,
    "w",
    encoding="utf-8",
) as shield_receipt_file:

    shield_receipt_file.write(
        "CLO-SKET FINAL VALIDATION SHIELD\n"
    )

    shield_receipt_file.write(
        "=" * 72 + "\n"
    )

    shield_receipt_file.write(
        f"backup_utc: {shield_backup_utc}\n"
    )

    shield_receipt_file.write(
        f"source_directory: {shield_source_directory}\n"
    )

    shield_receipt_file.write(
        f"drive_directory: {shield_drive_results_directory}\n"
    )

    shield_receipt_file.write(
        f"drive_zip: {shield_drive_zip}\n"
    )

    shield_receipt_file.write(
        f"zip_size_bytes: {shield_local_zip_size}\n"
    )

    shield_receipt_file.write(
        f"zip_sha256: {shield_drive_zip_sha256}\n"
    )

    shield_receipt_file.write(
        f"pickle_sha256: {shield_drive_pickle_sha256}\n"
    )

    shield_receipt_file.write(
        "verification: local and Drive hashes identical\n"
    )

# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

print("\nARCHIVE")
print("-" * 72)
print(f"Local ZIP       : {shield_local_zip}")
print(f"ZIP size        : {shield_local_zip_size:,} bytes")
print(f"ZIP SHA-256     : {shield_local_zip_sha256}")

print("\nGOOGLE DRIVE BACKUP")
print("-" * 72)
print(f"Destination     : {shield_drive_destination}")
print(f"Results folder  : {shield_drive_results_directory}")
print(f"ZIP archive     : {shield_drive_zip}")
print(f"Receipt         : {shield_backup_receipt_path}")
print(f"Pickle SHA-256  : {shield_drive_pickle_sha256}")

print("\n" + "=" * 72)
print("🟢 Final results directory archived")
print("🟢 ZIP copied to Google Drive")
print("🟢 Uncompressed results copied to Google Drive")
print("🟢 Local and Drive ZIP hashes match")
print("🟢 Local and Drive pickle hashes match")
print("🟢 Backup receipt written")
print("=" * 72)

🛡️ CELL 18 — FINAL ARCHIVE AND GOOGLE DRIVE BACKUP

ARCHIVE
------------------------------------------------------------------------
Local ZIP       : /content/CLO_SKET_Final_Validation_Shield_Results.zip
ZIP size        : 15,673 bytes
ZIP SHA-256     : 889b8bf5b043ffe64865bbc204c0b49bdd4b878a083c2384e4e99a2bd3d51168

GOOGLE DRIVE BACKUP
------------------------------------------------------------------------
Destination     : /content/drive/MyDrive/FashionAI/CLO_SKET_Final_Validation_Shield
Results folder  : /content/drive/MyDrive/FashionAI/CLO_SKET_Final_Validation_Shield/CLO_SKET_Final_Validation_Shield_Results
ZIP archive     : /content/drive/MyDrive/FashionAI/CLO_SKET_Final_Validation_Shield/CLO_SKET_Final_Validation_Shield_Results.zip
Receipt         : /content/drive/MyDrive/FashionAI/CLO_SKET_Final_Validation_Shield/CLO_SKET_Final_Validation_Shield_Backup_Receipt.txt
Pickle SHA-256  : c174f33bd40950bc4daa66d39ee0290869b79f758eac77f3ae9303847ed9a2df

🟢 Final results directory arc